### Diffusion of a gaussian packegue

Number density of each flavor component along $x$. Each curve is one snapshot; color is time.

In [ ]:
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import h5py
import numpy as np

try:
    here = Path(__file__).resolve().parent
except NameError:
    here = Path.cwd()

sys.path.append(str((here / '..' / 'plot_functions').resolve()))
sys.path.append(str((here / '..' / 'phys_const').resolve()))

import plot_functions as pf
import phys_const as pc

Gaussian diffusion test simulation paramters

In [ ]:
# Directory path
directory_path = '/mnt/scratch/erick/erick_scratch/emu_methods_paper_simulations/cassowary/gaussian_diffusion_test/gaussian_diffusion_test'

##################################
# convergence test on cell size

directory_path = '/mnt/scratch/erick/erick_scratch/emu_methods_paper_simulations/cassowary/gaussian_diffusion_test/gaussian_diffusion_test_ncell_1024_1_1_1_dts_1.0e-12'
ncell = (1024, 1, 1)

directory_path = '/mnt/scratch/erick/erick_scratch/emu_methods_paper_simulations/cassowary/gaussian_diffusion_test/gaussian_diffusion_test_ncell_768_1_1_1_dts_1.0e-12'
ncell = (768, 1, 1)

directory_path = '/mnt/scratch/erick/erick_scratch/emu_methods_paper_simulations/cassowary/gaussian_diffusion_test/gaussian_diffusion_test_ncell_512_1_1_1_dts_1.0e-12'
ncell = (512, 1, 1)

directory_path = '/mnt/scratch/erick/erick_scratch/emu_methods_paper_simulations/cassowary/gaussian_diffusion_test/gaussian_diffusion_test_ncell_256_1_1_1_dts_1.0e-12'
ncell = (256, 1, 1)

directory_path = '/mnt/scratch/erick/erick_scratch/emu_methods_paper_simulations/cassowary/gaussian_diffusion_test/gaussian_diffusion_test_ncell_128_1_1_1_dts_1.0e-12'
ncell = (128, 1, 1)

##################################

number_of_flavors = 3

# Simulation domain
Lx = 200.0 # cm
Ly = 0.1953125 # cm
Lz = 0.1953125  # cm
dx = Lx / ncell[0]
dy = Ly / ncell[1]
dz = Lz / ncell[2]
domain = (Lx, Ly, Lz)  # cm
x_cell_centers = np.arange(domain[0] / ncell[0] / 2, domain[0], domain[0] / ncell[0])

# Scattering opacities
kappar_cm = 5.91e-01
kapparbar_cm = 5.91e-01

# Initial conditions of Gaussian packet
# n = n_bg + A0 * exp(-(x - x0)^2 / (2 s0^2))
n_bg_ccm = 1.0e28
A0_ccm = 1.0e29
x0_cm = 100.0
s0_cm = 5.0

# Snapshots drawn on each spatial plot
lowlimfit = None
uplimfit = None

Analitic solution

In [ ]:
att_num_diff = 0.0

D_cm2_s = pc.PhysConst.c / (3.0 * kappar_cm)
D_num_cm2_s = pc.PhysConst.c * kappar_cm * dx**2 / 4.0
D_num_cm2_s *= att_num_diff

print(f'D_cm2_s (physical diffusion) = {D_cm2_s:.2e}')
print(f'D_num_cm2_s (numerical diffusion) = {D_num_cm2_s:.2e}')

Dnum_bigger_D = D_num_cm2_s > D_cm2_s

if Dnum_bigger_D:
    print('numerical diffusion is bigger than physical diffusion!')
    minval = 1.0e-2
    k_ideal  = np.sqrt( 4*minval / (dx     **2 * 3))
    dx_ideal = np.sqrt( 4*minval / (k_ideal**2 * 3))
    print(f"dx (previous): {dx:.2e}, dx_ideal (ideal): {dx_ideal:.2e}")
    print(f"kappar_cm (previous): {kappar_cm:.2e}, k_ideal_cm (ideal): {k_ideal:.2e}")    
else:
    print('physical diffusion is bigger than numerical diffusion!')

D_cm2_s += D_num_cm2_s

def analytic_solution(t_s, x_cm=None):
    """Cartesian reference density n(x, t) in cm^-3.

    n(x, t) = n_bg + A0 * s0 / sqrt(s0^2 + 2 D t)
              * exp(-(x - x0)^2 / (2 (s0^2 + 2 D t)))

    t_s is a time in seconds, or an array of times. x_cm defaults to the
    cell centers of the run. A scalar time returns shape (nx,); an array
    of times returns shape (nt, nx).
    """
    if x_cm is None:
        dx = domain[0] / ncell[0]
        x_cm = np.arange(dx / 2.0, domain[0], dx)
    x_cm = np.asarray(x_cm, dtype=float)
    t_in = np.asarray(t_s, dtype=float)
    variance = s0_cm**2 + 2.0 * D_cm2_s * np.atleast_1d(t_in)
    n = (
        n_bg_ccm
        + A0_ccm * s0_cm / np.sqrt(variance)[:, None]
        * np.exp(-((x_cm - x0_cm) ** 2) / (2.0 * variance[:, None]))
    )
    if t_in.ndim == 0:
        return n[0]
    return n

Reading simulation data

In [ ]:
all_data_path = os.path.join(directory_path, 'allData.h5')
with h5py.File(all_data_path, 'r') as all_data_file:
    nee = np.array(all_data_file['/N00_Re(1|ccm)'][:])
    nuu = np.array(all_data_file['/N11_Re(1|ccm)'][:])
    neu = np.sqrt(
        np.array(all_data_file['/N01_Re(1|ccm)'][:])**2
        + np.array(all_data_file['/N01_Im(1|ccm)'][:])**2
    )
    if number_of_flavors == 3:
        ntt = np.array(all_data_file['/N22_Re(1|ccm)'][:])
        net = np.sqrt(
            np.array(all_data_file['/N02_Re(1|ccm)'][:])**2
            + np.array(all_data_file['/N02_Im(1|ccm)'][:])**2
        )
        nut = np.sqrt(
            np.array(all_data_file['/N12_Re(1|ccm)'][:])**2
            + np.array(all_data_file['/N12_Im(1|ccm)'][:])**2
        )
    time_s = np.array(all_data_file['/t(s)'][:])

print(f"nee.shape: {nee.shape}")
print(f"time_s.shape: {time_s.shape}")


Plotting data

In [ ]:
pf.plot_colored_lines(
    x_cell_centers,                         # x-axis: space/grid centers, shape (1024,)
    nee[:, :, 0, 0],                      # y values: need shape (space, time)
    time_s,                                 # t-axis: list of times
    r'$z \, [\mathrm{cm}]$',                # x label
    r'$n_{ee} \, [\mathrm{cm}^{-3}$]',      # y label
    r'$t \, [s]$',                          # t label
    f"plots/{directory_path.split('/')[-1]}_n_ee_space.pdf",
    xlog= False, 
    ylog=False, 
    ylowerlimit=None, 
    yupperlimit=None,
    xlowerlimit=100.0,
    xupperlimit=140.0
)

Error stimation

In [ ]:
error_estimate_at_time = np.zeros_like(nee[:, :, 0, 0])
analytic_solution_all_times = np.zeros_like(nee[:, :, 0, 0])
for i in range(len(time_s)):
    analytic_solution_at_time = analytic_solution(time_s[i], x_cell_centers)
    error_estimate_at_time[i, :] = np.abs(nee[i, :, 0, 0] - analytic_solution_at_time)/analytic_solution_at_time
    analytic_solution_all_times[i, :] = analytic_solution_at_time

pf.plot_colored_lines(
    x_cell_centers,                         # x-axis: space/grid centers, shape (1024,)
    np.log10(error_estimate_at_time),                      # y values: need shape (space, time)
    time_s,                                 # t-axis: list of times
    r'$z \, [\mathrm{cm}]$',                # x label
    r'$\log_{10}\,(\,\mathrm{relative\,error}\,)$',      # y label
    r'$t \, [s]$',                          # t label
    f"plots/{directory_path.split('/')[-1]}_n_ee_error_space.pdf",
    xlog= False, 
    ylog= False, 
    ylowerlimit=None, 
    yupperlimit=None,
    xlowerlimit=100.0,
    xupperlimit=140.0
)

In [ ]:
def print_variable(name, value):
    """Print a pasteable assignment. '-' in the run name becomes 'min'."""
    run = directory_path.split("/")[-1].replace("-", "min").replace(".", "p")
    print(f"{name}_{run} = {value}")

# print last time step
print_variable("last_time_s", time_s[-1])
print_variable("last_nee_invccm", [float(x) for x in nee[-1, :, 0, 0]])
print_variable("last_analytic_solution_invccm", [float(x) for x in analytic_solution_all_times[-1, :]])
print_variable("x_cell_centers_cm", [float(x) for x in x_cell_centers])


Conservation of number of neutrinos

In [ ]:
number_of_neutrinos = np.zeros_like(time_s)
for i in range(len(time_s)):
    number_of_neutrinos[i] = np.sum(nee[i, :, 0, 0]*dx*dy*dz)

plt.figure(figsize=(8,6))
plt.plot(time_s, number_of_neutrinos/number_of_neutrinos[0] - 1, linewidth=2)
plt.xlabel(r'$t \, [s]$')
plt.ylabel(r'$N_{\nu}/N_{\nu}(t=0) - 1$')
plt.yscale("log")
plt.tight_layout()
plt.savefig(f"plots/{directory_path.split('/')[-1]}_n_ee_conservation_time.pdf")
plt.show()



# Plotting n for different spacial resolutions

In [ ]:
last_time_s_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 9.999999999999707e-09
last_nee_invccm_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = [1.000005457859812e+28, 1.000005473512734e+28, 1.000005504855123e+28, 1.0000055519625622e+28, 1.0000056149484218e+28, 1.0000056939643163e+28, 1.000005789200171e+28, 1.0000059008847793e+28, 1.000006029286136e+28, 1.0000061747119809e+28, 1.0000063375103634e+28, 1.0000065180702896e+28, 1.0000067168225473e+28, 1.000006934240443e+28, 1.000007170840645e+28, 1.0000074271842895e+28, 1.0000077038779974e+28, 1.0000080015748764e+28, 1.000008320975947e+28, 1.0000086628312836e+28, 1.000009027941507e+28, 1.0000094171591974e+28, 1.0000098313905233e+28, 1.000010271596865e+28, 1.0000107387966464e+28, 1.0000112340670338e+28, 1.0000117585460812e+28, 1.0000123134347287e+28, 1.0000128999988384e+28, 1.0000135195716998e+28, 1.0000141735561923e+28, 1.0000148634274458e+28, 1.000015590735318e+28, 1.0000163571072374e+28, 1.0000171642509553e+28, 1.0000180139576526e+28, 1.0000189081048611e+28, 1.0000198486599194e+28, 1.000020837683198e+28, 1.0000218773316546e+28, 1.0000229698625853e+28, 1.00002411763729e+28, 1.000025323125144e+28, 1.0000265889076504e+28, 1.0000279176828673e+28, 1.000029312269699e+28, 1.0000307756126368e+28, 1.0000323107864988e+28, 1.0000339210014486e+28, 1.0000356096081383e+28, 1.0000373801031253e+28, 1.0000392361344217e+28, 1.0000411815072193e+28, 1.0000432201899698e+28, 1.000045356320483e+28, 1.0000475942123526e+28, 1.0000499383617378e+28, 1.0000523934541055e+28, 1.000054964371426e+28, 1.0000576561995496e+28, 1.000060474235657e+28, 1.000063423996559e+28, 1.000066511226538e+28, 1.000069741905954e+28, 1.0000731222598616e+28, 1.0000766587670497e+28, 1.000080358169549e+28, 1.0000842274819902e+28, 1.0000882740018934e+28, 1.0000925053195722e+28, 1.0000969293290823e+28, 1.0001015542390533e+28, 1.000106388584032e+28, 1.0001114412361668e+28, 1.0001167214171832e+28, 1.0001222387110155e+28, 1.0001280030763828e+28, 1.0001340248603038e+28, 1.0001403148114547e+28, 1.0001468840944935e+28, 1.0001537443044264e+28, 1.0001609074815263e+28, 1.0001683861268201e+28, 1.0001761932177702e+28, 1.000184342224838e+28, 1.0001928471282844e+28, 1.0002017224352236e+28, 1.0002109831978244e+28, 1.0002206450314094e+28, 1.0002307241333608e+28, 1.0002412373027116e+28, 1.0002522019599303e+28, 1.0002636361676152e+28, 1.0002755586515783e+28, 1.000287988822607e+28, 1.000300946798729e+28, 1.0003144534282679e+28, 1.0003285303133695e+28, 1.0003431998342204e+28, 1.0003584851740515e+28, 1.0003744103445022e+28, 1.000391000212083e+28, 1.0004082805249292e+28, 1.0004262779405958e+28, 1.0004450200543618e+28, 1.0004645354282876e+28, 1.0004848536212562e+28, 1.0005060052193516e+28, 1.0005280218674275e+28, 1.0005509363013031e+28, 1.000574782380616e+28, 1.000599595122788e+28, 1.0006254107375768e+28, 1.0006522666624878e+28, 1.0006802015993072e+28, 1.000709255551098e+28, 1.0007394698604114e+28, 1.0007708872481982e+28, 1.0008035518537427e+28, 1.0008375092754708e+28, 1.000872806612545e+28, 1.0009094925078091e+28, 1.0009476171911536e+28, 1.000987232524302e+28, 1.0010283920463109e+28, 1.0010711510201771e+28, 1.001115566480325e+28, 1.0011616972813932e+28, 1.0012096041475908e+28, 1.0012593497234803e+28, 1.0013109986257273e+28, 1.0013646174956767e+28, 1.001420275053185e+28, 1.0014780421515867e+28, 1.0015379918335395e+28, 1.0016001993881017e+28, 1.001664742408762e+28, 1.00173170085279e+28, 1.0018011571014017e+28, 1.0018731960213194e+28, 1.0019479050272828e+28, 1.0020253741455715e+28, 1.002105696078986e+28, 1.0021889662726862e+28, 1.002275282981076e+28, 1.002364747336209e+28, 1.002457463416851e+28, 1.0025535383190696e+28, 1.0026530822276097e+28, 1.0027562084886808e+28, 1.002863033683679e+28, 1.002973677704152e+28, 1.0030882638276807e+28, 1.0032069187950561e+28, 1.0033297728886377e+28, 1.0034569600114602e+28, 1.003588617767652e+28, 1.0037248875440281e+28, 1.0038659145924703e+28, 1.0040118481134848e+28, 1.0041628413408585e+28, 1.0043190516274417e+28, 1.0044806405315755e+28, 1.004647773904322e+28, 1.0048206219787235e+28, 1.0049993594589522e+28, 1.00518416561061e+28, 1.0053752243522862e+28, 1.0055727243474258e+28, 1.00577685909776e+28, 1.0059878270368566e+28, 1.0062058316248485e+28, 1.0064310814437768e+28, 1.0066637902937417e+28, 1.0069041772895538e+28, 1.0071524669582402e+28, 1.0074088893370402e+28, 1.0076736800722208e+28, 1.007947080517774e+28, 1.0082293378352992e+28, 1.008520705094017e+28, 1.0088214413710105e+28, 1.0091318118521875e+28, 1.0094520879332292e+28, 1.0097825473206764e+28, 1.0101234741336617e+28, 1.0104751590051973e+28, 1.0108378991838228e+28, 1.0112119986352146e+28, 1.011597768143639e+28, 1.011995525413352e+28, 1.0124055951695953e+28, 1.0128283092596917e+28, 1.013264006753264e+28, 1.0137130340424888e+28, 1.0141757449415184e+28, 1.014652500785507e+28, 1.0151436705287966e+28, 1.0156496308425284e+28, 1.0161707662112135e+28, 1.0167074690284933e+28, 1.017260139691851e+28, 1.0178291866961453e+28, 1.0184150267259794e+28, 1.01901808474669e+28, 1.0196387940940227e+28, 1.0202775965621973e+28, 1.0209349424902923e+28, 1.0216112908471008e+28, 1.0223071093139714e+28, 1.0230228743657113e+28, 1.0237590713494153e+28, 1.024516194561166e+28, 1.0252947473204532e+28, 1.0260952420420913e+28, 1.0269182003055588e+28, 1.027764152921903e+28, 1.0286336399974737e+28, 1.029527210995128e+28, 1.0304454247920825e+28, 1.0313888497347087e+28, 1.0323580636901095e+28, 1.0333536540940827e+28, 1.0343762179955958e+28, 1.0354263620974114e+28, 1.036504702793064e+28, 1.0376118661996735e+28, 1.038748488186729e+28, 1.0399152144004082e+28, 1.0411127002835765e+28, 1.0423416110913314e+28, 1.0436026219017824e+28, 1.0448964176216904e+28, 1.0462236929874521e+28, 1.047585152560692e+28, 1.0489815107187084e+28, 1.0504134916393383e+28, 1.05188182928024e+28, 1.0533872673524547e+28, 1.0549305592881604e+28, 1.0565124682020146e+28, 1.0581337668467664e+28, 1.0597952375622065e+28, 1.0614976722176123e+28, 1.0632418721477628e+28, 1.065028648081886e+28, 1.0668588200659428e+28, 1.0687332173775613e+28, 1.070652678433865e+28, 1.0726180506919774e+28, 1.0746301905418978e+28, 1.0766899631918327e+28, 1.0787982425455756e+28, 1.0809559110721003e+28, 1.0831638596671728e+28, 1.0854229875063953e+28, 1.0877342018903584e+28, 1.0900984180811338e+28, 1.092516559130165e+28, 1.0949895556975264e+28, 1.09751834586246e+28, 1.100103874924704e+28, 1.1027470951971805e+28, 1.1054489657891241e+28, 1.1082104523804024e+28, 1.1110325269860483e+28, 1.11391616771172e+28, 1.1168623584994138e+28, 1.119872088863577e+28, 1.122946353617784e+28, 1.1260861525912332e+28, 1.129292490335709e+28, 1.1325663758225818e+28, 1.1359088221297023e+28, 1.1393208461185332e+28, 1.1428034681009462e+28, 1.1463577114961302e+28, 1.1499846024772809e+28, 1.153685169608049e+28, 1.15746044346899e+28, 1.161311456273602e+28, 1.165239241474314e+28, 1.169244833358238e+28, 1.1733292666325864e+28, 1.1774935760000886e+28, 1.181738795724287e+28, 1.1860659591843246e+28, 1.1904760984200989e+28, 1.1949702436672795e+28, 1.1995494228819316e+28, 1.2042146612558378e+28, 1.2089669807215656e+28, 1.213807399447779e+28, 1.218736931325203e+28, 1.2237565854425355e+28, 1.228867365553397e+28, 1.2340702695334655e+28, 1.2393662888288188e+28, 1.2447564078949626e+28, 1.25024160362688e+28, 1.2558228447806016e+28, 1.2615010913859266e+28, 1.267277294150674e+28, 1.273152393857053e+28, 1.2791273207496407e+28, 1.2852029939156126e+28, 1.2913803206575908e+28, 1.2976601958588816e+28, 1.304043501341655e+28, 1.3105311052181478e+28, 1.317123861235563e+28, 1.3238226081140923e+28, 1.330628168879404e+28, 1.3375413501890251e+28, 1.3445629416532373e+28, 1.3516937151510544e+28, 1.3589344241410456e+28, 1.366285802967875e+28, 1.3737485661646252e+28, 1.381323407751255e+28, 1.3890110005295067e+28, 1.3968119953750813e+28, 1.4047270205265935e+28, 1.4127566808727683e+28, 1.4209015572372566e+28, 1.429162205662277e+28, 1.437539156691072e+28, 1.4460329146495805e+28, 1.4546439569282124e+28, 1.4633727332636023e+28, 1.4722196650209781e+28, 1.4811851444780609e+28, 1.4902695341101507e+28, 1.4994731658775327e+28, 1.5087963405156529e+28, 1.5182393268279294e+28, 1.5278023609822057e+28, 1.5374856458119977e+28, 1.547289350121338e+28, 1.5572136079952596e+28, 1.5672585181159542e+28, 1.5774241430849486e+28, 1.5877105087517933e+28, 1.598117603550642e+28, 1.6086453778437444e+28, 1.6192937432739705e+28, 1.630062572125835e+28, 1.6409516966961707e+28, 1.6519609086746017e+28, 1.6630899585346873e+28, 1.674338554936156e+28, 1.68570636413877e+28, 1.6971930094285532e+28, 1.70879807055652e+28, 1.720521083191277e+28, 1.7323615383848694e+28, 1.744318882053822e+28, 1.7563925144746872e+28, 1.768581789795415e+28, 1.780886015562976e+28, 1.7933044522675964e+28, 1.8058363129041368e+28, 1.8184807625515725e+28, 1.8312369179706408e+28, 1.8441038472203284e+28, 1.8570805692940243e+28, 1.8701660537753753e+28, 1.883359220514783e+28, 1.8966589393267726e+28, 1.9100640297090203e+28, 1.9235732605831016e+28, 1.937185350058001e+28, 1.9508989652163923e+28, 1.9647127219243691e+28, 1.9786251846651737e+28, 1.9926348663972425e+28, 2.006740228436653e+28, 2.0209396803654845e+28, 2.0352315799651394e+28, 2.0496142331759434e+28, 2.064085894083267e+28, 2.0786447649300294e+28, 2.093288996156898e+28, 2.108016686469418e+28, 2.1228258829331787e+28, 2.1377145810974004e+28, 2.1526807251462888e+28, 2.1677222080796606e+28, 2.1828368719221256e+28, 2.198022507962013e+28, 2.213276857018873e+28, 2.2285976097416704e+28, 2.243982406935874e+28, 2.2594288399212767e+28, 2.2749344509192483e+28, 2.2904967334711114e+28, 2.30611313288632e+28, 2.32178104672102e+28, 2.3374978252879408e+28, 2.353260772195956e+28, 2.369067144921089e+28, 2.3849141554074375e+28, 2.400798970699374e+28, 2.4167187136037577e+28, 2.432670463383152e+28, 2.448651256479197e+28, 2.464658087266258e+28, 2.4806879088352796e+28, 2.496737633808261e+28, 2.5128041351815396e+28, 2.528884247199407e+28, 2.544974766257339e+28, 2.5610724518338554e+28, 2.5771740274513285e+28, 2.593276181666192e+28, 2.609375569086196e+28, 2.6254688114168228e+28, 2.641552498534287e+28, 2.6576231895863873e+28, 2.673677414119858e+28, 2.6897116732341374e+28, 2.705722440760576e+28, 2.7217061644680457e+28, 2.7376592672921372e+28, 2.753578148589796e+28, 2.7694591854169477e+28, 2.785298733829897e+28, 2.80109313020875e+28, 2.816838692603966e+28, 2.832531722102721e+28, 2.8481685042166126e+28, 2.863745310290302e+28, 2.8792583989277363e+28, 2.8947040174384905e+28, 2.9100784033008894e+28, 2.925377785642669e+28, 2.94059838673803e+28, 2.9557364235199165e+28, 2.97078810910689e+28, 2.985749654344492e+28, 3.0006172693582823e+28, 3.0153871651207016e+28, 3.030055555027981e+28, 3.044618656487632e+28, 3.0590726925156107e+28, 3.07341389334201e+28, 3.0876384980237973e+28, 3.101742756065019e+28, 3.115722929042025e+28, 3.129575292233468e+28, 3.1432961362546723e+28, 3.1568817686937065e+28, 3.1703285157500377e+28, 3.183632723873557e+28, 3.1967907614032788e+28, 3.2097990202048873e+28, 3.2226539173059285e+28, 3.235351896527693e+28, 3.247889430112704e+28, 3.2602630203464918e+28, 3.2724692011734337e+28, 3.284504539804723e+28, 3.2963656383180756e+28, 3.3080491352477074e+28, 3.3195517071636202e+28, 3.3308700702395447e+28, 3.3420009818080946e+28, 3.3529412419018827e+28, 3.363687694780937e+28, 3.3742372304427924e+28, 3.3845867861176323e+28, 3.3947333477445106e+28, 3.4046739514288695e+28, 3.41440568488107e+28, 3.423925688834028e+28, 3.4332311584381964e+28, 3.4423193446355037e+28, 3.4511875555089103e+28, 3.4598331576071066e+28, 3.4682535772450528e+28, 3.4764463017774645e+28, 3.4844088808445686e+28, 3.492138927591095e+28, 3.499634119855755e+28, 3.5068922013310254e+28, 3.513910982692727e+28, 3.5206883426978038e+28, 3.527222229251073e+28, 3.5335106604375533e+28, 3.539551725522425e+28, 3.5453435859172407e+28, 3.55088447611016e+28, 3.5561727045603905e+28, 3.5612066545579146e+28, 3.565984785044867e+28, 3.5705056314007426e+28, 3.574767806187416e+28, 3.578769999859666e+28, 3.582510981433103e+28, 3.585989599114005e+28, 3.5892047808889933e+28, 3.592155535074528e+28, 3.5948409508261215e+28, 3.597260198605196e+28, 3.599412530605658e+28, 3.6012972811385095e+28, 3.602913866973716e+28, 3.6042617876412123e+28, 3.6053406256877224e+28, 3.6061500468921813e+28, 3.606689800438261e+28, 3.606959719042923e+28, 3.606959719043177e+28, 3.6066898004390247e+28, 3.6061500468935407e+28, 3.6053406256894465e+28, 3.604261787643507e+28, 3.6029138669766026e+28, 3.6012972811419017e+28, 3.599412530609601e+28, 3.5972601986097113e+28, 3.5948409508312545e+28, 3.592155535080214e+28, 3.589204780895121e+28, 3.585989599120834e+28, 3.582510981440502e+28, 3.5787699998675005e+28, 3.574767806195945e+28, 3.5705056314099706e+28, 3.5659847850552005e+28, 3.5612066545689203e+28, 3.5561727045718747e+28, 3.5508844761220263e+28, 3.545343585929707e+28, 3.5395517255351913e+28, 3.5335106604507733e+28, 3.527222229264948e+28, 3.5206883427121352e+28, 3.5139109827072597e+28, 3.506892201345777e+28, 3.4996341198706757e+28, 3.492138927606184e+28, 3.4844088808598334e+28, 3.47644630179277e+28, 3.468253577260354e+28, 3.459833157622248e+28, 3.4511875555238817e+28, 3.44231934465049e+28, 3.4332311584532254e+28, 3.423925688848958e+28, 3.4144056848960485e+28, 3.404673951443799e+28, 3.3947333477592463e+28, 3.3845867861323675e+28, 3.3742372304574665e+28, 3.3636876947953815e+28, 3.352941241916339e+28, 3.3420009818222854e+28, 3.330870070253708e+28, 3.3195517071775497e+28, 3.3080491352614834e+28, 3.2963656383316577e+28, 3.284504539818042e+28, 3.2724692011865377e+28, 3.260263020359472e+28, 3.247889430125324e+28, 3.235351896540139e+28, 3.2226539173179906e+28, 3.209799020216448e+28, 3.196790761414575e+28, 3.1836327238846424e+28, 3.170328515760851e+28, 3.1568817687042785e+28, 3.1432961362650847e+28, 3.129575292243725e+28, 3.115722929052123e+28, 3.1017427560752474e+28, 3.0876384980340236e+28, 3.0734138933521197e+28, 3.0590726925257636e+28, 3.0446186564974934e+28, 3.0300555550376297e+28, 3.0153871651302383e+28, 3.0006172693675195e+28, 2.985749654353487e+28, 2.9707881091157675e+28, 2.9557364235284012e+28, 2.940598386746176e+28, 2.92537778565045e+28, 2.9100784033083036e+28, 2.8947040174456615e+28, 2.879258398934585e+28, 2.863745310296776e+28, 2.8481685042228253e+28, 2.832531722108572e+28, 2.816838692609603e+28, 2.8010931302138237e+28, 2.7852987338343126e+28, 2.7694591854210277e+28, 2.7535781485935986e+28, 2.7376592672955704e+28, 2.7217061644711723e+28, 2.705722440763338e+28, 2.6897116732365695e+28, 2.673677414122035e+28, 2.6576231895882054e+28, 2.641552498535732e+28, 2.6254688114179584e+28, 2.6093755690869235e+28, 2.5932761816665657e+28, 2.577174027451407e+28, 2.5610724518334323e+28, 2.5449747662566055e+28, 2.528884247198215e+28, 2.512804135179855e+28, 2.496737633806184e+28, 2.480687908832789e+28, 2.4646580872631065e+28, 2.4486512564756165e+28, 2.4326704633790907e+28, 2.4167187135990483e+28, 2.400798970694114e+28, 2.384914155401691e+28, 2.3690671449146947e+28, 2.3532607721889602e+28, 2.3374978252803445e+28, 2.3217810467128273e+28, 2.3061131328774174e+28, 2.2904967334616064e+28, 2.2749344509089714e+28, 2.25942883991019e+28, 2.243982406924109e+28, 2.2285976097290985e+28, 2.2132768570055106e+28, 2.1980225079478676e+28, 2.182836871907247e+28, 2.1677222080639547e+28, 2.152680725129784e+28, 2.137714581080051e+28, 2.12282588291492e+28, 2.108016686450273e+28, 2.0932889961368826e+28, 2.0786447649090964e+28, 2.064085894061352e+28, 2.049614233153136e+28, 2.03523157994132e+28, 2.020939680340579e+28, 2.0067402284107e+28, 1.9926348663702126e+28, 1.9786251846371104e+28, 1.964712721895164e+28, 1.9508989651860873e+28, 1.9371853500265646e+28, 1.9235732605505469e+28, 1.9100640296751358e+28, 1.8966589392908722e+28, 1.8833592204772165e+28, 1.870166053736562e+28, 1.857080569254107e+28, 1.8441038471793676e+28, 1.8312369179286007e+28, 1.818480762508525e+28, 1.8058363128601565e+28, 1.7933044522226387e+28, 1.780886015517078e+28, 1.768581789748519e+28, 1.7563925144269744e+28, 1.7443188820052884e+28, 1.732361538335525e+28, 1.7205210831412082e+28, 1.7087980705056925e+28, 1.6971930093769307e+28, 1.6857063640865023e+28, 1.6743385548831518e+28, 1.6630899584810157e+28, 1.6519609086203076e+28, 1.6409516966412457e+28, 1.6300625720702406e+28, 1.6192937432178079e+28, 1.6086453777870222e+28, 1.5981176034933347e+28, 1.5877105086939898e+28, 1.5774241430265905e+28, 1.567258518057126e+28, 1.5572136079359737e+28, 1.5472893500615093e+28, 1.537485645751728e+28, 1.527802360921553e+28, 1.518239326766796e+28, 1.5087963404541774e+28, 1.4994731658157312e+28, 1.4902695340478697e+28, 1.4811851444154567e+28, 1.4722196649580425e+28, 1.463372733200244e+28, 1.4546439568645137e+28, 1.4460329145854985e+28, 1.4375391566266465e+28, 1.4291622055975457e+28, 1.4209015571721822e+28, 1.4127566808074e+28, 1.404727020460953e+28, 1.3968119953090913e+28, 1.3890110004632238e+28, 1.381323407684631e+28, 1.3737485660977404e+28, 1.3662858029007682e+28, 1.3589344240737163e+28, 1.3516937150835276e+28, 1.3445629415855582e+28, 1.3375413501211303e+28, 1.330628168811368e+28, 1.3238226080459063e+28, 1.317123861167194e+28, 1.3105311051496181e+28, 1.3040435012730045e+28, 1.2976601957901094e+28, 1.2913803205887225e+28, 1.2852029938465971e+28, 1.2791273206804836e+28, 1.2731523937878243e+28, 1.2672772940813629e+28, 1.2615010913164667e+28, 1.2558228447110738e+28, 1.2502416035572713e+28, 1.2447564078252025e+28, 1.2393662887589807e+28, 1.2340702694635625e+28, 1.2288673654833882e+28, 1.2237565853725039e+28, 1.2187369312551566e+28, 1.2138073993776573e+28, 1.2089669806513756e+28, 1.2042146611856434e+28, 1.1995494228117437e+28, 1.1949702435970317e+28, 1.1904760983498696e+28, 1.186065959114082e+28, 1.1817387956540375e+28, 1.1774935759298443e+28, 1.1733292665623562e+28, 1.1692448332880475e+28, 1.165239241404125e+28, 1.1613114562034518e+28, 1.1574604433988586e+28, 1.1536851695379238e+28, 1.1499846024072037e+28, 1.1463577114260924e+28, 1.1428034680309302e+28, 1.1393208460485869e+28, 1.1359088220598127e+28, 1.1325663757526676e+28, 1.1292924902658888e+28, 1.1260861525214184e+28, 1.1229463535480434e+28, 1.1198720887938891e+28, 1.1168623584297204e+28, 1.1139161676421197e+28, 1.111032526916559e+28, 1.1082104523109304e+28, 1.1054489657196895e+28, 1.1027470951278055e+28, 1.1001038748553897e+28, 1.0975183457931868e+28, 1.0949895556283535e+28, 1.092516559060989e+28, 1.090098418012044e+28, 1.0877342018213244e+28, 1.0854229874374454e+28, 1.0831638595982954e+28, 1.0809559110033193e+28, 1.0787982424768267e+28, 1.0766899631231535e+28, 1.0746301904733131e+28, 1.072618050623457e+28, 1.0706526783653656e+28, 1.0687332173091475e+28, 1.0668588199976432e+28, 1.0650286480136527e+28, 1.06324187207959e+28, 1.06149767214959e+28, 1.059795237494247e+28, 1.0581337667789221e+28, 1.0565124681342794e+28, 1.054930559220562e+28, 1.0533872672849836e+28, 1.051881829212868e+28, 1.0504134915720847e+28, 1.0489815106515548e+28, 1.0475851524936692e+28, 1.0462236929205106e+28, 1.0448964175548801e+28, 1.043602621835098e+28, 1.042341611024739e+28, 1.0411127002170657e+28, 1.039915214334127e+28, 1.0387484881206145e+28, 1.0376118661337136e+28, 1.036504702727252e+28, 1.0354263620316965e+28, 1.0343762179299967e+28, 1.033353654028651e+28, 1.0323580636247237e+28, 1.0313888496694287e+28, 1.030445424726939e+28, 1.029527210930165e+28, 1.0286336399326679e+28, 1.0277641528572042e+28, 1.0269182002410186e+28, 1.0260952419776695e+28, 1.0252947472561324e+28, 1.0245161944969913e+28, 1.0237590712853313e+28, 1.023022874301757e+28, 1.022307109250146e+28, 1.021611290783415e+28, 1.0209349424268366e+28, 1.0202775964988166e+28, 1.0196387940307272e+28, 1.01901808468346e+28, 1.0184150266628314e+28, 1.0178291866330382e+28, 1.0172601396287763e+28, 1.0167074689654667e+28, 1.0161707661481904e+28, 1.0156496307795037e+28, 1.015143670465808e+28, 1.014652500722474e+28, 1.0141757448785058e+28, 1.013713033979458e+28, 1.0132640066901404e+28, 1.0128283091965432e+28, 1.0124055951064281e+28, 1.0119955253500384e+28, 1.011597768080281e+28, 1.0112119985717956e+28, 1.0108378991202695e+28, 1.0104751589415598e+28, 1.0101234740699197e+28, 1.0097825472568146e+28, 1.0094520878691892e+28, 1.0091318117880372e+28, 1.0088214413067166e+28, 1.0085207050295063e+28, 1.0082293377706225e+28, 1.0079470804529154e+28, 1.007673680007123e+28, 1.007408889271702e+28, 1.0071524668926174e+28, 1.0069041772236508e+28, 1.0066637902276118e+28, 1.0064310813774208e+28, 1.006205831558249e+28, 1.0059878269699768e+28, 1.0057768590306365e+28, 1.0055727242800339e+28, 1.005375224284599e+28, 1.0051841655427296e+28, 1.004999359390728e+28, 1.0048206219101223e+28, 1.0046477738352793e+28, 1.0044806404620338e+28, 1.0043190515574958e+28, 1.0041628412702437e+28, 1.0040118480420832e+28, 1.003865914520572e+28, 1.0037248874717273e+28, 1.0035886176949097e+28, 1.0034569599383264e+28, 1.0033297728151693e+28, 1.0032069187212131e+28, 1.0030882637534892e+28, 1.002973677629701e+28, 1.0028630336089347e+28, 1.0027562084136482e+28, 1.0026530821522668e+28, 1.0025535382434278e+28, 1.0024574633409539e+28, 1.0023647472600221e+28, 1.0022752829046076e+28, 1.0021889661959185e+28, 1.0021056960019722e+28, 1.002025374068267e+28, 1.001947904949711e+28, 1.0018731959434808e+28, 1.0018011570233325e+28, 1.0017317007744586e+28, 1.0016647423301742e+28, 1.0016001993093142e+28, 1.0015379917545343e+28, 1.0014780420723073e+28, 1.0014202749737298e+28, 1.0013646174159915e+28, 1.0013109985458462e+28, 1.0012593496434445e+28, 1.0012096040673466e+28, 1.0011616972009952e+28, 1.0011155663998048e+28, 1.0010711509394077e+28, 1.0010283919654172e+28, 1.0009872324432557e+28, 1.0009476171099114e+28, 1.0009094924264094e+28, 1.0008728065310465e+28, 1.000837509193751e+28, 1.0008035517718737e+28, 1.000770887166221e+28, 1.000739469778301e+28, 1.000709255468861e+28, 1.0006802015169725e+28, 1.0006522665800831e+28, 1.0006254106550553e+28, 1.0005995950402034e+28, 1.0005747822979913e+28, 1.0005509362185941e+28, 1.0005280217846374e+28, 1.0005060051365311e+28, 1.0004848535384511e+28, 1.000464535345474e+28, 1.000445019971546e+28, 1.0004262778578415e+28, 1.0004082804421535e+28, 1.000391000129363e+28, 1.0003744102618447e+28, 1.0003584850914381e+28, 1.0003431997517145e+28, 1.0003285302309385e+28, 1.0003144533459312e+28, 1.0003009467165163e+28, 1.0002879887405116e+28, 1.0002755585696288e+28, 1.0002636360858313e+28, 1.0002522018782834e+28, 1.0002412372212358e+28, 1.0002307240520844e+28, 1.0002206449502997e+28, 1.000210983116965e+28, 1.0002017223546017e+28, 1.0001928470478432e+28, 1.0001843421446827e+28, 1.000176193137886e+28, 1.0001683860471908e+28, 1.0001609074022598e+28, 1.0001537442254711e+28, 1.0001468840157968e+28, 1.0001403147330506e+28, 1.0001340247821903e+28, 1.0001280029986332e+28, 1.0001222386335815e+28, 1.000116721340136e+28, 1.0001114411595016e+28, 1.0001063885077755e+28, 1.0001015541632022e+28, 1.000096929253698e+28, 1.0000925052446093e+28, 1.0000882739274017e+28, 1.0000842274080436e+28, 1.0000803580960625e+28, 1.000076658694072e+28, 1.0000731221873967e+28, 1.000069741834051e+28, 1.000066511155146e+28, 1.0000634239256848e+28, 1.0000604741653142e+28, 1.000057656129764e+28, 1.000054964302359e+28, 1.0000523933856594e+28, 1.000049938293896e+28, 1.0000475941451887e+28, 1.0000453562539434e+28, 1.000043220124103e+28, 1.0000411814420693e+28, 1.0000392360699037e+28, 1.0000373800393386e+28, 1.0000356095450978e+28, 1.0000339209391288e+28, 1.0000323107249813e+28, 1.0000307755519437e+28, 1.0000293122097558e+28, 1.0000279176237369e+28, 1.0000265888493448e+28, 1.0000253230677074e+28, 1.0000241175807556e+28, 1.0000229698069326e+28, 1.0000218772769427e+28, 1.0000208376294015e+28, 1.000019848607056e+28, 1.0000189080529578e+28, 1.000018013906708e+28, 1.0000171642010258e+28, 1.0000163570583208e+28, 1.0000155906874408e+28, 1.000014863380621e+28, 1.0000141735104546e+28, 1.0000135195270356e+28, 1.000012899955315e+28, 1.0000123133923309e+28, 1.0000117585048673e+28, 1.000011234026979e+28, 1.0000107387578385e+28, 1.000010271559289e+28, 1.0000098313541965e+28, 1.0000094171241812e+28, 1.0000090279077584e+28, 1.0000086627988654e+28, 1.000008320944901e+28, 1.000008001545173e+28, 1.0000077038496997e+28, 1.000007427157456e+28, 1.0000071708152235e+28, 1.0000069342165148e+28, 1.0000067168001974e+28, 1.0000065180494574e+28, 1.0000063374910808e+28, 1.0000061746943838e+28, 1.0000060292701957e+28, 1.000005900870492e+28, 1.000005789187578e+28, 1.0000056939534358e+28, 1.0000056149392853e+28, 1.0000055519551702e+28, 1.000005504849576e+28, 1.0000054735090655e+28, 1.0000054578582651e+28]
last_analytic_solution_invccm_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = [1.0000028278404236e+28, 1.0000029837693927e+28, 1.0000031479657157e+28, 1.000003320848903e+28, 1.0000035028587152e+28, 1.0000036944560847e+28, 1.000003896124078e+28, 1.000004108368897e+28, 1.000004331720925e+28, 1.0000045667358128e+28, 1.000004813995615e+28, 1.0000050741099694e+28, 1.0000053477173286e+28, 1.000005635486241e+28, 1.0000059381166855e+28, 1.0000062563414605e+28, 1.0000065909276303e+28, 1.0000069426780308e+28, 1.0000073124328364e+28, 1.000007701071189e+28, 1.000008109512896e+28, 1.0000085387201942e+28, 1.0000089896995844e+28, 1.0000094635037413e+28, 1.0000099612334973e+28, 1.000010484039907e+28, 1.0000110331263912e+28, 1.0000116097509674e+28, 1.000012215228567e+28, 1.0000128509334411e+28, 1.000013518301665e+28, 1.0000142188337337e+28, 1.0000149540972615e+28, 1.0000157257297859e+28, 1.0000165354416755e+28, 1.0000173850191527e+28, 1.0000182763274286e+28, 1.0000192113139583e+28, 1.0000201920118182e+28, 1.0000212205432097e+28, 1.0000222991230957e+28, 1.0000234300629706e+28, 1.000024615774771e+28, 1.0000258587749309e+28, 1.0000271616885864e+28, 1.000028527253934e+28, 1.000029958326747e+28, 1.0000314578850581e+28, 1.0000330290340098e+28, 1.0000346750108783e+28, 1.0000363991902816e+28, 1.0000382050895683e+28, 1.0000400963744016e+28, 1.0000420768645391e+28, 1.0000441505398172e+28, 1.000046321546344e+28, 1.000048594202909e+28, 1.0000509730076164e+28, 1.0000534626447448e+28, 1.0000560679918473e+28, 1.0000587941270896e+28, 1.0000616463368434e+28, 1.0000646301235336e+28, 1.0000677512137518e+28, 1.0000710155666432e+28, 1.000074429382572e+28, 1.000077999112077e+28, 1.0000817314651222e+28, 1.0000856334206532e+28, 1.0000897122364657e+28, 1.000093975459397e+28, 1.0000984309358465e+28, 1.000103086822638e+28, 1.000107951598228e+28, 1.000113034074273e+28, 1.0001183434075664e+28, 1.0001238891123475e+28, 1.0001296810730036e+28, 1.000135729557163e+28, 1.0001420452291994e+28, 1.0001486391641515e+28, 1.00015552286207e+28, 1.0001627082628054e+28, 1.0001702077612426e+28, 1.0001780342229971e+28, 1.000186201000582e+28, 1.0001947219500588e+28, 1.0002036114481808e+28, 1.0002128844100434e+28, 1.0002225563072524e+28, 1.0002326431866195e+28, 1.0002431616894032e+28, 1.0002541290710994e+28, 1.0002655632218006e+28, 1.0002774826871336e+28, 1.0002899066897874e+28, 1.0003028551516454e+28, 1.0003163487165347e+28, 1.0003304087736047e+28, 1.0003450574813496e+28, 1.0003603177922848e+28, 1.0003762134782947e+28, 1.0003927691566612e+28, 1.0004100103167907e+28, 1.000427963347647e+28, 1.0004466555659107e+28, 1.0004661152448739e+28, 1.0004863716440835e+28, 1.0005074550397526e+28, 1.000529396755945e+28, 1.0005522291965542e+28, 1.0005759858780866e+28, 1.0006007014632628e+28, 1.0006264117954534e+28, 1.0006531539339582e+28, 1.0006809661901485e+28, 1.0007098881644788e+28, 1.0007399607843895e+28, 1.0007712263431062e+28, 1.0008037285393542e+28, 1.0008375125180003e+28, 1.0008726249116318e+28, 1.0009091138830891e+28, 1.000947029168965e+28, 1.0009864221240787e+28, 1.0010273457669413e+28, 1.0010698548262232e+28, 1.0011140057882337e+28, 1.0011598569454285e+28, 1.0012074684459498e+28, 1.0012569023442177e+28, 1.001308222652576e+28, 1.0013614953940083e+28, 1.0014167886559316e+28, 1.001474172645076e+28, 1.0015337197434622e+28, 1.0015955045654824e+28, 1.0016596040160969e+28, 1.0017260973501468e+28, 1.0017950662327979e+28, 1.0018665948011174e+28, 1.0019407697267886e+28, 1.002017680279974e+28, 1.0020974183943249e+28, 1.0021800787331452e+28, 1.0022657587567135e+28, 1.0023545587907625e+28, 1.0024465820961197e+28, 1.0025419349395101e+28, 1.0026407266655203e+28, 1.0027430697697237e+28, 1.0028490799729645e+28, 1.0029588762967992e+28, 1.003072581140089e+28, 1.0031903203567433e+28, 1.003312223334602e+28, 1.0034384230754577e+28, 1.003569056276203e+28, 1.0037042634110963e+28, 1.0038441888151398e+28, 1.0039889807685507e+28, 1.0041387915823204e+28, 1.0042937776848406e+28, 1.00445409970959e+28, 1.0046199225838538e+28, 1.0047914156184694e+28, 1.0049687525985694e+28, 1.0051521118753062e+28, 1.0053416764585354e+28, 1.0055376341104311e+28, 1.0057401774400098e+28, 1.0059495039985365e+28, 1.006165816375781e+28, 1.0063893222970961e+28, 1.006620234721286e+28, 1.0068587719392272e+28, 1.007105157673211e+28, 1.007359621176966e+28, 1.007622397336321e+28, 1.0078937267704715e+28, 1.0081738559337994e+28, 1.0084630372182057e+28, 1.008761529055907e+28, 1.009069596022646e+28, 1.0093875089412646e+28, 1.009715544985587e+28, 1.010053987784556e+28, 1.010403127526564e+28, 1.0107632610639223e+28, 1.0111346920173995e+28, 1.0115177308807726e+28, 1.0119126951253185e+28, 1.012319909304179e+28, 1.0127397051565272e+28, 1.0131724217114637e+28, 1.0136184053915631e+28, 1.0140780101159974e+28, 1.0145515974031512e+28, 1.0150395364726483e+28, 1.0155422043467043e+28, 1.0160599859507172e+28, 1.0165932742130056e+28, 1.017142470163603e+28, 1.0177079830320124e+28, 1.0182902303438253e+28, 1.0188896380161048e+28, 1.0195066404514308e+28, 1.0201416806305046e+28, 1.0207952102032033e+28, 1.0214676895779785e+28, 1.022159588009486e+28, 1.0228713836843336e+28, 1.0236035638048318e+28, 1.0243566246706284e+28, 1.0251310717581072e+28, 1.0259274197974276e+28, 1.0267461928470843e+28, 1.0275879243658539e+28, 1.0284531572820054e+28, 1.0293424440596414e+28, 1.0302563467620375e+28, 1.031195437111845e+28, 1.0321602965480204e+28, 1.0331515162793453e+28, 1.034169697334393e+28, 1.0352154506078013e+28, 1.0362893969027123e+28, 1.037392166969226e+28, 1.0385244015387284e+28, 1.039686751353944e+28, 1.0408798771945605e+28, 1.0421044498982822e+28, 1.043361150377152e+28, 1.0446506696289968e+28, 1.0459737087438388e+28, 1.0473309789051203e+28, 1.0487232013855846e+28, 1.0501511075376596e+28, 1.0516154387781891e+28, 1.05311694656735e+28, 1.054656392381604e+28, 1.056234547680525e+28, 1.0578521938673441e+28, 1.0595101222430571e+28, 1.0612091339539366e+28, 1.0629500399322941e+28, 1.064733660830334e+28, 1.0665608269469484e+28, 1.0684323781472937e+28, 1.0703491637750018e+28, 1.0723120425568676e+28, 1.0743218824998686e+28, 1.0763795607803618e+28, 1.0784859636253159e+28, 1.0806419861854286e+28, 1.0828485323999879e+28, 1.0851065148533358e+28, 1.0874168546227954e+28, 1.0897804811179238e+28, 1.0921983319109595e+28, 1.09467135255833e+28, 1.0972004964130971e+28, 1.0997867244282084e+28, 1.102431004950441e+28, 1.105134313504918e+28, 1.1078976325700812e+28, 1.110721951343016e+28, 1.11360826549502e+28, 1.1165575769173179e+28, 1.1195708934568227e+28, 1.122649228641859e+28, 1.1257936013977545e+28, 1.1290050357522285e+28, 1.1322845605304943e+28, 1.1356332090400133e+28, 1.1390520187448322e+28, 1.142542030929451e+28, 1.1461042903521671e+28, 1.1497398448878558e+28, 1.153449745160146e+28, 1.1572350441629618e+28, 1.1610967968714101e+28, 1.1650360598419939e+28, 1.1690538908021447e+28, 1.1731513482290752e+28, 1.1773294909179577e+28, 1.1815893775394439e+28, 1.1859320661865512e+28, 1.1903586139109475e+28, 1.1948700762486738e+28, 1.199467506735358e+28, 1.204151956410974e+28, 1.2089244733142197e+28, 1.2137861019665869e+28, 1.2187378828462112e+28, 1.2237808518516002e+28, 1.2289160397553427e+28, 1.2341444716479201e+28, 1.2394671663717403e+28, 1.244885135945537e+28, 1.2503993849792746e+28, 1.2560109100797207e+28, 1.2617206992468532e+28, 1.267529731261276e+28, 1.2734389750628384e+28, 1.279449389120654e+28, 1.2855619207947292e+28, 1.291777505689426e+28, 1.2980970669989877e+28, 1.3045215148453738e+28, 1.3110517456086563e+28, 1.3176886412502448e+28, 1.3244330686292151e+28, 1.3312858788120292e+28, 1.3382479063759473e+28, 1.3453199687064382e+28, 1.3525028652889116e+28, 1.3597973769951027e+28, 1.3672042653644501e+28, 1.3747242718808254e+28, 1.3823581172449716e+28, 1.390106500643033e+28, 1.3979700990115541e+28, 1.4059495662993505e+28, 1.4140455327266546e+28, 1.4222586040419531e+28, 1.4305893607769432e+28, 1.4390383575000433e+28, 1.4476061220689093e+28, 1.4562931548824038e+28, 1.4650999281324913e+28, 1.474026885056531e+28, 1.48307443919045e+28, 1.4922429736232898e+28, 1.5015328402536266e+28, 1.5109443590483759e+28, 1.5204778173044924e+28, 1.5301334689140992e+28, 1.539911533633567e+28, 1.5498121963570912e+28, 1.559835606395306e+28, 1.5699818767594903e+28, 1.5802510834519232e+28, 1.5906432647629503e+28, 1.6011584205753316e+28, 1.6117965116764432e+28, 1.622557459078911e+28, 1.633441143350259e+28, 1.6444474039521573e+28, 1.655576038589858e+28, 1.6668268025724127e+28, 1.6781994081842633e+28, 1.689693524068803e+28, 1.7013087746245053e+28, 1.713044739414217e+28, 1.7249009525882156e+28, 1.7368769023216306e+28, 1.7489720302668215e+28, 1.761185731021316e+28, 1.7735173516118973e+28, 1.7859661909954366e+28, 1.7985314995770637e+28, 1.8112124787462518e+28, 1.8240082804314163e+28, 1.836918006673589e+28, 1.8499407092197556e+28, 1.8630753891364148e+28, 1.8763209964439269e+28, 1.8896764297722075e+28, 1.9031405360383098e+28, 1.9167121101464424e+28, 1.930389894710951e+28, 1.9441725798027898e+28, 1.958058802719992e+28, 1.9720471477826507e+28, 1.9861361461528974e+28, 2.000324275680359e+28, 2.0146099607735727e+28, 2.0289915722978027e+28, 2.0434674274997153e+28, 2.058035789959338e+28, 2.072694869569723e+28, 2.087442822544711e+28, 2.1022777514551924e+28, 2.1171977052942294e+28, 2.1322006795714003e+28, 2.1472846164366997e+28, 2.162447404834322e+28, 2.1776868806866246e+28, 2.1930008271085627e+28, 2.2083869746528585e+28, 2.2238430015861554e+28, 2.2393665341963803e+28, 2.254955147131526e+28, 2.2706063637700387e+28, 2.2863176566229763e+28, 2.3020864477680835e+28, 2.3179101093159044e+28, 2.3337859639080357e+28, 2.349711285247596e+28, 2.365683298661972e+28, 2.381699181697859e+28, 2.397756064748626e+28, 2.413851031713974e+28, 2.4299811206918504e+28, 2.4461433247025614e+28, 2.4623345924449885e+28, 2.4785518290848022e+28, 2.494791897074526e+28, 2.5110516170052985e+28, 2.5273277684901346e+28, 2.543617091078483e+28, 2.5599162852018292e+28, 2.5762220131500923e+28, 2.592530900078511e+28, 2.6088395350447174e+28, 2.6251444720756473e+28, 2.641442231263923e+28, 2.657729299893313e+28, 2.6740021335928617e+28, 2.6902571575192272e+28, 2.7064907675667776e+28, 2.7226993316049343e+28, 2.73887919074226e+28, 2.7550266606167344e+28, 2.771138032711657e+28, 2.787209575696582e+28, 2.803237536792669e+28, 2.8192181431618103e+28, 2.835147603318869e+28, 2.8510221085663457e+28, 2.866837834450755e+28, 2.8825909422399967e+28, 2.8982775804209483e+28, 2.9138938862165183e+28, 2.9294359871213625e+28, 2.944900002455439e+28, 2.960282044934575e+28, 2.9755782222571836e+28, 2.9907846387062617e+28, 3.0058973967657685e+28, 3.0209125987504855e+28, 3.0358263484484166e+28, 3.050634752774799e+28, 3.065333923436753e+28, 3.0799199786076085e+28, 3.0943890446099073e+28, 3.1087372576060917e+28, 3.122960765295848e+28, 3.1370557286191013e+28, 3.1510183234635965e+28, 3.1648447423760413e+28, 3.1785311962757354e+28, 3.1920739161696256e+28, 3.2054691548677185e+28, 3.218713188697752e+28, 3.231802319218053e+28, 3.2447328749274786e+28, 3.2575012129713496e+28, 3.2701037208422655e+28, 3.2825368180747012e+28, 3.2947969579322838e+28, 3.306880629086634e+28, 3.318784357286668e+28, 3.330504707017261e+28, 3.3420382831461526e+28, 3.35338173255801e+28, 3.3645317457745434e+28, 3.375485058559576e+28, 3.386238453507999e+28, 3.396788761617506e+28, 3.407132863842064e+28, 3.4172676926260263e+28, 3.427190233417867e+28, 3.436897526162453e+28, 3.4463866667708623e+28, 3.455654808566698e+28, 3.464699163707899e+28, 3.4735170045830693e+28, 3.4821056651813185e+28, 3.490462542434682e+28, 3.4985850975321533e+28, 3.506470857204409e+28, 3.5141174149783184e+28, 3.5215224324003443e+28, 3.528683640227965e+28, 3.5355988395882674e+28, 3.5422659031028855e+28, 3.5486827759784796e+28, 3.554847477061966e+28, 3.560758099859753e+28, 3.566412813520233e+28, 3.571809863778838e+28, 3.576947573864956e+28, 3.5818243453700778e+28, 3.5864386590765198e+28, 3.5907890757461354e+28, 3.5948742368684426e+28, 3.598692865367612e+28, 3.6022437662678156e+28, 3.6055258273164374e+28, 3.608538019564697e+28, 3.6112793979052676e+28, 3.613749101566478e+28, 3.6159463545627604e+28, 3.617870466100991e+28, 3.619520830942441e+28, 3.620896929720068e+28, 3.621998329210909e+28, 3.6228246825633907e+28, 3.6233757294793652e+28, 3.623651296350768e+28, 3.623651296350768e+28, 3.6233757294793652e+28, 3.6228246825633907e+28, 3.621998329210909e+28, 3.620896929720068e+28, 3.619520830942441e+28, 3.617870466100991e+28, 3.6159463545627604e+28, 3.613749101566478e+28, 3.6112793979052676e+28, 3.608538019564697e+28, 3.6055258273164374e+28, 3.6022437662678156e+28, 3.598692865367612e+28, 3.5948742368684426e+28, 3.5907890757461354e+28, 3.5864386590765198e+28, 3.5818243453700778e+28, 3.576947573864956e+28, 3.571809863778838e+28, 3.566412813520233e+28, 3.560758099859753e+28, 3.554847477061966e+28, 3.5486827759784796e+28, 3.5422659031028855e+28, 3.5355988395882674e+28, 3.528683640227965e+28, 3.5215224324003443e+28, 3.5141174149783184e+28, 3.506470857204409e+28, 3.4985850975321533e+28, 3.490462542434682e+28, 3.4821056651813185e+28, 3.4735170045830693e+28, 3.464699163707899e+28, 3.455654808566698e+28, 3.4463866667708623e+28, 3.436897526162453e+28, 3.427190233417867e+28, 3.4172676926260263e+28, 3.407132863842064e+28, 3.396788761617506e+28, 3.386238453507999e+28, 3.375485058559576e+28, 3.3645317457745434e+28, 3.35338173255801e+28, 3.3420382831461526e+28, 3.330504707017261e+28, 3.318784357286668e+28, 3.306880629086634e+28, 3.2947969579322838e+28, 3.2825368180747012e+28, 3.2701037208422655e+28, 3.2575012129713496e+28, 3.2447328749274786e+28, 3.231802319218053e+28, 3.218713188697752e+28, 3.2054691548677185e+28, 3.1920739161696256e+28, 3.1785311962757354e+28, 3.1648447423760413e+28, 3.1510183234635965e+28, 3.1370557286191013e+28, 3.122960765295848e+28, 3.1087372576060917e+28, 3.0943890446099073e+28, 3.0799199786076085e+28, 3.065333923436753e+28, 3.050634752774799e+28, 3.0358263484484166e+28, 3.0209125987504855e+28, 3.0058973967657685e+28, 2.9907846387062617e+28, 2.9755782222571836e+28, 2.960282044934575e+28, 2.944900002455439e+28, 2.9294359871213625e+28, 2.9138938862165183e+28, 2.8982775804209483e+28, 2.8825909422399967e+28, 2.866837834450755e+28, 2.8510221085663457e+28, 2.835147603318869e+28, 2.8192181431618103e+28, 2.803237536792669e+28, 2.787209575696582e+28, 2.771138032711657e+28, 2.7550266606167344e+28, 2.73887919074226e+28, 2.7226993316049343e+28, 2.7064907675667776e+28, 2.6902571575192272e+28, 2.6740021335928617e+28, 2.657729299893313e+28, 2.641442231263923e+28, 2.6251444720756473e+28, 2.6088395350447174e+28, 2.592530900078511e+28, 2.5762220131500923e+28, 2.5599162852018292e+28, 2.543617091078483e+28, 2.5273277684901346e+28, 2.5110516170052985e+28, 2.494791897074526e+28, 2.4785518290848022e+28, 2.4623345924449885e+28, 2.4461433247025614e+28, 2.4299811206918504e+28, 2.413851031713974e+28, 2.397756064748626e+28, 2.381699181697859e+28, 2.365683298661972e+28, 2.349711285247596e+28, 2.3337859639080357e+28, 2.3179101093159044e+28, 2.3020864477680835e+28, 2.2863176566229763e+28, 2.2706063637700387e+28, 2.254955147131526e+28, 2.2393665341963803e+28, 2.2238430015861554e+28, 2.2083869746528585e+28, 2.1930008271085627e+28, 2.1776868806866246e+28, 2.162447404834322e+28, 2.1472846164366997e+28, 2.1322006795714003e+28, 2.1171977052942294e+28, 2.1022777514551924e+28, 2.087442822544711e+28, 2.072694869569723e+28, 2.058035789959338e+28, 2.0434674274997153e+28, 2.0289915722978027e+28, 2.0146099607735727e+28, 2.000324275680359e+28, 1.9861361461528974e+28, 1.9720471477826507e+28, 1.958058802719992e+28, 1.9441725798027898e+28, 1.930389894710951e+28, 1.9167121101464424e+28, 1.9031405360383098e+28, 1.8896764297722075e+28, 1.8763209964439269e+28, 1.8630753891364148e+28, 1.8499407092197556e+28, 1.836918006673589e+28, 1.8240082804314163e+28, 1.8112124787462518e+28, 1.7985314995770637e+28, 1.7859661909954366e+28, 1.7735173516118973e+28, 1.761185731021316e+28, 1.7489720302668215e+28, 1.7368769023216306e+28, 1.7249009525882156e+28, 1.713044739414217e+28, 1.7013087746245053e+28, 1.689693524068803e+28, 1.6781994081842633e+28, 1.6668268025724127e+28, 1.655576038589858e+28, 1.6444474039521573e+28, 1.633441143350259e+28, 1.622557459078911e+28, 1.6117965116764432e+28, 1.6011584205753316e+28, 1.5906432647629503e+28, 1.5802510834519232e+28, 1.5699818767594903e+28, 1.559835606395306e+28, 1.5498121963570912e+28, 1.539911533633567e+28, 1.5301334689140992e+28, 1.5204778173044924e+28, 1.5109443590483759e+28, 1.5015328402536266e+28, 1.4922429736232898e+28, 1.48307443919045e+28, 1.474026885056531e+28, 1.4650999281324913e+28, 1.4562931548824038e+28, 1.4476061220689093e+28, 1.4390383575000433e+28, 1.4305893607769432e+28, 1.4222586040419531e+28, 1.4140455327266546e+28, 1.4059495662993505e+28, 1.3979700990115541e+28, 1.390106500643033e+28, 1.3823581172449716e+28, 1.3747242718808254e+28, 1.3672042653644501e+28, 1.3597973769951027e+28, 1.3525028652889116e+28, 1.3453199687064382e+28, 1.3382479063759473e+28, 1.3312858788120292e+28, 1.3244330686292151e+28, 1.3176886412502448e+28, 1.3110517456086563e+28, 1.3045215148453738e+28, 1.2980970669989877e+28, 1.291777505689426e+28, 1.2855619207947292e+28, 1.279449389120654e+28, 1.2734389750628384e+28, 1.267529731261276e+28, 1.2617206992468532e+28, 1.2560109100797207e+28, 1.2503993849792746e+28, 1.244885135945537e+28, 1.2394671663717403e+28, 1.2341444716479201e+28, 1.2289160397553427e+28, 1.2237808518516002e+28, 1.2187378828462112e+28, 1.2137861019665869e+28, 1.2089244733142197e+28, 1.204151956410974e+28, 1.199467506735358e+28, 1.1948700762486738e+28, 1.1903586139109475e+28, 1.1859320661865512e+28, 1.1815893775394439e+28, 1.1773294909179577e+28, 1.1731513482290752e+28, 1.1690538908021447e+28, 1.1650360598419939e+28, 1.1610967968714101e+28, 1.1572350441629618e+28, 1.153449745160146e+28, 1.1497398448878558e+28, 1.1461042903521671e+28, 1.142542030929451e+28, 1.1390520187448322e+28, 1.1356332090400133e+28, 1.1322845605304943e+28, 1.1290050357522285e+28, 1.1257936013977545e+28, 1.122649228641859e+28, 1.1195708934568227e+28, 1.1165575769173179e+28, 1.11360826549502e+28, 1.110721951343016e+28, 1.1078976325700812e+28, 1.105134313504918e+28, 1.102431004950441e+28, 1.0997867244282084e+28, 1.0972004964130971e+28, 1.09467135255833e+28, 1.0921983319109595e+28, 1.0897804811179238e+28, 1.0874168546227954e+28, 1.0851065148533358e+28, 1.0828485323999879e+28, 1.0806419861854286e+28, 1.0784859636253159e+28, 1.0763795607803618e+28, 1.0743218824998686e+28, 1.0723120425568676e+28, 1.0703491637750018e+28, 1.0684323781472937e+28, 1.0665608269469484e+28, 1.064733660830334e+28, 1.0629500399322941e+28, 1.0612091339539366e+28, 1.0595101222430571e+28, 1.0578521938673441e+28, 1.056234547680525e+28, 1.054656392381604e+28, 1.05311694656735e+28, 1.0516154387781891e+28, 1.0501511075376596e+28, 1.0487232013855846e+28, 1.0473309789051203e+28, 1.0459737087438388e+28, 1.0446506696289968e+28, 1.043361150377152e+28, 1.0421044498982822e+28, 1.0408798771945605e+28, 1.039686751353944e+28, 1.0385244015387284e+28, 1.037392166969226e+28, 1.0362893969027123e+28, 1.0352154506078013e+28, 1.034169697334393e+28, 1.0331515162793453e+28, 1.0321602965480204e+28, 1.031195437111845e+28, 1.0302563467620375e+28, 1.0293424440596414e+28, 1.0284531572820054e+28, 1.0275879243658539e+28, 1.0267461928470843e+28, 1.0259274197974276e+28, 1.0251310717581072e+28, 1.0243566246706284e+28, 1.0236035638048318e+28, 1.0228713836843336e+28, 1.022159588009486e+28, 1.0214676895779785e+28, 1.0207952102032033e+28, 1.0201416806305046e+28, 1.0195066404514308e+28, 1.0188896380161048e+28, 1.0182902303438253e+28, 1.0177079830320124e+28, 1.017142470163603e+28, 1.0165932742130056e+28, 1.0160599859507172e+28, 1.0155422043467043e+28, 1.0150395364726483e+28, 1.0145515974031512e+28, 1.0140780101159974e+28, 1.0136184053915631e+28, 1.0131724217114637e+28, 1.0127397051565272e+28, 1.012319909304179e+28, 1.0119126951253185e+28, 1.0115177308807726e+28, 1.0111346920173995e+28, 1.0107632610639223e+28, 1.010403127526564e+28, 1.010053987784556e+28, 1.009715544985587e+28, 1.0093875089412646e+28, 1.009069596022646e+28, 1.008761529055907e+28, 1.0084630372182057e+28, 1.0081738559337994e+28, 1.0078937267704715e+28, 1.007622397336321e+28, 1.007359621176966e+28, 1.007105157673211e+28, 1.0068587719392272e+28, 1.006620234721286e+28, 1.0063893222970961e+28, 1.006165816375781e+28, 1.0059495039985365e+28, 1.0057401774400098e+28, 1.0055376341104311e+28, 1.0053416764585354e+28, 1.0051521118753062e+28, 1.0049687525985694e+28, 1.0047914156184694e+28, 1.0046199225838538e+28, 1.00445409970959e+28, 1.0042937776848406e+28, 1.0041387915823204e+28, 1.0039889807685507e+28, 1.0038441888151398e+28, 1.0037042634110963e+28, 1.003569056276203e+28, 1.0034384230754577e+28, 1.003312223334602e+28, 1.0031903203567433e+28, 1.003072581140089e+28, 1.0029588762967992e+28, 1.0028490799729645e+28, 1.0027430697697237e+28, 1.0026407266655203e+28, 1.0025419349395101e+28, 1.0024465820961197e+28, 1.0023545587907625e+28, 1.0022657587567135e+28, 1.0021800787331452e+28, 1.0020974183943249e+28, 1.002017680279974e+28, 1.0019407697267886e+28, 1.0018665948011174e+28, 1.0017950662327979e+28, 1.0017260973501468e+28, 1.0016596040160969e+28, 1.0015955045654824e+28, 1.0015337197434622e+28, 1.001474172645076e+28, 1.0014167886559316e+28, 1.0013614953940083e+28, 1.001308222652576e+28, 1.0012569023442177e+28, 1.0012074684459498e+28, 1.0011598569454285e+28, 1.0011140057882337e+28, 1.0010698548262232e+28, 1.0010273457669413e+28, 1.0009864221240787e+28, 1.000947029168965e+28, 1.0009091138830891e+28, 1.0008726249116318e+28, 1.0008375125180003e+28, 1.0008037285393542e+28, 1.0007712263431062e+28, 1.0007399607843895e+28, 1.0007098881644788e+28, 1.0006809661901485e+28, 1.0006531539339582e+28, 1.0006264117954534e+28, 1.0006007014632628e+28, 1.0005759858780866e+28, 1.0005522291965542e+28, 1.000529396755945e+28, 1.0005074550397526e+28, 1.0004863716440835e+28, 1.0004661152448739e+28, 1.0004466555659107e+28, 1.000427963347647e+28, 1.0004100103167907e+28, 1.0003927691566612e+28, 1.0003762134782947e+28, 1.0003603177922848e+28, 1.0003450574813496e+28, 1.0003304087736047e+28, 1.0003163487165347e+28, 1.0003028551516454e+28, 1.0002899066897874e+28, 1.0002774826871336e+28, 1.0002655632218006e+28, 1.0002541290710994e+28, 1.0002431616894032e+28, 1.0002326431866195e+28, 1.0002225563072524e+28, 1.0002128844100434e+28, 1.0002036114481808e+28, 1.0001947219500588e+28, 1.000186201000582e+28, 1.0001780342229971e+28, 1.0001702077612426e+28, 1.0001627082628054e+28, 1.00015552286207e+28, 1.0001486391641515e+28, 1.0001420452291994e+28, 1.000135729557163e+28, 1.0001296810730036e+28, 1.0001238891123475e+28, 1.0001183434075664e+28, 1.000113034074273e+28, 1.000107951598228e+28, 1.000103086822638e+28, 1.0000984309358465e+28, 1.000093975459397e+28, 1.0000897122364657e+28, 1.0000856334206532e+28, 1.0000817314651222e+28, 1.000077999112077e+28, 1.000074429382572e+28, 1.0000710155666432e+28, 1.0000677512137518e+28, 1.0000646301235336e+28, 1.0000616463368434e+28, 1.0000587941270896e+28, 1.0000560679918473e+28, 1.0000534626447448e+28, 1.0000509730076164e+28, 1.000048594202909e+28, 1.000046321546344e+28, 1.0000441505398172e+28, 1.0000420768645391e+28, 1.0000400963744016e+28, 1.0000382050895683e+28, 1.0000363991902816e+28, 1.0000346750108783e+28, 1.0000330290340098e+28, 1.0000314578850581e+28, 1.000029958326747e+28, 1.000028527253934e+28, 1.0000271616885864e+28, 1.0000258587749309e+28, 1.000024615774771e+28, 1.0000234300629706e+28, 1.0000222991230957e+28, 1.0000212205432097e+28, 1.0000201920118182e+28, 1.0000192113139583e+28, 1.0000182763274286e+28, 1.0000173850191527e+28, 1.0000165354416755e+28, 1.0000157257297859e+28, 1.0000149540972615e+28, 1.0000142188337337e+28, 1.000013518301665e+28, 1.0000128509334411e+28, 1.000012215228567e+28, 1.0000116097509674e+28, 1.0000110331263912e+28, 1.000010484039907e+28, 1.0000099612334973e+28, 1.0000094635037413e+28, 1.0000089896995844e+28, 1.0000085387201942e+28, 1.000008109512896e+28, 1.000007701071189e+28, 1.0000073124328364e+28, 1.0000069426780308e+28, 1.0000065909276303e+28, 1.0000062563414605e+28, 1.0000059381166855e+28, 1.000005635486241e+28, 1.0000053477173286e+28, 1.0000050741099694e+28, 1.000004813995615e+28, 1.0000045667358128e+28, 1.000004331720925e+28, 1.000004108368897e+28, 1.000003896124078e+28, 1.0000036944560847e+28, 1.0000035028587152e+28, 1.000003320848903e+28, 1.0000031479657157e+28, 1.0000029837693927e+28, 1.0000028278404236e+28]
x_cell_centers_cm_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = [0.09765625, 0.29296875, 0.48828125, 0.68359375, 0.87890625, 1.07421875, 1.26953125, 1.46484375, 1.66015625, 1.85546875, 2.05078125, 2.24609375, 2.44140625, 2.63671875, 2.83203125, 3.02734375, 3.22265625, 3.41796875, 3.61328125, 3.80859375, 4.00390625, 4.19921875, 4.39453125, 4.58984375, 4.78515625, 4.98046875, 5.17578125, 5.37109375, 5.56640625, 5.76171875, 5.95703125, 6.15234375, 6.34765625, 6.54296875, 6.73828125, 6.93359375, 7.12890625, 7.32421875, 7.51953125, 7.71484375, 7.91015625, 8.10546875, 8.30078125, 8.49609375, 8.69140625, 8.88671875, 9.08203125, 9.27734375, 9.47265625, 9.66796875, 9.86328125, 10.05859375, 10.25390625, 10.44921875, 10.64453125, 10.83984375, 11.03515625, 11.23046875, 11.42578125, 11.62109375, 11.81640625, 12.01171875, 12.20703125, 12.40234375, 12.59765625, 12.79296875, 12.98828125, 13.18359375, 13.37890625, 13.57421875, 13.76953125, 13.96484375, 14.16015625, 14.35546875, 14.55078125, 14.74609375, 14.94140625, 15.13671875, 15.33203125, 15.52734375, 15.72265625, 15.91796875, 16.11328125, 16.30859375, 16.50390625, 16.69921875, 16.89453125, 17.08984375, 17.28515625, 17.48046875, 17.67578125, 17.87109375, 18.06640625, 18.26171875, 18.45703125, 18.65234375, 18.84765625, 19.04296875, 19.23828125, 19.43359375, 19.62890625, 19.82421875, 20.01953125, 20.21484375, 20.41015625, 20.60546875, 20.80078125, 20.99609375, 21.19140625, 21.38671875, 21.58203125, 21.77734375, 21.97265625, 22.16796875, 22.36328125, 22.55859375, 22.75390625, 22.94921875, 23.14453125, 23.33984375, 23.53515625, 23.73046875, 23.92578125, 24.12109375, 24.31640625, 24.51171875, 24.70703125, 24.90234375, 25.09765625, 25.29296875, 25.48828125, 25.68359375, 25.87890625, 26.07421875, 26.26953125, 26.46484375, 26.66015625, 26.85546875, 27.05078125, 27.24609375, 27.44140625, 27.63671875, 27.83203125, 28.02734375, 28.22265625, 28.41796875, 28.61328125, 28.80859375, 29.00390625, 29.19921875, 29.39453125, 29.58984375, 29.78515625, 29.98046875, 30.17578125, 30.37109375, 30.56640625, 30.76171875, 30.95703125, 31.15234375, 31.34765625, 31.54296875, 31.73828125, 31.93359375, 32.12890625, 32.32421875, 32.51953125, 32.71484375, 32.91015625, 33.10546875, 33.30078125, 33.49609375, 33.69140625, 33.88671875, 34.08203125, 34.27734375, 34.47265625, 34.66796875, 34.86328125, 35.05859375, 35.25390625, 35.44921875, 35.64453125, 35.83984375, 36.03515625, 36.23046875, 36.42578125, 36.62109375, 36.81640625, 37.01171875, 37.20703125, 37.40234375, 37.59765625, 37.79296875, 37.98828125, 38.18359375, 38.37890625, 38.57421875, 38.76953125, 38.96484375, 39.16015625, 39.35546875, 39.55078125, 39.74609375, 39.94140625, 40.13671875, 40.33203125, 40.52734375, 40.72265625, 40.91796875, 41.11328125, 41.30859375, 41.50390625, 41.69921875, 41.89453125, 42.08984375, 42.28515625, 42.48046875, 42.67578125, 42.87109375, 43.06640625, 43.26171875, 43.45703125, 43.65234375, 43.84765625, 44.04296875, 44.23828125, 44.43359375, 44.62890625, 44.82421875, 45.01953125, 45.21484375, 45.41015625, 45.60546875, 45.80078125, 45.99609375, 46.19140625, 46.38671875, 46.58203125, 46.77734375, 46.97265625, 47.16796875, 47.36328125, 47.55859375, 47.75390625, 47.94921875, 48.14453125, 48.33984375, 48.53515625, 48.73046875, 48.92578125, 49.12109375, 49.31640625, 49.51171875, 49.70703125, 49.90234375, 50.09765625, 50.29296875, 50.48828125, 50.68359375, 50.87890625, 51.07421875, 51.26953125, 51.46484375, 51.66015625, 51.85546875, 52.05078125, 52.24609375, 52.44140625, 52.63671875, 52.83203125, 53.02734375, 53.22265625, 53.41796875, 53.61328125, 53.80859375, 54.00390625, 54.19921875, 54.39453125, 54.58984375, 54.78515625, 54.98046875, 55.17578125, 55.37109375, 55.56640625, 55.76171875, 55.95703125, 56.15234375, 56.34765625, 56.54296875, 56.73828125, 56.93359375, 57.12890625, 57.32421875, 57.51953125, 57.71484375, 57.91015625, 58.10546875, 58.30078125, 58.49609375, 58.69140625, 58.88671875, 59.08203125, 59.27734375, 59.47265625, 59.66796875, 59.86328125, 60.05859375, 60.25390625, 60.44921875, 60.64453125, 60.83984375, 61.03515625, 61.23046875, 61.42578125, 61.62109375, 61.81640625, 62.01171875, 62.20703125, 62.40234375, 62.59765625, 62.79296875, 62.98828125, 63.18359375, 63.37890625, 63.57421875, 63.76953125, 63.96484375, 64.16015625, 64.35546875, 64.55078125, 64.74609375, 64.94140625, 65.13671875, 65.33203125, 65.52734375, 65.72265625, 65.91796875, 66.11328125, 66.30859375, 66.50390625, 66.69921875, 66.89453125, 67.08984375, 67.28515625, 67.48046875, 67.67578125, 67.87109375, 68.06640625, 68.26171875, 68.45703125, 68.65234375, 68.84765625, 69.04296875, 69.23828125, 69.43359375, 69.62890625, 69.82421875, 70.01953125, 70.21484375, 70.41015625, 70.60546875, 70.80078125, 70.99609375, 71.19140625, 71.38671875, 71.58203125, 71.77734375, 71.97265625, 72.16796875, 72.36328125, 72.55859375, 72.75390625, 72.94921875, 73.14453125, 73.33984375, 73.53515625, 73.73046875, 73.92578125, 74.12109375, 74.31640625, 74.51171875, 74.70703125, 74.90234375, 75.09765625, 75.29296875, 75.48828125, 75.68359375, 75.87890625, 76.07421875, 76.26953125, 76.46484375, 76.66015625, 76.85546875, 77.05078125, 77.24609375, 77.44140625, 77.63671875, 77.83203125, 78.02734375, 78.22265625, 78.41796875, 78.61328125, 78.80859375, 79.00390625, 79.19921875, 79.39453125, 79.58984375, 79.78515625, 79.98046875, 80.17578125, 80.37109375, 80.56640625, 80.76171875, 80.95703125, 81.15234375, 81.34765625, 81.54296875, 81.73828125, 81.93359375, 82.12890625, 82.32421875, 82.51953125, 82.71484375, 82.91015625, 83.10546875, 83.30078125, 83.49609375, 83.69140625, 83.88671875, 84.08203125, 84.27734375, 84.47265625, 84.66796875, 84.86328125, 85.05859375, 85.25390625, 85.44921875, 85.64453125, 85.83984375, 86.03515625, 86.23046875, 86.42578125, 86.62109375, 86.81640625, 87.01171875, 87.20703125, 87.40234375, 87.59765625, 87.79296875, 87.98828125, 88.18359375, 88.37890625, 88.57421875, 88.76953125, 88.96484375, 89.16015625, 89.35546875, 89.55078125, 89.74609375, 89.94140625, 90.13671875, 90.33203125, 90.52734375, 90.72265625, 90.91796875, 91.11328125, 91.30859375, 91.50390625, 91.69921875, 91.89453125, 92.08984375, 92.28515625, 92.48046875, 92.67578125, 92.87109375, 93.06640625, 93.26171875, 93.45703125, 93.65234375, 93.84765625, 94.04296875, 94.23828125, 94.43359375, 94.62890625, 94.82421875, 95.01953125, 95.21484375, 95.41015625, 95.60546875, 95.80078125, 95.99609375, 96.19140625, 96.38671875, 96.58203125, 96.77734375, 96.97265625, 97.16796875, 97.36328125, 97.55859375, 97.75390625, 97.94921875, 98.14453125, 98.33984375, 98.53515625, 98.73046875, 98.92578125, 99.12109375, 99.31640625, 99.51171875, 99.70703125, 99.90234375, 100.09765625, 100.29296875, 100.48828125, 100.68359375, 100.87890625, 101.07421875, 101.26953125, 101.46484375, 101.66015625, 101.85546875, 102.05078125, 102.24609375, 102.44140625, 102.63671875, 102.83203125, 103.02734375, 103.22265625, 103.41796875, 103.61328125, 103.80859375, 104.00390625, 104.19921875, 104.39453125, 104.58984375, 104.78515625, 104.98046875, 105.17578125, 105.37109375, 105.56640625, 105.76171875, 105.95703125, 106.15234375, 106.34765625, 106.54296875, 106.73828125, 106.93359375, 107.12890625, 107.32421875, 107.51953125, 107.71484375, 107.91015625, 108.10546875, 108.30078125, 108.49609375, 108.69140625, 108.88671875, 109.08203125, 109.27734375, 109.47265625, 109.66796875, 109.86328125, 110.05859375, 110.25390625, 110.44921875, 110.64453125, 110.83984375, 111.03515625, 111.23046875, 111.42578125, 111.62109375, 111.81640625, 112.01171875, 112.20703125, 112.40234375, 112.59765625, 112.79296875, 112.98828125, 113.18359375, 113.37890625, 113.57421875, 113.76953125, 113.96484375, 114.16015625, 114.35546875, 114.55078125, 114.74609375, 114.94140625, 115.13671875, 115.33203125, 115.52734375, 115.72265625, 115.91796875, 116.11328125, 116.30859375, 116.50390625, 116.69921875, 116.89453125, 117.08984375, 117.28515625, 117.48046875, 117.67578125, 117.87109375, 118.06640625, 118.26171875, 118.45703125, 118.65234375, 118.84765625, 119.04296875, 119.23828125, 119.43359375, 119.62890625, 119.82421875, 120.01953125, 120.21484375, 120.41015625, 120.60546875, 120.80078125, 120.99609375, 121.19140625, 121.38671875, 121.58203125, 121.77734375, 121.97265625, 122.16796875, 122.36328125, 122.55859375, 122.75390625, 122.94921875, 123.14453125, 123.33984375, 123.53515625, 123.73046875, 123.92578125, 124.12109375, 124.31640625, 124.51171875, 124.70703125, 124.90234375, 125.09765625, 125.29296875, 125.48828125, 125.68359375, 125.87890625, 126.07421875, 126.26953125, 126.46484375, 126.66015625, 126.85546875, 127.05078125, 127.24609375, 127.44140625, 127.63671875, 127.83203125, 128.02734375, 128.22265625, 128.41796875, 128.61328125, 128.80859375, 129.00390625, 129.19921875, 129.39453125, 129.58984375, 129.78515625, 129.98046875, 130.17578125, 130.37109375, 130.56640625, 130.76171875, 130.95703125, 131.15234375, 131.34765625, 131.54296875, 131.73828125, 131.93359375, 132.12890625, 132.32421875, 132.51953125, 132.71484375, 132.91015625, 133.10546875, 133.30078125, 133.49609375, 133.69140625, 133.88671875, 134.08203125, 134.27734375, 134.47265625, 134.66796875, 134.86328125, 135.05859375, 135.25390625, 135.44921875, 135.64453125, 135.83984375, 136.03515625, 136.23046875, 136.42578125, 136.62109375, 136.81640625, 137.01171875, 137.20703125, 137.40234375, 137.59765625, 137.79296875, 137.98828125, 138.18359375, 138.37890625, 138.57421875, 138.76953125, 138.96484375, 139.16015625, 139.35546875, 139.55078125, 139.74609375, 139.94140625, 140.13671875, 140.33203125, 140.52734375, 140.72265625, 140.91796875, 141.11328125, 141.30859375, 141.50390625, 141.69921875, 141.89453125, 142.08984375, 142.28515625, 142.48046875, 142.67578125, 142.87109375, 143.06640625, 143.26171875, 143.45703125, 143.65234375, 143.84765625, 144.04296875, 144.23828125, 144.43359375, 144.62890625, 144.82421875, 145.01953125, 145.21484375, 145.41015625, 145.60546875, 145.80078125, 145.99609375, 146.19140625, 146.38671875, 146.58203125, 146.77734375, 146.97265625, 147.16796875, 147.36328125, 147.55859375, 147.75390625, 147.94921875, 148.14453125, 148.33984375, 148.53515625, 148.73046875, 148.92578125, 149.12109375, 149.31640625, 149.51171875, 149.70703125, 149.90234375, 150.09765625, 150.29296875, 150.48828125, 150.68359375, 150.87890625, 151.07421875, 151.26953125, 151.46484375, 151.66015625, 151.85546875, 152.05078125, 152.24609375, 152.44140625, 152.63671875, 152.83203125, 153.02734375, 153.22265625, 153.41796875, 153.61328125, 153.80859375, 154.00390625, 154.19921875, 154.39453125, 154.58984375, 154.78515625, 154.98046875, 155.17578125, 155.37109375, 155.56640625, 155.76171875, 155.95703125, 156.15234375, 156.34765625, 156.54296875, 156.73828125, 156.93359375, 157.12890625, 157.32421875, 157.51953125, 157.71484375, 157.91015625, 158.10546875, 158.30078125, 158.49609375, 158.69140625, 158.88671875, 159.08203125, 159.27734375, 159.47265625, 159.66796875, 159.86328125, 160.05859375, 160.25390625, 160.44921875, 160.64453125, 160.83984375, 161.03515625, 161.23046875, 161.42578125, 161.62109375, 161.81640625, 162.01171875, 162.20703125, 162.40234375, 162.59765625, 162.79296875, 162.98828125, 163.18359375, 163.37890625, 163.57421875, 163.76953125, 163.96484375, 164.16015625, 164.35546875, 164.55078125, 164.74609375, 164.94140625, 165.13671875, 165.33203125, 165.52734375, 165.72265625, 165.91796875, 166.11328125, 166.30859375, 166.50390625, 166.69921875, 166.89453125, 167.08984375, 167.28515625, 167.48046875, 167.67578125, 167.87109375, 168.06640625, 168.26171875, 168.45703125, 168.65234375, 168.84765625, 169.04296875, 169.23828125, 169.43359375, 169.62890625, 169.82421875, 170.01953125, 170.21484375, 170.41015625, 170.60546875, 170.80078125, 170.99609375, 171.19140625, 171.38671875, 171.58203125, 171.77734375, 171.97265625, 172.16796875, 172.36328125, 172.55859375, 172.75390625, 172.94921875, 173.14453125, 173.33984375, 173.53515625, 173.73046875, 173.92578125, 174.12109375, 174.31640625, 174.51171875, 174.70703125, 174.90234375, 175.09765625, 175.29296875, 175.48828125, 175.68359375, 175.87890625, 176.07421875, 176.26953125, 176.46484375, 176.66015625, 176.85546875, 177.05078125, 177.24609375, 177.44140625, 177.63671875, 177.83203125, 178.02734375, 178.22265625, 178.41796875, 178.61328125, 178.80859375, 179.00390625, 179.19921875, 179.39453125, 179.58984375, 179.78515625, 179.98046875, 180.17578125, 180.37109375, 180.56640625, 180.76171875, 180.95703125, 181.15234375, 181.34765625, 181.54296875, 181.73828125, 181.93359375, 182.12890625, 182.32421875, 182.51953125, 182.71484375, 182.91015625, 183.10546875, 183.30078125, 183.49609375, 183.69140625, 183.88671875, 184.08203125, 184.27734375, 184.47265625, 184.66796875, 184.86328125, 185.05859375, 185.25390625, 185.44921875, 185.64453125, 185.83984375, 186.03515625, 186.23046875, 186.42578125, 186.62109375, 186.81640625, 187.01171875, 187.20703125, 187.40234375, 187.59765625, 187.79296875, 187.98828125, 188.18359375, 188.37890625, 188.57421875, 188.76953125, 188.96484375, 189.16015625, 189.35546875, 189.55078125, 189.74609375, 189.94140625, 190.13671875, 190.33203125, 190.52734375, 190.72265625, 190.91796875, 191.11328125, 191.30859375, 191.50390625, 191.69921875, 191.89453125, 192.08984375, 192.28515625, 192.48046875, 192.67578125, 192.87109375, 193.06640625, 193.26171875, 193.45703125, 193.65234375, 193.84765625, 194.04296875, 194.23828125, 194.43359375, 194.62890625, 194.82421875, 195.01953125, 195.21484375, 195.41015625, 195.60546875, 195.80078125, 195.99609375, 196.19140625, 196.38671875, 196.58203125, 196.77734375, 196.97265625, 197.16796875, 197.36328125, 197.55859375, 197.75390625, 197.94921875, 198.14453125, 198.33984375, 198.53515625, 198.73046875, 198.92578125, 199.12109375, 199.31640625, 199.51171875, 199.70703125, 199.90234375]

last_time_s_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 9.999999999999707e-09
last_nee_invccm_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = [1.0000059900988918e+28, 1.000006020219399e+28, 1.0000060805857866e+28, 1.0000061714527025e+28, 1.0000062932029084e+28, 1.0000064463487043e+28, 1.0000066315337119e+28, 1.000006849534968e+28, 1.0000071012654004e+28, 1.0000073877770791e+28, 1.0000077102644914e+28, 1.0000080700685368e+28, 1.0000084686810324e+28, 1.0000089077494864e+28, 1.000009389082429e+28, 1.0000099146554461e+28, 1.000010486617421e+28, 1.00001110729753e+28, 1.000011779212755e+28, 1.0000125050757826e+28, 1.0000132878036719e+28, 1.0000141305271146e+28, 1.000015036600261e+28, 1.0000160096110211e+28, 1.0000170533924232e+28, 1.0000181720342968e+28, 1.0000193698956997e+28, 1.0000206516184094e+28, 1.0000220221407957e+28, 1.000023486712567e+28, 1.0000250509105733e+28, 1.000026720655218e+28, 1.0000285022276748e+28, 1.000030402288466e+28, 1.0000324278965162e+28, 1.0000345865294197e+28, 1.0000368861046928e+28, 1.0000393350022557e+28, 1.000041942087527e+28, 1.0000447167363925e+28, 1.0000476688608605e+28, 1.0000508089359363e+28, 1.0000541480282328e+28, 1.0000576978254132e+28, 1.0000614706673404e+28, 1.000065479578324e+28, 1.0000697383015271e+28, 1.0000742613342636e+28, 1.000079063965076e+28, 1.0000841623125671e+28, 1.0000895733659846e+28, 1.000095315027506e+28, 1.0001014061562503e+28, 1.0001078666145587e+28, 1.0001147173159192e+28, 1.0001219802751036e+28, 1.0001296786602843e+28, 1.000137836847569e+28, 1.0001464804775145e+28, 1.0001556365142352e+28, 1.0001653333068945e+28, 1.000175600653323e+28, 1.0001864698667018e+28, 1.0001979738446198e+28, 1.0002101471410213e+28, 1.0002230260407918e+28, 1.0002366486372884e+28, 1.000251054913077e+28, 1.0002662868234503e+28, 1.0002823883831608e+28, 1.0002994057567394e+28, 1.0003173873516559e+28, 1.0003363839152775e+28, 1.0003564486353242e+28, 1.0003776372437722e+28, 1.0004000081245795e+28, 1.0004236224253554e+28, 1.000448544172796e+28, 1.000474840391997e+28, 1.0005025812303105e+28, 1.0005318400849681e+28, 1.0005626937349952e+28, 1.000595222478156e+28, 1.000629510271518e+28, 1.0006656448771407e+28, 1.0007037180124215e+28, 1.0007438255050137e+28, 1.0007860674527442e+28, 1.0008305483884907e+28, 1.000877377450176e+28, 1.000926668555692e+28, 1.0009785405833233e+28, 1.001033117557361e+28, 1.0010905288389598e+28, 1.0011509093229268e+28, 1.0012143996395492e+28, 1.0012811463623313e+28, 1.0013513022215446e+28, 1.0014250263232119e+28, 1.0015024843742075e+28, 1.0015838489132158e+28, 1.0016692995475367e+28, 1.0017590231958772e+28, 1.0018532143374244e+28, 1.0019520752667224e+28, 1.0020558163547002e+28, 1.0021646563161527e+28, 1.0022788224829484e+28, 1.002398551083603e+28, 1.0025240875292568e+28, 1.0026556867054236e+28, 1.002793613270014e+28, 1.0029381419578635e+28, 1.0030895578909902e+28, 1.003248156894811e+28, 1.003414245820919e+28, 1.0035881428753872e+28, 1.003770177953115e+28, 1.0039606929779918e+28, 1.004160042248792e+28, 1.0043685927905921e+28, 1.0045867247120504e+28, 1.0048148315675267e+28, 1.0050533207244568e+28, 1.0053026137367138e+28, 1.0055631467213958e+28, 1.0058353707414181e+28, 1.0061197521924184e+28, 1.0064167731935738e+28, 1.0067269319828236e+28, 1.0070507433157731e+28, 1.0073887388678316e+28, 1.007741467640159e+28, 1.008109496368303e+28, 1.0084934099330126e+28, 1.008893811773955e+28, 1.009311324305034e+28, 1.0097465893312392e+28, 1.0102002684667108e+28, 1.0106730435535035e+28, 1.0111656170804143e+28, 1.0116787126019602e+28, 1.0122130751564928e+28, 1.0127694716829407e+28, 1.0133486914363432e+28, 1.0139515464007868e+28, 1.0145788716993507e+28, 1.015231526001239e+28, 1.0159103919244585e+28, 1.0166163764338014e+28, 1.017350411233876e+28, 1.0181134531560612e+28, 1.0189064845384303e+28, 1.0197305135987734e+28, 1.0205865747994398e+28, 1.0214757292027527e+28, 1.0223990648174284e+28, 1.0233576969344772e+28, 1.0243527684509465e+28, 1.0253854501826585e+28, 1.0264569411633314e+28, 1.0275684689297394e+28, 1.0287212897923763e+28, 1.0299166890904855e+28, 1.0311559814298866e+28, 1.0324405109035721e+28, 1.0337716512942046e+28, 1.0351508062559917e+28, 1.0365794094768032e+28, 1.0380589248186306e+28, 1.0395908464349442e+28, 1.0411766988651345e+28, 1.0428180371034838e+28, 1.044516446643152e+28, 1.046273543492896e+28, 1.0480909741659556e+28, 1.049970415639899e+28, 1.051913575286747e+28, 1.0539221907716146e+28, 1.0559980299191948e+28, 1.0581428905471544e+28, 1.060358600264906e+28, 1.0626470162367769e+28, 1.0650100249091042e+28, 1.0674495416993262e+28, 1.0699675106462162e+28, 1.0725659040210741e+28, 1.0752467218974524e+28, 1.0780119916792312e+28, 1.0808637675864874e+28, 1.08380413009675e+28, 1.0868351853419564e+28, 1.0899590644597984e+28, 1.0931779228983012e+28, 1.0964939396728935e+28, 1.0999093165753097e+28, 1.1034262773333474e+28, 1.1070470667202254e+28, 1.1107739496140187e+28, 1.1146092100051178e+28, 1.1185551499516351e+28, 1.1226140884824822e+28, 1.1267883604468306e+28, 1.1310803153098611e+28, 1.1354923158944379e+28, 1.1400267370677522e+28, 1.144685964373225e+28, 1.1494723926068483e+28, 1.1543884243380538e+28, 1.1594364683747548e+28, 1.1646189381725336e+28, 1.169938250187907e+28, 1.175396822175624e+28, 1.1809970714303372e+28, 1.1867414129725568e+28, 1.1926322576791854e+28, 1.1986720103593076e+28, 1.2048630677750899e+28, 1.2112078166087266e+28, 1.2177086313760227e+28, 1.2243678722868477e+28, 1.2311878830538573e+28, 1.2381709886498686e+28, 1.2453194930149094e+28, 1.2526356767142441e+28, 1.2601217945482997e+28, 1.2677800731156867e+28, 1.2756127083306713e+28, 1.2836218628967212e+28, 1.291809663737681e+28, 1.3001781993873714e+28, 1.308729517340826e+28, 1.3174656213677355e+28, 1.326388468790262e+28, 1.335499967728012e+28, 1.3448019743113676e+28, 1.354296289865636e+28, 1.3639846580687965e+28, 1.373868762084925e+28, 1.3839502216755455e+28, 1.3942305902920558e+28, 1.4047113521518478e+28, 1.4153939193003218e+28, 1.4262796286624428e+28, 1.4373697390868448e+28, 1.44866542838442e+28, 1.4601677903664968e+28, 1.471877831883666e+28, 1.483796469870519e+28, 1.4959245283979572e+28, 1.5082627357386885e+28, 1.5208117214470098e+28, 1.5335720134577643e+28, 1.546544035209506e+28, 1.5597281027923883e+28, 1.5731244221270632e+28, 1.586733086178641e+28, 1.6005540722070872e+28, 1.6145872390606112e+28, 1.628832324515533e+28, 1.6432889426649534e+28, 1.6579565813619078e+28, 1.672834599721611e+28, 1.6879222256844382e+28, 1.7032185536457232e+28, 1.7187225421567599e+28, 1.7344330116992477e+28, 1.7503486425382427e+28, 1.7664679726593255e+28, 1.7827893957904935e+28, 1.7993111595157536e+28, 1.816031363484458e+28, 1.8329479577176439e+28, 1.8500587410176534e+28, 1.8673613594854828e+28, 1.884853305146957e+28, 1.9025319146932413e+28, 1.9203943683406386e+28, 1.9384376888102014e+28, 1.9566587404322183e+28, 1.9750542283809163e+28, 1.9936206980390043e+28, 2.0123545344973456e+28, 2.0312519621941907e+28, 2.050309044693666e+28, 2.069521684608455e+28, 2.088885623670741e+28, 2.1083964429504735e+28, 2.1280495632253004e+28, 2.1478402455068897e+28, 2.167763591720534e+28, 2.187814545543858e+28, 2.2079878934067243e+28, 2.2282782656511135e+28, 2.2486801378537193e+28, 2.269187832315556e+28, 2.2897955197137347e+28, 2.3104972209207718e+28, 2.3312868089922527e+28, 2.3521580113199053e+28, 2.3731044119525026e+28, 2.3941194540871295e+28, 2.415196442725358e+28, 2.43632854749688e+28, 2.457508805652457e+28, 2.4787301252201854e+28, 2.499985288326792e+28, 2.5212669546852152e+28, 2.5425676652411406e+28, 2.563879845981179e+28, 2.585195811901239e+28, 2.6065077711286726e+28, 2.6278078291990364e+28, 2.649087993486514e+28, 2.6703401777794497e+28, 2.6915562070024215e+28, 2.7127278220828628e+28, 2.7338466849521485e+28, 2.7549043836835925e+28, 2.7758924377625733e+28, 2.7968023034804317e+28, 2.8176253794518455e+28, 2.838353012251562e+28, 2.8589765021604025e+28, 2.8794871090209995e+28, 2.899876058197648e+28, 2.920134546630093e+28, 2.940253748979813e+28, 2.960224823865049e+28, 2.9800389201712345e+28, 2.999687183437214e+28, 3.019160762310715e+28, 3.038450815061734e+28, 3.057548516151886e+28, 3.0764450628531963e+28, 3.0951316819052395e+28, 3.113599636206926e+28, 3.1318402315386375e+28, 3.1498448233001687e+28, 3.167604823262811e+28, 3.1851117063293624e+28, 3.202357017288523e+28, 3.2193323775617594e+28, 3.236029491935325e+28, 3.252440155264558e+28, 3.26855625914785e+28, 3.2843697985633606e+28, 3.299872878455553e+28, 3.3150577202689315e+28, 3.329916668421108e+28, 3.3444421967038704e+28, 3.358626914608717e+28, 3.372463573569321e+28, 3.3859450731102688e+28, 3.3990644668966843e+28, 3.411814968681431e+28, 3.4241899581347732e+28, 3.4361829865548843e+28, 3.447787782454607e+28, 3.4589982570095495e+28, 3.469808509368954e+28, 3.4802128318192478e+28, 3.490205714793073e+28, 3.499781851719607e+28, 3.5089361437114613e+28, 3.5176637040777573e+28, 3.525959862660885e+28, 3.533820169993833e+28, 3.5412404012677005e+28, 3.548216560106957e+28, 3.5547448821507314e+28, 3.560821838430096e+28, 3.5664441385377173e+28, 3.571608733595419e+28, 3.5763128190012322e+28, 3.5805538369615944e+28, 3.584329478806612e+28, 3.587637687077994e+28, 3.59047665739386e+28, 3.592844840087213e+28, 3.594740941610901e+28, 3.5961639257123836e+28, 3.5971130143777238e+28, 3.597587688537399e+28, 3.597587688537532e+28, 3.597113014378821e+28, 3.596163925714262e+28, 3.594740941613024e+28, 3.59284484009031e+28, 3.590476657397948e+28, 3.587637687082341e+28, 3.5843294788119102e+28, 3.5805538369679025e+28, 3.576312819007831e+28, 3.571608733602967e+28, 3.5664441385462518e+28, 3.5608218384392367e+28, 3.5547448821613874e+28, 3.54821656011838e+28, 3.5412404012794363e+28, 3.5338201700063137e+28, 3.525959862674161e+28, 3.5176637040913266e+28, 3.508936143725749e+28, 3.499781851734353e+28, 3.490205714807789e+28, 3.4802128318343273e+28, 3.4698085093842976e+28, 3.4589982570245438e+28, 3.4477877824695296e+28, 3.436182986569903e+28, 3.4241899581494156e+28, 3.411814968696344e+28, 3.399064466911573e+28, 3.385945073124706e+28, 3.3724635735839368e+28, 3.358626914623274e+28, 3.3444421967179547e+28, 3.329916668435144e+28, 3.3150577202829573e+28, 3.299872878469003e+28, 3.28436979857665e+28, 3.2685562591610818e+28, 3.252440155277129e+28, 3.2360294919477177e+28, 3.2193323775738716e+28, 3.2023570172997974e+28, 3.1851117063404432e+28, 3.167604823273755e+28, 3.149844823310464e+28, 3.1318402315488972e+28, 3.1135996362170767e+28, 3.0951316819150326e+28, 3.0764450628633197e+28, 3.057548516162008e+28, 3.0384508150713397e+28, 3.019160762320151e+28, 2.9996871834465475e+28, 2.9800389201799474e+28, 2.960224823873561e+28, 2.9402537489880426e+28, 2.9201345466374793e+28, 2.8998760582048703e+28, 2.8794871090278916e+28, 2.8589765021665044e+28, 2.838353012257479e+28, 2.817625379457513e+28, 2.796802303485091e+28, 2.7758924377667207e+28, 2.7549043836874487e+28, 2.733846684955208e+28, 2.71272782208567e+28, 2.691556207004993e+28, 2.6703401777811764e+28, 2.649087993488e+28, 2.6278078292002235e+28, 2.606507771129027e+28, 2.585195811901281e+28, 2.563879845980899e+28, 2.542567665239977e+28, 2.5212669546836442e+28, 2.499985288324789e+28, 2.4787301252172106e+28, 2.457508805648963e+28, 2.436328547492799e+28, 2.41519644272031e+28, 2.394119454081535e+28, 2.3731044119461457e+28, 2.3521580113124783e+28, 2.331286808984231e+28, 2.3104972209119985e+28, 2.289795519703766e+28, 2.2691878323047503e+28, 2.24868013784201e+28, 2.2282782656380926e+28, 2.207987893392864e+28, 2.187814545529035e+28, 2.167763591704439e+28, 2.1478402454898173e+28, 2.128049563207186e+28, 2.108396442930939e+28, 2.088885623650131e+28, 2.0695216845867093e+28, 2.050309044670405e+28, 2.031251962169681e+28, 2.0123545344715498e+28, 1.993620698011588e+28, 1.975054228352126e+28, 1.9566587404020858e+28, 1.938437688778357e+28, 1.9203943683073161e+28, 1.9025319146576892e+28, 1.8848533051088297e+28, 1.8673613594461497e+28, 1.8500587409768796e+28, 1.8329479576751838e+28, 1.8160313634409915e+28, 1.7993111594709854e+28, 1.7827893957441462e+28, 1.7664679726120604e+28, 1.7503486424899238e+28, 1.73443301164953e+28, 1.718722542106325e+28, 1.703218553594264e+28, 1.6879222256318167e+28, 1.6728345996683428e+28, 1.657956581307758e+28, 1.6432889426096902e+28, 1.6288323244597372e+28, 1.6145872390040255e+28, 1.6005540721495058e+28, 1.5867330861206238e+28, 1.5731244220683902e+28, 1.5597281027327605e+28, 1.5465440351495205e+28, 1.5335720133972212e+28, 1.5208117213856393e+28, 1.5082627356771256e+28, 1.4959245283358475e+28, 1.4837964698076698e+28, 1.47187783182056e+28, 1.4601677903028885e+28, 1.4486654283201788e+28, 1.4373697390223143e+28, 1.4262796285974915e+28, 1.4153939192347394e+28, 1.404711352086101e+28, 1.3942305902259028e+28, 1.3839502216087612e+28, 1.3738687620179648e+28, 1.3639846580015374e+28, 1.3542962897979345e+28, 1.3448019742436434e+28, 1.3354999676600477e+28, 1.3263884687219103e+28, 1.3174656212993272e+28, 1.3087295172722523e+28, 1.300178199318457e+28, 1.2918096636687767e+28, 1.2836218628276938e+28, 1.275612708261274e+28, 1.2677800730463473e+28, 1.2601217944788359e+28, 1.2526356766444632e+28, 1.2453194929451476e+28, 1.2381709885799975e+28, 1.2311878829837892e+28, 1.2243678722168442e+28, 1.2177086313059214e+28, 1.2112078165384793e+28, 1.2048630677049397e+28, 1.1986720102891154e+28, 1.1926322576088417e+28, 1.1867414129023382e+28, 1.1809970713601543e+28, 1.1753968221053037e+28, 1.1699382501177474e+28, 1.164618938102418e+28, 1.1594364683045374e+28, 1.1543884242680116e+28, 1.1494723925368336e+28, 1.1446859643031302e+28, 1.1400267369978552e+28, 1.135492315824579e+28, 1.1310803152399302e+28, 1.1267883603770987e+28, 1.1226140884128085e+28, 1.1185551498819193e+28, 1.1146092099355946e+28, 1.1107739495445683e+28, 1.1070470666507092e+28, 1.1034262772640232e+28, 1.0999093165061148e+28, 1.0964939396036086e+28, 1.0931779228292119e+28, 1.089959064390802e+28, 1.0868351852729196e+28, 1.0838041300279778e+28, 1.080863767517789e+28, 1.0780119916104878e+28, 1.0752467218289095e+28, 1.0725659039526254e+28, 1.069967510577766e+28, 1.0674495416310637e+28, 1.0650100248410033e+28, 1.0626470161686742e+28, 1.0603586001970231e+28, 1.0581428904794636e+28, 1.0559980298515517e+28, 1.053922190704232e+28, 1.051913575219487e+28, 1.0499704155726983e+28, 1.048090974099009e+28, 1.0462735434261025e+28, 1.0445164465763984e+28, 1.0428180370369987e+28, 1.0411766987988377e+28, 1.039590846368784e+28, 1.0380589247527822e+28, 1.0365794094111366e+28, 1.0351508061903476e+28, 1.0337716512288279e+28, 1.0324405108383293e+28, 1.0311559813646776e+28, 1.029916689025661e+28, 1.0287212897277235e+28, 1.0275684688651092e+28, 1.0264569410990062e+28, 1.0253854501185216e+28, 1.0243527683868274e+28, 1.0233576968706076e+28, 1.0223990647537949e+28, 1.0214757291392058e+28, 1.0205865747361365e+28, 1.0197305135355918e+28, 1.0189064844751963e+28, 1.018113453093015e+28, 1.0173504111708932e+28, 1.0166163763707064e+28, 1.015910391861477e+28, 1.0152315259382892e+28, 1.0145788716362416e+28, 1.0139515463377439e+28, 1.0133486913732686e+28, 1.0127694716196591e+28, 1.0122130750931886e+28, 1.0116787125386055e+28, 1.0111656170168142e+28, 1.0106730434898625e+28, 1.010200268402912e+28, 1.0097465892671559e+28, 1.0093113242408518e+28, 1.0088938117095384e+28, 1.0084934098682578e+28, 1.0081094963033883e+28, 1.007741467574921e+28, 1.007388738802099e+28, 1.0070507432498231e+28, 1.0067269319165913e+28, 1.006416773126891e+28, 1.0061197521254948e+28, 1.0058353706741483e+28, 1.0055631466536563e+28, 1.0053026136687387e+28, 1.005053320656025e+28, 1.0048148314983307e+28, 1.0045867246424165e+28, 1.0043685927201288e+28, 1.0041600421773608e+28, 1.0039606929060072e+28, 1.0037701778805474e+28, 1.003588142802224e+28, 1.0034142457473597e+28, 1.0032481568208074e+28, 1.0030895578165045e+28, 1.0029381418831563e+28, 1.0027936131948315e+28, 1.0026556866297631e+28, 1.0025240874533746e+28, 1.0023985510073454e+28, 1.0022788224061778e+28, 1.0021646562391521e+28, 1.0020558162773232e+28, 1.0019520751888798e+28, 1.0018532142593857e+28, 1.00175902311751e+28, 1.0016692994687551e+28, 1.0015838488342682e+28, 1.0015024842949574e+28, 1.0014250262435578e+28, 1.0013513021417556e+28, 1.0012811462823172e+28, 1.0012143995591927e+28, 1.0011509092424874e+28, 1.0010905287582713e+28, 1.0010331174763104e+28, 1.0009785405022473e+28, 1.0009266684743802e+28, 1.0008773773685221e+28, 1.000830548306798e+28, 1.0007860673708426e+28, 1.000743825422886e+28, 1.0007037179302693e+28, 1.0006656447948605e+28, 1.0006295101890288e+28, 1.0005952223956972e+28, 1.0005626936524694e+28, 1.0005318400022426e+28, 1.0005025811477432e+28, 1.0004748403094431e+28, 1.0004485440901247e+28, 1.0004236223428736e+28, 1.0004000080421267e+28, 1.0003776371612917e+28, 1.0003564485531219e+28, 1.0003363838331638e+28, 1.0003173872695866e+28, 1.0002994056749786e+28, 1.0002823883015852e+28, 1.0002662867419707e+28, 1.000251054831972e+28, 1.000236648556437e+28, 1.0002230259600705e+28, 1.0002101470607666e+28, 1.0001979737646651e+28, 1.0001864697869848e+28, 1.000175600574129e+28, 1.0001653332280963e+28, 1.0001556364357583e+28, 1.000146480399547e+28, 1.0001378367699886e+28, 1.00012967858309e+28, 1.0001219801985156e+28, 1.0001147172398546e+28, 1.0001078665389306e+28, 1.0001014060813436e+28, 1.0000953149532108e+28, 1.0000895732922459e+28, 1.000084162239618e+28, 1.0000790638927954e+28, 1.0000742612625726e+28, 1.0000697382306867e+28, 1.000065479508151e+28, 1.0000614705978e+28, 1.0000576977569236e+28, 1.0000541479605904e+28, 1.0000508088690215e+28, 1.0000476687949476e+28, 1.000044716671351e+28, 1.0000419420232734e+28, 1.0000393349391063e+28, 1.0000368860425445e+28, 1.0000345864681633e+28, 1.000032427836441e+28, 1.0000304022294554e+28, 1.0000285021696441e+28, 1.0000267205984954e+28, 1.0000250508550334e+28, 1.0000234866581104e+28, 1.0000220220877154e+28, 1.000020651566595e+28, 1.0000193698450522e+28, 1.0000181719851062e+28, 1.0000170533446068e+28, 1.0000160095644768e+28, 1.0000150365552713e+28, 1.0000141304836155e+28, 1.0000132877615586e+28, 1.000012505035357e+28, 1.000011779173942e+28, 1.0000111072602513e+28, 1.000010486581938e+28, 1.0000099146216524e+28, 1.0000093890503005e+28, 1.0000089077192883e+28, 1.0000084686526773e+28, 1.000008070041962e+28, 1.000007710239988e+28, 1.0000073877545713e+28, 1.0000071012448356e+28, 1.0000068495166936e+28, 1.000006631517622e+28, 1.0000064463346805e+28, 1.0000062931912668e+28, 1.0000061714433437e+28, 1.000006080578673e+28, 1.0000060202148502e+28, 1.0000059900971183e+28]
last_analytic_solution_invccm_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = [1.0000028532716783e+28, 1.0000030648083796e+28, 1.0000032914133786e+28, 1.0000035341130411e+28, 1.00000379400017e+28, 1.000004072238057e+28, 1.000004370064766e+28, 1.000004688797658e+28, 1.0000050298381704e+28, 1.0000053946768655e+28, 1.0000057848987579e+28, 1.00000620218894e+28, 1.0000066483385176e+28, 1.0000071252508694e+28, 1.0000076349482532e+28, 1.000008179578767e+28, 1.0000087614236908e+28, 1.0000093829052215e+28, 1.0000100465946249e+28, 1.00001075522082e+28, 1.0000115116794217e+28, 1.0000123190422589e+28, 1.0000131805673944e+28, 1.0000140997096673e+28, 1.0000150801317844e+28, 1.0000161257159851e+28, 1.0000172405763068e+28, 1.0000184290714775e+28, 1.0000196958184652e+28, 1.0000210457067132e+28, 1.000022483913092e+28, 1.0000240159176006e+28, 1.0000256475198486e+28, 1.0000273848563557e+28, 1.0000292344187017e+28, 1.0000312030725668e+28, 1.0000332980776976e+28, 1.0000355271088388e+28, 1.0000378982776735e+28, 1.0000404201558123e+28, 1.0000431017988749e+28, 1.0000459527717097e+28, 1.000048983174799e+28, 1.0000522036718956e+28, 1.0000556255189425e+28, 1.000059260594325e+28, 1.0000631214305104e+28, 1.0000672212471243e+28, 1.0000715739855283e+28, 1.0000761943449455e+28, 1.0000810978202034e+28, 1.0000863007411467e+28, 1.000091820313788e+28, 1.0000976746632582e+28, 1.0001038828786215e+28, 1.0001104650596257e+28, 1.0001174423654532e+28, 1.0001248370655469e+28, 1.0001326725925817e+28, 1.0001409735976568e+28, 1.0001497660077844e+28, 1.0001590770857518e+28, 1.0001689354924404e+28, 1.0001793713516751e+28, 1.0001904163176948e+28, 1.0002021036453233e+28, 1.0002144682629268e+28, 1.000227546848249e+28, 1.0002413779072085e+28, 1.0002560018557503e+28, 1.0002714611048452e+28, 1.0002878001487267e+28, 1.0003050656564634e+28, 1.0003233065669587e+28, 1.0003425741874788e+28, 1.0003629222958046e+28, 1.0003844072461038e+28, 1.0004070880786305e+28, 1.000431026633343e+28, 1.000456287667551e+28, 1.0004829389776844e+28, 1.000511051525294e+28, 1.0005406995673826e+28, 1.0005719607911682e+28, 1.0006049164533866e+28, 1.0006396515242307e+28, 1.0006762548360335e+28, 1.0007148192367923e+28, 1.0007554417486398e+28, 1.0007982237313591e+28, 1.000843271051043e+28, 1.0008906942539962e+28, 1.0009406087459768e+28, 1.0009931349768703e+28, 1.0010483986308936e+28, 1.0011065308224124e+28, 1.0011676682974684e+28, 1.0012319536410952e+28, 1.0012995354905094e+28, 1.001370568754254e+28, 1.001445214837372e+28, 1.0015236418726762e+28, 1.0016060249581925e+28, 1.0016925464008276e+28, 1.0017833959663276e+28, 1.001878771135577e+28, 1.0019788773672842e+28, 1.002083928367095e+28, 1.0021941463631681e+28, 1.0023097623882416e+28, 1.002431016568207e+28, 1.0025581584172085e+28, 1.0026914471392658e+28, 1.002831151936423e+28, 1.0029775523234057e+28, 1.0031309384487644e+28, 1.0032916114224749e+28, 1.003459883649951e+28, 1.0036360791724141e+28, 1.0038205340135603e+28, 1.004013596532443e+28, 1.0042156277824855e+28, 1.0044270018765229e+28, 1.0046481063577545e+28, 1.0048793425764843e+28, 1.0051211260725029e+28, 1.0053738869629534e+28, 1.0056380703355138e+28, 1.005914136646701e+28, 1.0062025621250988e+28, 1.006503839179283e+28, 1.0068184768102126e+28, 1.0071470010278237e+28, 1.0074899552715601e+28, 1.007847900834542e+28, 1.0082214172910679e+28, 1.0086111029271116e+28, 1.0090175751734732e+28, 1.009441471041205e+28, 1.0098834475589269e+28, 1.0103441822116152e+28, 1.0108243733804341e+28, 1.0113247407831531e+28, 1.0118460259146722e+28, 1.0123889924871562e+28, 1.0129544268692564e+28, 1.0135431385238717e+28, 1.0141559604438843e+28, 1.0147937495852744e+28, 1.0154573872970025e+28, 1.0161477797470183e+28, 1.0168658583437348e+28, 1.0176125801522803e+28, 1.018388928304821e+28, 1.0191959124042176e+28, 1.0200345689202623e+28, 1.020905961577711e+28, 1.0218111817353118e+28, 1.0227513487549982e+28, 1.0237276103603993e+28, 1.024741142983794e+28, 1.0257931521006134e+28, 1.0268848725505734e+28, 1.0280175688445037e+28, 1.0291925354559079e+28, 1.0304110970962791e+28, 1.0316746089731713e+28, 1.0329844570300072e+28, 1.0343420581665867e+28, 1.0357488604392424e+28, 1.0372063432395705e+28, 1.0387160174506532e+28, 1.04027942557967e+28, 1.0418981418657852e+28, 1.0435737723621837e+28, 1.0453079549911207e+28, 1.0471023595708325e+28, 1.0489586878131588e+28, 1.0508786732907096e+28, 1.0528640813724084e+28, 1.0549167091262405e+28, 1.0570383851880313e+28, 1.059230969595078e+28, 1.0614963535834603e+28, 1.0638364593478604e+28, 1.066253239762721e+28, 1.0687486780635866e+28, 1.0713247874874732e+28, 1.0739836108711254e+28, 1.0767272202060356e+28, 1.0795577161491138e+28, 1.082477227487907e+28, 1.0854879105593018e+28, 1.088591948620651e+28, 1.0917915511722972e+28, 1.0950889532304937e+28, 1.09848641454975e+28, 1.1019862187936615e+28, 1.1055906726533182e+28, 1.1093021049124288e+28, 1.1131228654583243e+28, 1.1170553242380652e+28, 1.1211018701589063e+28, 1.1252649099324287e+28, 1.1295468668617007e+28, 1.1339501795708763e+28, 1.1384773006767028e+28, 1.143130695401465e+28, 1.147912840126954e+28, 1.1528262208891173e+28, 1.1578733318131084e+28, 1.1630566734885307e+28, 1.1683787512847348e+28, 1.1738420736061103e+28, 1.1794491500873908e+28, 1.1852024897290632e+28, 1.1911045989730645e+28, 1.197157979719031e+28, 1.2033651272814467e+28, 1.2097285282881384e+28, 1.2162506585206468e+28, 1.2229339806971038e+28, 1.2297809421983348e+28, 1.2367939727380131e+28, 1.2439754819777839e+28, 1.251327857088377e+28, 1.2588534602578447e+28, 1.2665546261481398e+28, 1.2744336593013789e+28, 1.2824928314972319e+28, 1.2907343790629847e+28, 1.2991605001379369e+28, 1.3077733518938983e+28, 1.3165750477136704e+28, 1.3255676543294918e+28, 1.3347531889235526e+28, 1.3441336161927866e+28, 1.3537108453802594e+28, 1.3634867272755811e+28, 1.373463051186887e+28, 1.38364154188703e+28, 1.3940238565367404e+28, 1.4046115815876168e+28, 1.4154062296679133e+28, 1.4264092364541902e+28, 1.437621957532007e+28, 1.4490456652489195e+28, 1.4606815455631568e+28, 1.472530694891433e+28, 1.4845941169594556e+28, 1.4968727196587724e+28, 1.5093673119136774e+28, 1.522078600561998e+28, 1.5350071872536506e+28, 1.5481535653709204e+28, 1.5615181169745113e+28, 1.5751011097794659e+28, 1.5889026941651044e+28, 1.6029229002232232e+28, 1.617161634848813e+28, 1.6316186788776066e+28, 1.646293684274835e+28, 1.661186171379577e+28, 1.6762955262091163e+28, 1.6916209978277777e+28, 1.7071616957846954e+28, 1.7229165876249836e+28, 1.7388844964788129e+28, 1.755064098732859e+28, 1.7714539217885933e+28, 1.7880523419118875e+28, 1.8048575821783516e+28, 1.8218677105188159e+28, 1.8390806378693246e+28, 1.856494116429966e+28, 1.874105738036799e+28, 1.8919129326511046e+28, 1.9099129669700938e+28, 1.928102943163142e+28, 1.9464797977375422e+28, 1.9650403005376785e+28, 1.9837810538813983e+28, 2.0026984918372996e+28, 2.02178887964651e+28, 2.0410483132923957e+28, 2.060472719221562e+28, 2.0800578542193245e+28, 2.0997993054426966e+28, 2.1196924906138115e+28, 2.1397326583765174e+28, 2.159914888818708e+28, 2.1802340941628254e+28, 2.2006850196267437e+28, 2.221262244457075e+28, 2.241960183136769e+28, 2.2627730867686456e+28, 2.2836950446363157e+28, 2.304719985943735e+28, 2.3258416817344264e+28, 2.347053746991157e+28, 2.3683496429166847e+28, 2.3897226793959054e+28, 2.4111660176395333e+28, 2.4326726730091963e+28, 2.4542355180236e+28, 2.4758472855451376e+28, 2.497500572146125e+28, 2.5191878416535572e+28, 2.5409014288710187e+28, 2.562633543476178e+28, 2.5843762740919985e+28, 2.606121592529544e+28, 2.6278613582000247e+28, 2.6495873226934734e+28, 2.67129113452114e+28, 2.6929643440185064e+28, 2.71459840840553e+28, 2.7361846970004475e+28, 2.757714496583299e+28, 2.779179016905013e+28, 2.8005693963376634e+28, 2.8218767076613173e+28, 2.8430919639825784e+28, 2.8642061247797693e+28, 2.8852101020694334e+28, 2.9060947666886034e+28, 2.9268509546871076e+28, 2.947469473823951e+28, 2.967941110161582e+28, 2.9882566347517172e+28, 3.0084068104061557e+28, 3.0283823985458485e+28, 3.0481741661213317e+28, 3.0677728925974535e+28, 3.0871693769951572e+28, 3.106354444982967e+28, 3.1253189560106694e+28, 3.14405381047754e+28, 3.1625499569273895e+28, 3.1807983992625645e+28, 3.1987902039689433e+28, 3.216516507343913e+28, 3.233968522719215e+28, 3.251137547670476e+28, 3.2680149712052435e+28, 3.284592280921234e+28, 3.3008610701265463e+28, 3.3168130449135298e+28, 3.332440031178033e+28, 3.3477339815757315e+28, 3.3626869824073008e+28, 3.3772912604242028e+28, 3.3915391895469187e+28, 3.4054232974875267e+28, 3.4189362722685906e+28, 3.432070968630398e+28, 3.444820414318732e+28, 3.4571778162454205e+28, 3.469136566514047e+28, 3.480690248303376e+28, 3.491832641601139e+28, 3.5025577287810294e+28, 3.5128597000159062e+28, 3.5227329585203987e+28, 3.532172125616281e+28, 3.541172045614214e+28, 3.5497277905056495e+28, 3.557834664458915e+28, 3.5654882081137507e+28, 3.572684202668785e+28, 3.579418673756721e+28, 3.5856878951022345e+28, 3.591488391957894e+28, 3.596816944313637e+28, 3.6016705898756808e+28, 3.6060466268109953e+28, 3.6099426162537827e+28, 3.6133563845707037e+28, 3.616286025381916e+28, 3.618729901335283e+28, 3.620686645631448e+28, 3.6221551632977798e+28, 3.6231346322095343e+28, 3.6236245038568975e+28, 3.6236245038568975e+28, 3.6231346322095343e+28, 3.6221551632977798e+28, 3.620686645631448e+28, 3.618729901335284e+28, 3.616286025381917e+28, 3.6133563845707037e+28, 3.6099426162537827e+28, 3.606046626810996e+28, 3.6016705898756816e+28, 3.596816944313638e+28, 3.591488391957895e+28, 3.5856878951022354e+28, 3.579418673756722e+28, 3.5726842026687866e+28, 3.5654882081137516e+28, 3.557834664458916e+28, 3.5497277905056504e+28, 3.5411720456142155e+28, 3.532172125616282e+28, 3.5227329585204005e+28, 3.512859700015908e+28, 3.5025577287810303e+28, 3.4918326416011415e+28, 3.4806902483033785e+28, 3.469136566514049e+28, 3.457177816245423e+28, 3.444820414318735e+28, 3.432070968630399e+28, 3.4189362722685924e+28, 3.4054232974875294e+28, 3.3915391895469196e+28, 3.3772912604242045e+28, 3.3626869824073034e+28, 3.3477339815757333e+28, 3.3324400311780356e+28, 3.3168130449135315e+28, 3.300861070126548e+28, 3.284592280921237e+28, 3.268014971205246e+28, 3.2511375476704777e+28, 3.2339685227192175e+28, 3.2165165073439155e+28, 3.198790203968945e+28, 3.180798399262568e+28, 3.162549956927393e+28, 3.1440538104775425e+28, 3.125318956010672e+28, 3.1063544449829697e+28, 3.087169376995159e+28, 3.067772892597457e+28, 3.0481741661213352e+28, 3.0283823985458503e+28, 3.008406810406159e+28, 2.9882566347517207e+28, 2.967941110161584e+28, 2.9474694738239545e+28, 2.926850954687111e+28, 2.9060947666886056e+28, 2.885210102069437e+28, 2.864206124779773e+28, 2.84309196398258e+28, 2.821876707661321e+28, 2.800569396337667e+28, 2.779179016905015e+28, 2.7577144965833025e+28, 2.7361846970004497e+28, 2.7145984084055336e+28, 2.6929643440185104e+28, 2.6712911345211416e+28, 2.649587322693477e+28, 2.627861358200028e+28, 2.606121592529546e+28, 2.5843762740920024e+28, 2.562633543476181e+28, 2.5409014288710205e+28, 2.5191878416535608e+28, 2.497500572146129e+28, 2.47584728554514e+28, 2.454235518023604e+28, 2.4326726730092e+28, 2.4111660176395355e+28, 2.389722679395909e+28, 2.368349642916688e+28, 2.3470537469911595e+28, 2.3258416817344295e+28, 2.3047199859437387e+28, 2.2836950446363175e+28, 2.262773086768649e+28, 2.2419601831367724e+28, 2.2212622444570774e+28, 2.2006850196267472e+28, 2.1802340941628284e+28, 2.15991488881871e+28, 2.1397326583765204e+28, 2.119692490613815e+28, 2.099799305442699e+28, 2.080057854219328e+28, 2.0604727192215653e+28, 2.041048313292398e+28, 2.021788879646513e+28, 2.0026984918373027e+28, 1.9837810538814e+28, 1.9650403005376816e+28, 1.9464797977375453e+28, 1.928102943163144e+28, 1.9099129669700969e+28, 1.8919129326511076e+28, 1.8741057380368011e+28, 1.8564941164299668e+28, 1.8390806378693266e+28, 1.8218677105188176e+28, 1.8048575821783525e+28, 1.7880523419118892e+28, 1.7714539217885953e+28, 1.7550640987328596e+28, 1.7388844964788146e+28, 1.7229165876249853e+28, 1.7071616957846963e+28, 1.6916209978277795e+28, 1.676295526209118e+28, 1.6611861713795778e+28, 1.6462936842748368e+28, 1.631618678877608e+28, 1.6171616348488138e+28, 1.6029229002232247e+28, 1.588902694165106e+28, 1.5751011097794665e+28, 1.561518116974513e+28, 1.5481535653709217e+28, 1.5350071872536515e+28, 1.5220786005619994e+28, 1.509367311913679e+28, 1.4968727196587733e+28, 1.4845941169594567e+28, 1.472530694891434e+28, 1.4606815455631575e+28, 1.4490456652489208e+28, 1.437621957532008e+28, 1.4264092364541904e+28, 1.4154062296679138e+28, 1.404611581587618e+28, 1.3940238565367406e+28, 1.3836415418870307e+28, 1.373463051186888e+28, 1.3634867272755814e+28, 1.35371084538026e+28, 1.3441336161927877e+28, 1.3347531889235528e+28, 1.3255676543294923e+28, 1.3165750477136715e+28, 1.3077733518938986e+28, 1.2991605001379373e+28, 1.2907343790629856e+28, 1.282492831497232e+28, 1.2744336593013793e+28, 1.2665546261481404e+28, 1.258853460257845e+28, 1.2513278570883774e+28, 1.2439754819777843e+28, 1.2367939727380134e+28, 1.2297809421983352e+28, 1.2229339806971045e+28, 1.216250658520647e+28, 1.2097285282881386e+28, 1.2033651272814471e+28, 1.1971579797190313e+28, 1.191104598973065e+28, 1.1852024897290637e+28, 1.1794491500873908e+28, 1.1738420736061106e+28, 1.1683787512847353e+28, 1.163056673488531e+28, 1.1578733318131088e+28, 1.1528262208891177e+28, 1.147912840126954e+28, 1.1431306954014651e+28, 1.1384773006767033e+28, 1.1339501795708763e+28, 1.129546866861701e+28, 1.1252649099324291e+28, 1.1211018701589063e+28, 1.1170553242380655e+28, 1.1131228654583245e+28, 1.1093021049124288e+28, 1.1055906726533185e+28, 1.1019862187936617e+28, 1.0984864145497501e+28, 1.095088953230494e+28, 1.0917915511722975e+28, 1.0885919486206512e+28, 1.085487910559302e+28, 1.0824772274879072e+28, 1.0795577161491138e+28, 1.0767272202060358e+28, 1.0739836108711256e+28, 1.0713247874874732e+28, 1.0687486780635868e+28, 1.0662532397627211e+28, 1.0638364593478604e+28, 1.0614963535834605e+28, 1.0592309695950781e+28, 1.0570383851880313e+28, 1.0549167091262407e+28, 1.0528640813724086e+28, 1.0508786732907098e+28, 1.048958687813159e+28, 1.0471023595708327e+28, 1.0453079549911207e+28, 1.043573772362184e+28, 1.0418981418657854e+28, 1.0402794255796702e+28, 1.0387160174506532e+28, 1.0372063432395707e+28, 1.0357488604392424e+28, 1.0343420581665867e+28, 1.0329844570300072e+28, 1.0316746089731713e+28, 1.0304110970962791e+28, 1.029192535455908e+28, 1.028017568844504e+28, 1.0268848725505734e+28, 1.0257931521006134e+28, 1.024741142983794e+28, 1.0237276103603993e+28, 1.0227513487549982e+28, 1.0218111817353118e+28, 1.020905961577711e+28, 1.0200345689202623e+28, 1.0191959124042176e+28, 1.018388928304821e+28, 1.0176125801522805e+28, 1.0168658583437348e+28, 1.0161477797470185e+28, 1.0154573872970025e+28, 1.0147937495852744e+28, 1.0141559604438843e+28, 1.0135431385238717e+28, 1.0129544268692564e+28, 1.0123889924871562e+28, 1.0118460259146722e+28, 1.0113247407831531e+28, 1.0108243733804341e+28, 1.0103441822116152e+28, 1.0098834475589269e+28, 1.009441471041205e+28, 1.0090175751734732e+28, 1.0086111029271116e+28, 1.0082214172910679e+28, 1.007847900834542e+28, 1.0074899552715601e+28, 1.0071470010278237e+28, 1.0068184768102126e+28, 1.006503839179283e+28, 1.0062025621250988e+28, 1.0059141366467013e+28, 1.0056380703355138e+28, 1.0053738869629534e+28, 1.0051211260725029e+28, 1.0048793425764843e+28, 1.0046481063577545e+28, 1.0044270018765229e+28, 1.0042156277824858e+28, 1.004013596532443e+28, 1.0038205340135603e+28, 1.0036360791724141e+28, 1.003459883649951e+28, 1.0032916114224749e+28, 1.0031309384487644e+28, 1.0029775523234057e+28, 1.002831151936423e+28, 1.0026914471392658e+28, 1.0025581584172085e+28, 1.002431016568207e+28, 1.0023097623882416e+28, 1.0021941463631681e+28, 1.002083928367095e+28, 1.0019788773672842e+28, 1.001878771135577e+28, 1.0017833959663276e+28, 1.0016925464008276e+28, 1.0016060249581925e+28, 1.0015236418726762e+28, 1.001445214837372e+28, 1.001370568754254e+28, 1.0012995354905094e+28, 1.0012319536410952e+28, 1.0011676682974684e+28, 1.0011065308224124e+28, 1.0010483986308936e+28, 1.0009931349768703e+28, 1.0009406087459768e+28, 1.0008906942539962e+28, 1.000843271051043e+28, 1.0007982237313591e+28, 1.0007554417486398e+28, 1.0007148192367923e+28, 1.0006762548360335e+28, 1.0006396515242307e+28, 1.0006049164533866e+28, 1.0005719607911682e+28, 1.0005406995673826e+28, 1.000511051525294e+28, 1.0004829389776844e+28, 1.000456287667551e+28, 1.000431026633343e+28, 1.0004070880786305e+28, 1.0003844072461038e+28, 1.0003629222958046e+28, 1.000342574187479e+28, 1.0003233065669587e+28, 1.0003050656564634e+28, 1.0002878001487267e+28, 1.0002714611048452e+28, 1.0002560018557503e+28, 1.0002413779072085e+28, 1.000227546848249e+28, 1.0002144682629268e+28, 1.0002021036453233e+28, 1.0001904163176948e+28, 1.0001793713516751e+28, 1.0001689354924404e+28, 1.0001590770857518e+28, 1.0001497660077844e+28, 1.0001409735976568e+28, 1.0001326725925817e+28, 1.0001248370655469e+28, 1.0001174423654532e+28, 1.0001104650596257e+28, 1.0001038828786215e+28, 1.0000976746632582e+28, 1.000091820313788e+28, 1.0000863007411467e+28, 1.0000810978202034e+28, 1.0000761943449455e+28, 1.0000715739855283e+28, 1.0000672212471243e+28, 1.0000631214305104e+28, 1.000059260594325e+28, 1.0000556255189425e+28, 1.0000522036718956e+28, 1.000048983174799e+28, 1.0000459527717097e+28, 1.0000431017988749e+28, 1.0000404201558123e+28, 1.0000378982776735e+28, 1.0000355271088388e+28, 1.0000332980776976e+28, 1.0000312030725668e+28, 1.0000292344187017e+28, 1.0000273848563557e+28, 1.0000256475198486e+28, 1.0000240159176006e+28, 1.000022483913092e+28, 1.0000210457067132e+28, 1.0000196958184652e+28, 1.0000184290714775e+28, 1.0000172405763068e+28, 1.0000161257159851e+28, 1.0000150801317844e+28, 1.0000140997096673e+28, 1.0000131805673944e+28, 1.0000123190422589e+28, 1.0000115116794217e+28, 1.00001075522082e+28, 1.0000100465946249e+28, 1.0000093829052215e+28, 1.0000087614236908e+28, 1.000008179578767e+28, 1.0000076349482532e+28, 1.0000071252508694e+28, 1.0000066483385176e+28, 1.00000620218894e+28, 1.0000057848987579e+28, 1.0000053946768655e+28, 1.0000050298381704e+28, 1.000004688797658e+28, 1.000004370064766e+28, 1.000004072238057e+28, 1.00000379400017e+28, 1.0000035341130411e+28, 1.0000032914133786e+28, 1.0000030648083796e+28, 1.0000028532716783e+28]
x_cell_centers_cm_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = [0.13020833333333334, 0.390625, 0.6510416666666666, 0.9114583333333333, 1.1718749999999998, 1.4322916666666663, 1.692708333333333, 1.9531249999999998, 2.2135416666666665, 2.473958333333333, 2.7343749999999996, 2.9947916666666665, 3.255208333333333, 3.5156249999999996, 3.7760416666666665, 4.036458333333333, 4.296874999999999, 4.557291666666666, 4.817708333333332, 5.078124999999999, 5.338541666666665, 5.598958333333332, 5.859374999999999, 6.119791666666665, 6.380208333333332, 6.640624999999999, 6.901041666666665, 7.161458333333332, 7.421874999999999, 7.682291666666665, 7.942708333333332, 8.203125, 8.463541666666666, 8.723958333333332, 8.984375, 9.244791666666666, 9.505208333333332, 9.765625, 10.026041666666666, 10.286458333333332, 10.546874999999998, 10.807291666666666, 11.067708333333332, 11.328124999999998, 11.588541666666666, 11.848958333333332, 12.109374999999998, 12.369791666666666, 12.630208333333332, 12.890624999999998, 13.151041666666666, 13.411458333333332, 13.671874999999998, 13.932291666666666, 14.192708333333332, 14.453124999999998, 14.713541666666666, 14.973958333333332, 15.234374999999998, 15.494791666666664, 15.755208333333332, 16.015624999999996, 16.276041666666664, 16.53645833333333, 16.796874999999996, 17.057291666666664, 17.31770833333333, 17.578124999999996, 17.838541666666664, 18.09895833333333, 18.359374999999996, 18.619791666666664, 18.88020833333333, 19.140624999999996, 19.401041666666664, 19.66145833333333, 19.921874999999996, 20.182291666666664, 20.44270833333333, 20.703124999999996, 20.96354166666666, 21.22395833333333, 21.484374999999996, 21.74479166666666, 22.00520833333333, 22.265624999999996, 22.52604166666666, 22.78645833333333, 23.046874999999996, 23.30729166666666, 23.56770833333333, 23.828124999999996, 24.08854166666666, 24.34895833333333, 24.609374999999996, 24.86979166666666, 25.13020833333333, 25.390624999999996, 25.65104166666666, 25.91145833333333, 26.171874999999996, 26.43229166666666, 26.69270833333333, 26.953124999999996, 27.21354166666666, 27.47395833333333, 27.734374999999996, 27.99479166666666, 28.25520833333333, 28.515624999999996, 28.77604166666666, 29.03645833333333, 29.296874999999996, 29.55729166666666, 29.81770833333333, 30.078124999999993, 30.33854166666666, 30.59895833333333, 30.859374999999993, 31.11979166666666, 31.38020833333333, 31.640624999999993, 31.90104166666666, 32.16145833333333, 32.421875, 32.682291666666664, 32.94270833333333, 33.203125, 33.463541666666664, 33.72395833333333, 33.984375, 34.244791666666664, 34.50520833333333, 34.765625, 35.026041666666664, 35.28645833333333, 35.546875, 35.807291666666664, 36.06770833333333, 36.328125, 36.588541666666664, 36.84895833333333, 37.109375, 37.369791666666664, 37.63020833333333, 37.890625, 38.151041666666664, 38.41145833333333, 38.671875, 38.932291666666664, 39.19270833333333, 39.453125, 39.713541666666664, 39.97395833333333, 40.234375, 40.494791666666664, 40.75520833333333, 41.015625, 41.276041666666664, 41.53645833333333, 41.79687499999999, 42.057291666666664, 42.31770833333333, 42.57812499999999, 42.838541666666664, 43.09895833333333, 43.35937499999999, 43.619791666666664, 43.88020833333333, 44.14062499999999, 44.401041666666664, 44.66145833333333, 44.92187499999999, 45.182291666666664, 45.44270833333333, 45.70312499999999, 45.963541666666664, 46.22395833333333, 46.48437499999999, 46.744791666666664, 47.00520833333333, 47.26562499999999, 47.526041666666664, 47.78645833333333, 48.04687499999999, 48.307291666666664, 48.56770833333333, 48.82812499999999, 49.088541666666664, 49.34895833333333, 49.60937499999999, 49.869791666666664, 50.13020833333333, 50.39062499999999, 50.651041666666664, 50.91145833333333, 51.17187499999999, 51.432291666666664, 51.69270833333333, 51.95312499999999, 52.213541666666664, 52.47395833333333, 52.73437499999999, 52.994791666666664, 53.25520833333333, 53.51562499999999, 53.776041666666664, 54.03645833333333, 54.29687499999999, 54.557291666666664, 54.81770833333333, 55.07812499999999, 55.338541666666664, 55.59895833333333, 55.85937499999999, 56.119791666666664, 56.38020833333333, 56.64062499999999, 56.901041666666664, 57.16145833333333, 57.42187499999999, 57.682291666666664, 57.94270833333333, 58.20312499999999, 58.463541666666664, 58.72395833333333, 58.98437499999999, 59.24479166666666, 59.50520833333333, 59.76562499999999, 60.02604166666666, 60.28645833333333, 60.54687499999999, 60.80729166666666, 61.06770833333333, 61.32812499999999, 61.58854166666666, 61.84895833333333, 62.10937499999999, 62.36979166666666, 62.63020833333333, 62.89062499999999, 63.15104166666666, 63.41145833333333, 63.67187499999999, 63.93229166666666, 64.19270833333331, 64.45312499999999, 64.71354166666666, 64.97395833333331, 65.23437499999999, 65.49479166666666, 65.75520833333331, 66.01562499999999, 66.27604166666666, 66.53645833333331, 66.79687499999999, 67.05729166666666, 67.31770833333331, 67.57812499999999, 67.83854166666666, 68.09895833333331, 68.35937499999999, 68.61979166666666, 68.88020833333331, 69.14062499999999, 69.40104166666666, 69.66145833333331, 69.92187499999999, 70.18229166666666, 70.44270833333331, 70.70312499999999, 70.96354166666666, 71.22395833333331, 71.48437499999999, 71.74479166666666, 72.00520833333331, 72.26562499999999, 72.52604166666666, 72.78645833333331, 73.04687499999999, 73.30729166666666, 73.56770833333331, 73.82812499999999, 74.08854166666666, 74.34895833333331, 74.60937499999999, 74.86979166666666, 75.13020833333331, 75.39062499999999, 75.65104166666666, 75.91145833333331, 76.17187499999999, 76.43229166666666, 76.69270833333331, 76.95312499999999, 77.21354166666666, 77.47395833333331, 77.73437499999999, 77.99479166666666, 78.25520833333331, 78.51562499999999, 78.77604166666666, 79.03645833333331, 79.29687499999999, 79.55729166666666, 79.81770833333331, 80.07812499999999, 80.33854166666666, 80.59895833333331, 80.85937499999999, 81.11979166666666, 81.38020833333331, 81.64062499999999, 81.90104166666666, 82.16145833333331, 82.42187499999999, 82.68229166666666, 82.94270833333331, 83.20312499999999, 83.46354166666664, 83.72395833333331, 83.98437499999999, 84.24479166666664, 84.50520833333331, 84.76562499999999, 85.02604166666664, 85.28645833333331, 85.54687499999999, 85.80729166666664, 86.06770833333331, 86.32812499999999, 86.58854166666664, 86.84895833333331, 87.10937499999999, 87.36979166666664, 87.63020833333331, 87.89062499999999, 88.15104166666664, 88.41145833333331, 88.67187499999999, 88.93229166666664, 89.19270833333331, 89.45312499999999, 89.71354166666664, 89.97395833333331, 90.23437499999999, 90.49479166666664, 90.75520833333331, 91.01562499999999, 91.27604166666664, 91.53645833333331, 91.79687499999999, 92.05729166666664, 92.31770833333331, 92.57812499999999, 92.83854166666664, 93.09895833333331, 93.35937499999999, 93.61979166666664, 93.88020833333331, 94.14062499999999, 94.40104166666664, 94.66145833333331, 94.92187499999999, 95.18229166666664, 95.44270833333331, 95.70312499999999, 95.96354166666664, 96.22395833333331, 96.48437499999999, 96.74479166666664, 97.00520833333331, 97.26562499999999, 97.52604166666664, 97.78645833333331, 98.04687499999999, 98.30729166666664, 98.56770833333331, 98.82812499999999, 99.08854166666664, 99.34895833333331, 99.60937499999999, 99.86979166666664, 100.13020833333331, 100.39062499999999, 100.65104166666664, 100.91145833333331, 101.17187499999999, 101.43229166666664, 101.69270833333331, 101.95312499999999, 102.21354166666664, 102.47395833333331, 102.73437499999999, 102.99479166666664, 103.25520833333331, 103.51562499999999, 103.77604166666664, 104.03645833333331, 104.29687499999999, 104.55729166666664, 104.81770833333331, 105.07812499999999, 105.33854166666664, 105.59895833333331, 105.85937499999999, 106.11979166666664, 106.38020833333331, 106.64062499999999, 106.90104166666664, 107.16145833333331, 107.42187499999999, 107.68229166666664, 107.94270833333331, 108.20312499999999, 108.46354166666664, 108.72395833333331, 108.98437499999999, 109.24479166666664, 109.50520833333331, 109.76562499999999, 110.02604166666664, 110.28645833333331, 110.54687499999999, 110.80729166666664, 111.06770833333331, 111.32812499999999, 111.58854166666664, 111.84895833333331, 112.10937499999999, 112.36979166666664, 112.63020833333331, 112.89062499999999, 113.15104166666664, 113.41145833333331, 113.67187499999999, 113.93229166666664, 114.19270833333331, 114.45312499999999, 114.71354166666664, 114.97395833333331, 115.23437499999999, 115.49479166666664, 115.75520833333331, 116.01562499999999, 116.27604166666664, 116.53645833333331, 116.79687499999999, 117.05729166666664, 117.31770833333331, 117.57812499999997, 117.83854166666664, 118.09895833333331, 118.35937499999997, 118.61979166666664, 118.88020833333331, 119.14062499999997, 119.40104166666664, 119.66145833333331, 119.92187499999997, 120.18229166666664, 120.44270833333331, 120.70312499999997, 120.96354166666664, 121.22395833333331, 121.48437499999997, 121.74479166666664, 122.00520833333331, 122.26562499999997, 122.52604166666664, 122.78645833333331, 123.04687499999997, 123.30729166666664, 123.56770833333331, 123.82812499999997, 124.08854166666664, 124.34895833333331, 124.60937499999997, 124.86979166666664, 125.13020833333331, 125.39062499999997, 125.65104166666664, 125.91145833333331, 126.17187499999997, 126.43229166666664, 126.69270833333331, 126.95312499999997, 127.21354166666664, 127.47395833333331, 127.73437499999997, 127.99479166666664, 128.25520833333331, 128.515625, 128.77604166666666, 129.03645833333331, 129.296875, 129.55729166666666, 129.81770833333331, 130.078125, 130.33854166666666, 130.59895833333331, 130.859375, 131.11979166666666, 131.38020833333331, 131.640625, 131.90104166666666, 132.16145833333331, 132.421875, 132.68229166666666, 132.94270833333331, 133.203125, 133.46354166666666, 133.72395833333331, 133.984375, 134.24479166666666, 134.50520833333331, 134.765625, 135.02604166666666, 135.28645833333331, 135.546875, 135.80729166666666, 136.06770833333331, 136.328125, 136.58854166666666, 136.84895833333331, 137.109375, 137.36979166666666, 137.63020833333331, 137.890625, 138.15104166666666, 138.41145833333331, 138.671875, 138.93229166666666, 139.19270833333331, 139.453125, 139.71354166666666, 139.97395833333331, 140.234375, 140.49479166666666, 140.75520833333331, 141.015625, 141.27604166666666, 141.53645833333331, 141.796875, 142.05729166666666, 142.31770833333331, 142.578125, 142.83854166666666, 143.09895833333331, 143.359375, 143.61979166666666, 143.88020833333331, 144.140625, 144.40104166666666, 144.66145833333331, 144.921875, 145.18229166666666, 145.44270833333331, 145.703125, 145.96354166666666, 146.22395833333331, 146.484375, 146.74479166666666, 147.00520833333331, 147.265625, 147.52604166666666, 147.78645833333331, 148.046875, 148.30729166666666, 148.56770833333331, 148.828125, 149.08854166666666, 149.34895833333331, 149.609375, 149.86979166666666, 150.13020833333331, 150.390625, 150.65104166666666, 150.91145833333331, 151.171875, 151.43229166666666, 151.69270833333331, 151.953125, 152.21354166666666, 152.47395833333331, 152.734375, 152.99479166666666, 153.25520833333331, 153.515625, 153.77604166666666, 154.03645833333331, 154.296875, 154.55729166666666, 154.81770833333331, 155.078125, 155.33854166666666, 155.59895833333331, 155.859375, 156.11979166666666, 156.38020833333331, 156.640625, 156.90104166666666, 157.16145833333331, 157.421875, 157.68229166666666, 157.94270833333331, 158.203125, 158.46354166666666, 158.72395833333331, 158.984375, 159.24479166666666, 159.50520833333331, 159.765625, 160.02604166666666, 160.28645833333331, 160.546875, 160.80729166666666, 161.06770833333331, 161.328125, 161.58854166666666, 161.84895833333331, 162.109375, 162.36979166666666, 162.63020833333331, 162.890625, 163.15104166666666, 163.41145833333331, 163.671875, 163.93229166666666, 164.19270833333331, 164.453125, 164.71354166666666, 164.97395833333331, 165.234375, 165.49479166666666, 165.75520833333331, 166.015625, 166.27604166666666, 166.53645833333331, 166.79687499999997, 167.05729166666666, 167.31770833333331, 167.57812499999997, 167.83854166666666, 168.09895833333331, 168.35937499999997, 168.61979166666666, 168.88020833333331, 169.14062499999997, 169.40104166666666, 169.66145833333331, 169.92187499999997, 170.18229166666666, 170.44270833333331, 170.70312499999997, 170.96354166666666, 171.22395833333331, 171.48437499999997, 171.74479166666666, 172.00520833333331, 172.26562499999997, 172.52604166666666, 172.78645833333331, 173.04687499999997, 173.30729166666666, 173.56770833333331, 173.82812499999997, 174.08854166666666, 174.34895833333331, 174.60937499999997, 174.86979166666666, 175.13020833333331, 175.39062499999997, 175.65104166666666, 175.91145833333331, 176.17187499999997, 176.43229166666666, 176.69270833333331, 176.95312499999997, 177.21354166666666, 177.47395833333331, 177.73437499999997, 177.99479166666666, 178.25520833333331, 178.51562499999997, 178.77604166666666, 179.03645833333331, 179.29687499999997, 179.55729166666666, 179.81770833333331, 180.07812499999997, 180.33854166666666, 180.59895833333331, 180.85937499999997, 181.11979166666666, 181.38020833333331, 181.64062499999997, 181.90104166666666, 182.16145833333331, 182.42187499999997, 182.68229166666666, 182.94270833333331, 183.20312499999997, 183.46354166666666, 183.72395833333331, 183.98437499999997, 184.24479166666666, 184.50520833333331, 184.76562499999997, 185.02604166666666, 185.28645833333331, 185.54687499999997, 185.80729166666666, 186.06770833333331, 186.32812499999997, 186.58854166666666, 186.84895833333331, 187.10937499999997, 187.36979166666666, 187.63020833333331, 187.89062499999997, 188.15104166666666, 188.41145833333331, 188.67187499999997, 188.93229166666666, 189.19270833333331, 189.45312499999997, 189.71354166666666, 189.97395833333331, 190.23437499999997, 190.49479166666666, 190.75520833333331, 191.01562499999997, 191.27604166666666, 191.53645833333331, 191.79687499999997, 192.05729166666666, 192.31770833333331, 192.57812499999997, 192.83854166666666, 193.09895833333331, 193.35937499999997, 193.61979166666666, 193.88020833333331, 194.14062499999997, 194.40104166666666, 194.66145833333331, 194.92187499999997, 195.18229166666666, 195.44270833333331, 195.70312499999997, 195.96354166666666, 196.22395833333331, 196.48437499999997, 196.74479166666666, 197.00520833333331, 197.26562499999997, 197.52604166666666, 197.78645833333331, 198.04687499999997, 198.30729166666666, 198.56770833333331, 198.82812499999997, 199.08854166666666, 199.34895833333331, 199.60937499999997, 199.86979166666666]

last_time_s_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 9.999999999999707e-09
last_nee_invccm_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = [1.0000077647962259e+28, 1.000007849254834e+28, 1.0000080189367797e+28, 1.0000082753817075e+28, 1.0000086209125978e+28, 1.0000090586523265e+28, 1.000009592545858e+28, 1.0000102273882467e+28, 1.000010968858256e+28, 1.0000118235582866e+28, 1.0000127990604245e+28, 1.0000139039587522e+28, 1.0000151479287172e+28, 1.0000165417931975e+28, 1.000018097595886e+28, 1.000019828682485e+28, 1.0000217497895682e+28, 1.0000238771420254e+28, 1.0000262285593394e+28, 1.000028823570987e+28, 1.0000316835416947e+28, 1.0000348318069627e+28, 1.0000382938194425e+28, 1.0000420973068827e+28, 1.0000462724419563e+28, 1.0000508520249468e+28, 1.0000558716799565e+28, 1.0000613700652377e+28, 1.0000673890984463e+28, 1.000073974197957e+28, 1.0000811745403973e+28, 1.0000890433362643e+28, 1.0000976381237798e+28, 1.0001070210820857e+28, 1.0001172593651372e+28, 1.0001284254569464e+28, 1.0001405975497686e+28, 1.000153859945724e+28, 1.0001683034835044e+28, 1.0001840259911416e+28, 1.0002011327664247e+28, 1.0002197370854922e+28, 1.0002399607417342e+28, 1.0002619346160667e+28, 1.0002857992797966e+28, 1.0003117056314654e+28, 1.0003398155694311e+28, 1.0003703027014133e+28, 1.0004033530922956e+28, 1.000439166052223e+28, 1.0004779549659504e+28, 1.0005199481651846e+28, 1.0005653898455685e+28, 1.0006145410296045e+28, 1.0006676805771475e+28, 1.0007251062450407e+28, 1.0007871357973075e+28, 1.0008541081672312e+28, 1.0009263846731975e+28, 1.0010043502892227e+28, 1.0010884149717476e+28, 1.0011790150441757e+28, 1.0012766146399972e+28, 1.0013817072060491e+28, 1.0014948170668754e+28, 1.0016165010510164e+28, 1.0017473501805273e+28, 1.0018879914241183e+28, 1.0020390895147022e+28, 1.002201348832143e+28, 1.0023755153510862e+28, 1.0025623786545e+28, 1.002762774012772e+28, 1.0029775845282946e+28, 1.003207743344988e+28, 1.0034542359225093e+28, 1.0037181023739398e+28, 1.004000439866232e+28, 1.0043024050819776e+28, 1.0046252167406007e+28, 1.0049701581776336e+28, 1.005338579979492e+28, 1.0057319026707805e+28, 1.0061516194524034e+28, 1.0065992989856854e+28, 1.0070765882205553e+28, 1.0075852152626133e+28, 1.008126992274904e+28, 1.008703818409609e+28, 1.0093176827647934e+28, 1.0099706673596287e+28, 1.0106649501227035e+28, 1.0114028078866773e+28, 1.0121866193817173e+28, 1.013018868220681e+28, 1.0139021458675454e+28, 1.0148391545806829e+28, 1.015832710321999e+28, 1.0168857456220417e+28, 1.0180013123911798e+28, 1.0191825846662627e+28, 1.0204328612815164e+28, 1.021755568452091e+28, 1.0231542622580696e+28, 1.0246326310165116e+28, 1.0261944975283264e+28, 1.027843821186146e+28, 1.029584699929654e+28, 1.0314213720336995e+28, 1.0333582177143397e+28, 1.0353997605374194e+28, 1.037550668614552e+28, 1.0398157555701104e+28, 1.0421999812631471e+28, 1.0447084522478496e+28, 1.047346421955601e+28, 1.0501192905823922e+28, 1.0530326046636419e+28, 1.056092056320039e+28, 1.0593034821568314e+28, 1.0626728617999681e+28, 1.0662063160515696e+28, 1.0699101046480574e+28, 1.0737906236043034e+28, 1.0778544021273425e+28, 1.082108099083563e+28, 1.0865584990037894e+28, 1.0912125076110622e+28, 1.0960771468564961e+28, 1.1011595494493817e+28, 1.1064669528681186e+28, 1.1120066928397025e+28, 1.117786196276314e+28, 1.1238129736581628e+28, 1.1300946108533107e+28, 1.1366387603662071e+28, 1.1434531320075115e+28, 1.150545482979609e+28, 1.1579236073733957e+28, 1.1655953250732535e+28, 1.1735684700690734e+28, 1.1818508781755355e+28, 1.1904503741602509e+28, 1.1993747582854618e+28, 1.2086317922679846e+28, 1.218229184665609e+28, 1.2281745756995319e+28, 1.2384755215241863e+28, 1.2491394779585921e+28, 1.2601737836952316e+28, 1.2715856430042034e+28, 1.2833821079537088e+28, 1.2955700601692083e+28, 1.3081561921562667e+28, 1.3211469882149253e+28, 1.334548704974729e+28, 1.3483673515828172e+28, 1.3626086695795184e+28, 1.377278112497787e+28, 1.3923808252258854e+28, 1.4079216231745581e+28, 1.4239049712917202e+28, 1.4403349629707466e+28, 1.4572152988997493e+28, 1.4745492659022279e+28, 1.4923397158192869e+28, 1.5105890444894186e+28, 1.5292991708780695e+28, 1.5484715164160987e+28, 1.568106984605031e+28, 1.5882059409478875e+28, 1.6087681932673132e+28, 1.6297929724731577e+28, 1.651278913841267e+28, 1.673224038868333e+28, 1.695625737765996e+28, 1.7184807526588845e+28, 1.741785161551311e+28, 1.7655343631269243e+28, 1.789723062445376e+28, 1.814345257600322e+28, 1.8393942274010437e+28, 1.8648625201399975e+28, 1.890741943507625e+28, 1.9170235557129085e+28, 1.9436976578683805e+28, 1.9707537876951025e+28, 1.998180714601402e+28, 2.0259664361867737e+28, 2.0540981762201554e+28, 2.0825623841378954e+28, 2.1113447361050765e+28, 2.1404301376795e+28, 2.1698027281144718e+28, 2.199445886333163e+28, 2.2293422386028336e+28, 2.2594736679334203e+28, 2.2898213252210753e+28, 2.320365642151856e+28, 2.3510863458767632e+28, 2.381962475465045e+28, 2.412972400136495e+28, 2.4440938392697723e+28, 2.4753038841778208e+28, 2.5065790216363008e+28, 2.5378951591464025e+28, 2.5692276519073936e+28, 2.600551331468765e+28, 2.631840536027034e+28, 2.663069142326868e+28, 2.694210599120229e+28, 2.7252379621322977e+28, 2.7561239304783054e+28, 2.786840884468745e+28, 2.8173609247374293e+28, 2.8476559126196597e+28, 2.8776975117046857e+28, 2.907457230481801e+28, 2.9369064659941425e+28, 2.966016548410625e+28, 2.9947587864226793e+28, 3.0231045133675513e+28, 3.0510251339783453e+28, 3.0784921716561813e+28, 3.1054773161561553e+28, 3.1319524715795523e+28, 3.157889804558815e+28, 3.1832617925207424e+28, 3.2080412719131564e+28, 3.232201486277885e+28, 3.2557161340509926e+28, 3.2785594159728713e+28, 3.3007060819891614e+28, 3.322131477522996e+28, 3.3428115890018184e+28, 3.3627230885206853e+28, 3.381843377526523e+28, 3.4001506294097166e+28, 3.4176238308903744e+28, 3.4342428220902195e+28, 3.449988335184064e+28, 3.464842031525498e+28, 3.478786537148596e+28, 3.491805476550236e+28, 3.5038835046591568e+28, 3.5150063369058437e+28, 3.5251607773122334e+28, 3.5343347445214756e+28, 3.5425172956965403e+28, 3.5496986482247167e+28, 3.555870199164216e+28, 3.5610245423802943e+28, 3.565155483322981e+28, 3.5682580514058307e+28, 3.570328509950794e+28, 3.5713643636711863e+28, 3.5713643636716736e+28, 3.5703285099522935e+28, 3.568258051408348e+28, 3.565155483326512e+28, 3.561024542384922e+28, 3.555870199169948e+28, 3.5496986482315495e+28, 3.5425172957045514e+28, 3.534334744530883e+28, 3.525160777322991e+28, 3.515006336917589e+28, 3.5038835046717787e+28, 3.4918054765637397e+28, 3.4787865371627317e+28, 3.4648420315399733e+28, 3.4499883351987703e+28, 3.434242822104943e+28, 3.4176238309048625e+28, 3.400150629424158e+28, 3.3818433775409418e+28, 3.362723088534923e+28, 3.342811589015888e+28, 3.322131477536825e+28, 3.3007060820026107e+28, 3.2785594159860527e+28, 3.25571613406375e+28, 3.232201486290148e+28, 3.2080412719248698e+28, 3.1832617925317547e+28, 3.157889804569315e+28, 3.131952471589646e+28, 3.1054773161658675e+28, 3.0784921716658746e+28, 3.0510251339880307e+28, 3.023104513376786e+28, 2.994758786431558e+28, 2.966016548419046e+28, 2.9369064660019424e+28, 2.907457230488928e+28, 2.8776975117111777e+28, 2.847655912625475e+28, 2.817360924742598e+28, 2.78684088447305e+28, 2.7561239304817157e+28, 2.7252379621350685e+28, 2.6942105991223042e+28, 2.6630691423282818e+28, 2.631840536027762e+28, 2.600551331468754e+28, 2.5692276519066697e+28, 2.5378951591448557e+28, 2.5065790216338427e+28, 2.475303884174419e+28, 2.4440938392653967e+28, 2.4129724001309803e+28, 2.3819624754583936e+28, 2.3510863458689765e+28, 2.3203656421427545e+28, 2.2898213252106167e+28, 2.2594736679214647e+28, 2.229342238589351e+28, 2.1994458863181873e+28, 2.169802728097858e+28, 2.1404301376611824e+28, 2.111344736085044e+28, 2.0825623841160547e+28, 2.0540981761963786e+28, 2.025966436160943e+28, 1.9981807145734525e+28, 1.9707537876649722e+28, 1.943697657835977e+28, 1.9170235556776665e+28, 1.8907419434693038e+28, 1.864862520099393e+28, 1.8393942273583044e+28, 1.8143452575556687e+28, 1.7897230623988499e+28, 1.7655343630786258e+28, 1.741785161501463e+28, 1.718480752607522e+28, 1.695625737713234e+28, 1.6732240388142234e+28, 1.651278913785894e+28, 1.6297929724165865e+28, 1.6087681932096717e+28, 1.5882059408891586e+28, 1.5681069845453587e+28, 1.5484715163555406e+28, 1.529299170816643e+28, 1.5105890444272683e+28, 1.492339715756387e+28, 1.4745492658385332e+28, 1.457215298835431e+28, 1.4403349629057386e+28, 1.4239049712261049e+28, 1.4079216231083035e+28, 1.3923808251590277e+28, 1.3772781124304368e+28, 1.3626086695117894e+28, 1.348367351514731e+28, 1.3345487049062996e+28, 1.3211469881462113e+28, 1.3081561920872968e+28, 1.2955700600999857e+28, 1.2833821078842942e+28, 1.2715856429345663e+28, 1.2601737836253714e+28, 1.2491394778885741e+28, 1.2384755214540056e+28, 1.2281745756292616e+28, 1.218229184595281e+28, 1.208631792197585e+28, 1.1993747582150427e+28, 1.1904503740898232e+28, 1.1818508781051327e+28, 1.17356846999875e+28, 1.1655953250029344e+28, 1.1579236073031383e+28, 1.1505454829094323e+28, 1.1434531319374005e+28, 1.1366387602961788e+28, 1.130094610783375e+28, 1.1238129735883315e+28, 1.117786196206606e+28, 1.1120066927700966e+28, 1.1064669527985983e+28, 1.101159549379981e+28, 1.096077146787231e+28, 1.0912125075419214e+28, 1.0865584989347882e+28, 1.0821080990147045e+28, 1.0778544020586023e+28, 1.07379062353572e+28, 1.0699101045796418e+28, 1.066206315983337e+28, 1.062672861731952e+28, 1.0593034820890388e+28, 1.056092056252471e+28, 1.053032604596331e+28, 1.0501192905152734e+28, 1.047346421888711e+28, 1.0447084521812396e+28, 1.042199981196854e+28, 1.0398157555040992e+28, 1.0375506685487802e+28, 1.035399760471857e+28, 1.0333582176490383e+28, 1.0314213719686924e+28, 1.029584699864901e+28, 1.0278438211216367e+28, 1.0261944974640613e+28, 1.0246326309525024e+28, 1.0231542621943454e+28, 1.0217555683885475e+28, 1.0204328612180979e+28, 1.0191825846029346e+28, 1.0180013123278767e+28, 1.0168857455587461e+28, 1.0158327102586908e+28, 1.0148391545173029e+28, 1.013902145804031e+28, 1.0130188681570484e+28, 1.0121866193178777e+28, 1.0114028078226143e+28, 1.0106649500583796e+28, 1.0099706672949849e+28, 1.0093176826997868e+28, 1.0087038183441624e+28, 1.0081269922089466e+28, 1.0075852151961947e+28, 1.0070765881536217e+28, 1.0065992989182557e+28, 1.006151619384448e+28, 1.0057319026022037e+28, 1.0053385799100437e+28, 1.0049701581071545e+28, 1.0046252166689765e+28, 1.0043024050094338e+28, 1.0040004397929447e+28, 1.0037181022999494e+28, 1.00345423584793e+28, 1.0032077432698703e+28, 1.002977584452612e+28, 1.0027627739365361e+28, 1.0025623785777046e+28, 1.0023755152737776e+28, 1.002201348754335e+28, 1.0020390894364183e+28, 1.001887991345382e+28, 1.0017473501013864e+28, 1.0016165009714718e+28, 1.0014948169869651e+28, 1.001381707125842e+28, 1.0012766145594507e+28, 1.0011790149633147e+28, 1.0010884148906058e+28, 1.0010043502077773e+28, 1.0009263845915064e+28, 1.0008541080853339e+28, 1.0007871357152443e+28, 1.0007251061628676e+28, 1.0006676804948416e+28, 1.0006145409472566e+28, 1.0005653897632307e+28, 1.000519948082897e+28, 1.0004779548837626e+28, 1.0004391659701838e+28, 1.0004033530104347e+28, 1.0003703026197996e+28, 1.0003398154881379e+28, 1.0003117055504813e+28, 1.0002857991992024e+28, 1.0002619345359202e+28, 1.0002399606620634e+28, 1.0002197370063561e+28, 1.000201132687916e+28, 1.0001840259132359e+28, 1.0001683034062272e+28, 1.0001538598691563e+28, 1.0001405974739715e+28, 1.0001284253820097e+28, 1.000117259291099e+28, 1.0001070210090614e+28, 1.0000976380517673e+28, 1.0000890432652985e+28, 1.0000811744704984e+28, 1.0000739741292408e+28, 1.000067389031013e+28, 1.0000613699990909e+28, 1.0000558716151622e+28, 1.0000508519615222e+28, 1.0000462723799786e+28, 1.0000420972464773e+28, 1.0000382937606093e+28, 1.000034831749788e+28, 1.0000316834862758e+28, 1.0000288235174015e+28, 1.000026228507626e+28, 1.000023877092252e+28, 1.0000217497417986e+28, 1.0000198286368264e+28, 1.0000180975523922e+28, 1.00001654175195e+28, 1.000015147889839e+28, 1.0000139039223264e+28, 1.0000127990265248e+28, 1.000011823527044e+28, 1.0000109688296964e+28, 1.0000102273625078e+28, 1.0000095925230952e+28, 1.0000090586326098e+28, 1.0000086208961032e+28, 1.0000082753685044e+28, 1.000008018926953e+28, 1.0000078492485439e+28, 1.0000077647939804e+28]
last_analytic_solution_invccm_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = [1.0000029047969417e+28, 1.0000032332944516e+28, 1.000003597429294e+28, 1.0000040008918146e+28, 1.0000044477347653e+28, 1.00000494240676e+28, 1.0000054897885947e+28, 1.0000060952326607e+28, 1.0000067646056849e+28, 1.0000075043350547e+28, 1.0000083214589994e+28, 1.0000092236809176e+28, 1.0000102194281597e+28, 1.0000113179155963e+28, 1.0000125292143188e+28, 1.0000138643258482e+28, 1.000015335262243e+28, 1.000016955132529e+28, 1.0000187382358918e+28, 1.0000207001621048e+28, 1.0000228578996896e+28, 1.000025229952333e+28, 1.0000278364641228e+28, 1.000030699354181e+28, 1.000033842461322e+28, 1.0000372916993784e+28, 1.000041075223891e+28, 1.000045223610874e+28, 1.000049770048423e+28, 1.0000547505419536e+28, 1.0000602041339084e+28, 1.0000661731388058e+28, 1.0000727033945397e+28, 1.0000798445308913e+28, 1.000087650256243e+28, 1.0000961786635357e+28, 1.0001054925565523e+28, 1.0001156597976482e+28, 1.0001267536781028e+28, 1.0001388533122977e+28, 1.0001520440569833e+28, 1.0001664179569303e+28, 1.0001820742183073e+28, 1.0001991197111734e+28, 1.000217669502508e+28, 1.0002378474212477e+28, 1.0002597866568361e+28, 1.0002836303928288e+28, 1.0003095324771361e+28, 1.0003376581305141e+28, 1.000368184694949e+28, 1.0004013024236052e+28, 1.0004372153140365e+28, 1.0004761419863742e+28, 1.0005183166082287e+28, 1.0005639898680488e+28, 1.0006134299986927e+28, 1.000666923852967e+28, 1.0007247780328815e+28, 1.0007873200743612e+28, 1.0008548996891356e+28, 1.0009278900655037e+28, 1.0010066892296329e+28, 1.001091721469016e+28, 1.0011834388196505e+28, 1.0012823226184483e+28, 1.00138888512231e+28, 1.001503671195212e+28, 1.001627260064566e+28, 1.0017602671479952e+28, 1.001903345951557e+28, 1.002057190040307e+28, 1.0022225350819471e+28, 1.0024001609641457e+28, 1.0025908939859305e+28, 1.0027956091233694e+28, 1.0030152323695395e+28, 1.0032507431485576e+28, 1.0035031768032067e+28, 1.003773627155424e+28, 1.0040632491386403e+28, 1.0043732615006641e+28, 1.0047049495754834e+28, 1.005059668122023e+28, 1.0054388442275482e+28, 1.0058439802730178e+28, 1.0062766569573156e+28, 1.006738536376862e+28, 1.007231365156689e+28, 1.007756977628607e+28, 1.0083172990516316e+28, 1.0089143488693516e+28, 1.0095502439984174e+28, 1.0102272021418212e+28, 1.010947545120096e+28, 1.0117137022130306e+28, 1.0125282135039214e+28, 1.0133937332178266e+28, 1.014313033044697e+28, 1.015289005437673e+28, 1.0163246668762348e+28, 1.0174231610832953e+28, 1.018587762184717e+28, 1.0198218777991219e+28, 1.0211290520452626e+28, 1.0225129684536205e+28, 1.0239774527682882e+28, 1.0255264756246203e+28, 1.0271641550875462e+28, 1.0288947590348848e+28, 1.03072270736945e+28, 1.0326525740432153e+28, 1.0346890888763067e+28, 1.036837139153117e+28, 1.0391017709774022e+28, 1.0414881903678062e+28, 1.044001764074902e+28, 1.0466480201005107e+28, 1.0494326478997844e+28, 1.05236149824631e+28, 1.0554405827403276e+28, 1.0586760729400346e+28, 1.0620742990959049e+28, 1.065641748467964e+28, 1.0693850632060517e+28, 1.0733110377732587e+28, 1.0774266158929695e+28, 1.0817388870002596e+28, 1.0862550821787974e+28, 1.0909825695649026e+28, 1.0959288492009834e+28, 1.1011015473212666e+28, 1.1065084100534978e+28, 1.1121572965211647e+28, 1.118056171331774e+28, 1.1242130964377816e+28, 1.1306362223579585e+28, 1.13733377874826e+28, 1.1443140643126533e+28, 1.1515854360458515e+28, 1.1591562978015063e+28, 1.1670350881811121e+28, 1.1752302677406803e+28, 1.1837503055141525e+28, 1.1926036648545304e+28, 1.2017987885957926e+28, 1.211344083540878e+28, 1.22124790428329e+28, 1.2315185363722388e+28, 1.242164178833705e+28, 1.2531929260623035e+28, 1.264612749101429e+28, 1.276431476331807e+28, 1.2886567735912746e+28, 1.301296123751351e+28, 1.3143568057789514e+28, 1.3278458733143922e+28, 1.341770132799663e+28, 1.3561361211937727e+28, 1.3709500833148045e+28, 1.3862179488511182e+28, 1.4019453090869409e+28, 1.4181373933903133e+28, 1.434799045514078e+28, 1.4519346997632157e+28, 1.4695483570844006e+28, 1.4876435611361182e+28, 1.506223374400058e+28, 1.5252903543967437e+28, 1.544846530070501e+28, 1.5648933784108395e+28, 1.5854318013791623e+28, 1.6064621032113852e+28, 1.627983968168527e+28, 1.6499964388086344e+28, 1.6724978948544875e+28, 1.6954860327324163e+28, 1.7189578458582e+28, 1.7429096057464347e+28, 1.76733684401992e+28, 1.7922343353955315e+28, 1.8175960817226741e+28, 1.8434152971498052e+28, 1.8696843944935945e+28, 1.8963949728841072e+28, 1.9235378067579212e+28, 1.9511028362693112e+28, 1.9790791591875773e+28, 2.0074550243462326e+28, 2.03621782670711e+28, 2.0653541040994985e+28, 2.0948495356911803e+28, 2.1246889422447134e+28, 2.1548562882084872e+28, 2.185334685688001e+28, 2.2161064003384517e+28, 2.247152859215113e+28, 2.2784546606131204e+28, 2.309991585923185e+28, 2.3417426135244346e+28, 2.3736859347300574e+28, 2.4057989717957058e+28, 2.438058397994719e+28, 2.4704401597581775e+28, 2.502919500871608e+28, 2.5354709887138675e+28, 2.5680685425173063e+28, 2.60068546362188e+28, 2.63329446768931e+28, 2.665867718836876e+28, 2.698376865643865e+28, 2.7307930789771576e+28, 2.763087091575978e+28, 2.7952292393294113e+28, 2.827189504173992e+28, 2.858937558532488e+28, 2.8904428112089853e+28, 2.9216744546495135e+28, 2.952601513471829e+28, 2.983192894162538e+28, 3.013417435834583e+28, 3.043243961933242e+28, 3.0726413327741835e+28, 3.1015784987928694e+28, 3.130024554380681e+28, 3.1579487921795707e+28, 3.1853207577038895e+28, 3.2121103041552446e+28, 3.238287647293903e+28, 3.2638234202283155e+28, 3.2886887279828596e+28, 3.31285520170287e+28, 3.336295052355456e+28, 3.3589811237845176e+28, 3.3808869449787453e+28, 3.4019867814122782e+28, 3.4222556853190035e+28, 3.441669544763345e+28, 3.4602051313726805e+28, 3.477840146599322e+28, 3.494553266383221e+28, 3.5103241840903323e+28, 3.5251336516056494e+28, 3.538963518464622e+28, 3.551796768911612e+28, 3.5636175567795166e+28, 3.574411238090507e+28, 3.5841644012840045e+28, 3.592864894984568e+28, 3.600501853229212e+28, 3.607065718080859e+28, 3.612548259562043e+28, 3.6169425928506705e+28, 3.6202431926875543e+28, 3.6224459049534984e+28, 3.623547955382007e+28, 3.623547955382007e+28, 3.6224459049534984e+28, 3.6202431926875543e+28, 3.6169425928506705e+28, 3.612548259562043e+28, 3.607065718080859e+28, 3.600501853229212e+28, 3.592864894984568e+28, 3.5841644012840045e+28, 3.574411238090507e+28, 3.5636175567795166e+28, 3.551796768911612e+28, 3.538963518464622e+28, 3.5251336516056494e+28, 3.5103241840903323e+28, 3.494553266383221e+28, 3.477840146599322e+28, 3.4602051313726805e+28, 3.441669544763345e+28, 3.4222556853190035e+28, 3.4019867814122782e+28, 3.3808869449787453e+28, 3.3589811237845176e+28, 3.336295052355456e+28, 3.31285520170287e+28, 3.2886887279828596e+28, 3.2638234202283155e+28, 3.238287647293903e+28, 3.2121103041552446e+28, 3.1853207577038895e+28, 3.1579487921795707e+28, 3.130024554380681e+28, 3.1015784987928694e+28, 3.0726413327741835e+28, 3.043243961933242e+28, 3.013417435834583e+28, 2.983192894162538e+28, 2.952601513471829e+28, 2.9216744546495135e+28, 2.8904428112089853e+28, 2.858937558532488e+28, 2.827189504173992e+28, 2.7952292393294113e+28, 2.763087091575978e+28, 2.7307930789771576e+28, 2.698376865643865e+28, 2.665867718836876e+28, 2.63329446768931e+28, 2.60068546362188e+28, 2.5680685425173063e+28, 2.5354709887138675e+28, 2.502919500871608e+28, 2.4704401597581775e+28, 2.438058397994719e+28, 2.4057989717957058e+28, 2.3736859347300574e+28, 2.3417426135244346e+28, 2.309991585923185e+28, 2.2784546606131204e+28, 2.247152859215113e+28, 2.2161064003384517e+28, 2.185334685688001e+28, 2.1548562882084872e+28, 2.1246889422447134e+28, 2.0948495356911803e+28, 2.0653541040994985e+28, 2.03621782670711e+28, 2.0074550243462326e+28, 1.9790791591875773e+28, 1.9511028362693112e+28, 1.9235378067579212e+28, 1.8963949728841072e+28, 1.8696843944935945e+28, 1.8434152971498052e+28, 1.8175960817226741e+28, 1.7922343353955315e+28, 1.76733684401992e+28, 1.7429096057464347e+28, 1.7189578458582e+28, 1.6954860327324163e+28, 1.6724978948544875e+28, 1.6499964388086344e+28, 1.627983968168527e+28, 1.6064621032113852e+28, 1.5854318013791623e+28, 1.5648933784108395e+28, 1.544846530070501e+28, 1.5252903543967437e+28, 1.506223374400058e+28, 1.4876435611361182e+28, 1.4695483570844006e+28, 1.4519346997632157e+28, 1.434799045514078e+28, 1.4181373933903133e+28, 1.4019453090869409e+28, 1.3862179488511182e+28, 1.3709500833148045e+28, 1.3561361211937727e+28, 1.341770132799663e+28, 1.3278458733143922e+28, 1.3143568057789514e+28, 1.301296123751351e+28, 1.2886567735912746e+28, 1.276431476331807e+28, 1.264612749101429e+28, 1.2531929260623035e+28, 1.242164178833705e+28, 1.2315185363722388e+28, 1.22124790428329e+28, 1.211344083540878e+28, 1.2017987885957926e+28, 1.1926036648545304e+28, 1.1837503055141525e+28, 1.1752302677406803e+28, 1.1670350881811121e+28, 1.1591562978015063e+28, 1.1515854360458515e+28, 1.1443140643126533e+28, 1.13733377874826e+28, 1.1306362223579585e+28, 1.1242130964377816e+28, 1.118056171331774e+28, 1.1121572965211647e+28, 1.1065084100534978e+28, 1.1011015473212666e+28, 1.0959288492009834e+28, 1.0909825695649026e+28, 1.0862550821787974e+28, 1.0817388870002596e+28, 1.0774266158929695e+28, 1.0733110377732587e+28, 1.0693850632060517e+28, 1.065641748467964e+28, 1.0620742990959049e+28, 1.0586760729400346e+28, 1.0554405827403276e+28, 1.05236149824631e+28, 1.0494326478997844e+28, 1.0466480201005107e+28, 1.044001764074902e+28, 1.0414881903678062e+28, 1.0391017709774022e+28, 1.036837139153117e+28, 1.0346890888763067e+28, 1.0326525740432153e+28, 1.03072270736945e+28, 1.0288947590348848e+28, 1.0271641550875462e+28, 1.0255264756246203e+28, 1.0239774527682882e+28, 1.0225129684536205e+28, 1.0211290520452626e+28, 1.0198218777991219e+28, 1.018587762184717e+28, 1.0174231610832953e+28, 1.0163246668762348e+28, 1.015289005437673e+28, 1.014313033044697e+28, 1.0133937332178266e+28, 1.0125282135039214e+28, 1.0117137022130306e+28, 1.010947545120096e+28, 1.0102272021418212e+28, 1.0095502439984174e+28, 1.0089143488693516e+28, 1.0083172990516316e+28, 1.007756977628607e+28, 1.007231365156689e+28, 1.006738536376862e+28, 1.0062766569573156e+28, 1.0058439802730178e+28, 1.0054388442275482e+28, 1.005059668122023e+28, 1.0047049495754834e+28, 1.0043732615006641e+28, 1.0040632491386403e+28, 1.003773627155424e+28, 1.0035031768032067e+28, 1.0032507431485576e+28, 1.0030152323695395e+28, 1.0027956091233694e+28, 1.0025908939859305e+28, 1.0024001609641457e+28, 1.0022225350819471e+28, 1.002057190040307e+28, 1.001903345951557e+28, 1.0017602671479952e+28, 1.001627260064566e+28, 1.001503671195212e+28, 1.00138888512231e+28, 1.0012823226184483e+28, 1.0011834388196505e+28, 1.001091721469016e+28, 1.0010066892296329e+28, 1.0009278900655037e+28, 1.0008548996891356e+28, 1.0007873200743612e+28, 1.0007247780328815e+28, 1.000666923852967e+28, 1.0006134299986927e+28, 1.0005639898680488e+28, 1.0005183166082287e+28, 1.0004761419863742e+28, 1.0004372153140365e+28, 1.0004013024236052e+28, 1.000368184694949e+28, 1.0003376581305141e+28, 1.0003095324771361e+28, 1.0002836303928288e+28, 1.0002597866568361e+28, 1.0002378474212477e+28, 1.000217669502508e+28, 1.0001991197111734e+28, 1.0001820742183073e+28, 1.0001664179569303e+28, 1.0001520440569833e+28, 1.0001388533122977e+28, 1.0001267536781028e+28, 1.0001156597976482e+28, 1.0001054925565523e+28, 1.0000961786635357e+28, 1.000087650256243e+28, 1.0000798445308913e+28, 1.0000727033945397e+28, 1.0000661731388058e+28, 1.0000602041339084e+28, 1.0000547505419536e+28, 1.000049770048423e+28, 1.000045223610874e+28, 1.000041075223891e+28, 1.0000372916993784e+28, 1.000033842461322e+28, 1.000030699354181e+28, 1.0000278364641228e+28, 1.000025229952333e+28, 1.0000228578996896e+28, 1.0000207001621048e+28, 1.0000187382358918e+28, 1.000016955132529e+28, 1.000015335262243e+28, 1.0000138643258482e+28, 1.0000125292143188e+28, 1.0000113179155963e+28, 1.0000102194281597e+28, 1.0000092236809176e+28, 1.0000083214589994e+28, 1.0000075043350547e+28, 1.0000067646056849e+28, 1.0000060952326607e+28, 1.0000054897885947e+28, 1.00000494240676e+28, 1.0000044477347653e+28, 1.0000040008918146e+28, 1.000003597429294e+28, 1.0000032332944516e+28, 1.0000029047969417e+28]
x_cell_centers_cm_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = [0.1953125, 0.5859375, 0.9765625, 1.3671875, 1.7578125, 2.1484375, 2.5390625, 2.9296875, 3.3203125, 3.7109375, 4.1015625, 4.4921875, 4.8828125, 5.2734375, 5.6640625, 6.0546875, 6.4453125, 6.8359375, 7.2265625, 7.6171875, 8.0078125, 8.3984375, 8.7890625, 9.1796875, 9.5703125, 9.9609375, 10.3515625, 10.7421875, 11.1328125, 11.5234375, 11.9140625, 12.3046875, 12.6953125, 13.0859375, 13.4765625, 13.8671875, 14.2578125, 14.6484375, 15.0390625, 15.4296875, 15.8203125, 16.2109375, 16.6015625, 16.9921875, 17.3828125, 17.7734375, 18.1640625, 18.5546875, 18.9453125, 19.3359375, 19.7265625, 20.1171875, 20.5078125, 20.8984375, 21.2890625, 21.6796875, 22.0703125, 22.4609375, 22.8515625, 23.2421875, 23.6328125, 24.0234375, 24.4140625, 24.8046875, 25.1953125, 25.5859375, 25.9765625, 26.3671875, 26.7578125, 27.1484375, 27.5390625, 27.9296875, 28.3203125, 28.7109375, 29.1015625, 29.4921875, 29.8828125, 30.2734375, 30.6640625, 31.0546875, 31.4453125, 31.8359375, 32.2265625, 32.6171875, 33.0078125, 33.3984375, 33.7890625, 34.1796875, 34.5703125, 34.9609375, 35.3515625, 35.7421875, 36.1328125, 36.5234375, 36.9140625, 37.3046875, 37.6953125, 38.0859375, 38.4765625, 38.8671875, 39.2578125, 39.6484375, 40.0390625, 40.4296875, 40.8203125, 41.2109375, 41.6015625, 41.9921875, 42.3828125, 42.7734375, 43.1640625, 43.5546875, 43.9453125, 44.3359375, 44.7265625, 45.1171875, 45.5078125, 45.8984375, 46.2890625, 46.6796875, 47.0703125, 47.4609375, 47.8515625, 48.2421875, 48.6328125, 49.0234375, 49.4140625, 49.8046875, 50.1953125, 50.5859375, 50.9765625, 51.3671875, 51.7578125, 52.1484375, 52.5390625, 52.9296875, 53.3203125, 53.7109375, 54.1015625, 54.4921875, 54.8828125, 55.2734375, 55.6640625, 56.0546875, 56.4453125, 56.8359375, 57.2265625, 57.6171875, 58.0078125, 58.3984375, 58.7890625, 59.1796875, 59.5703125, 59.9609375, 60.3515625, 60.7421875, 61.1328125, 61.5234375, 61.9140625, 62.3046875, 62.6953125, 63.0859375, 63.4765625, 63.8671875, 64.2578125, 64.6484375, 65.0390625, 65.4296875, 65.8203125, 66.2109375, 66.6015625, 66.9921875, 67.3828125, 67.7734375, 68.1640625, 68.5546875, 68.9453125, 69.3359375, 69.7265625, 70.1171875, 70.5078125, 70.8984375, 71.2890625, 71.6796875, 72.0703125, 72.4609375, 72.8515625, 73.2421875, 73.6328125, 74.0234375, 74.4140625, 74.8046875, 75.1953125, 75.5859375, 75.9765625, 76.3671875, 76.7578125, 77.1484375, 77.5390625, 77.9296875, 78.3203125, 78.7109375, 79.1015625, 79.4921875, 79.8828125, 80.2734375, 80.6640625, 81.0546875, 81.4453125, 81.8359375, 82.2265625, 82.6171875, 83.0078125, 83.3984375, 83.7890625, 84.1796875, 84.5703125, 84.9609375, 85.3515625, 85.7421875, 86.1328125, 86.5234375, 86.9140625, 87.3046875, 87.6953125, 88.0859375, 88.4765625, 88.8671875, 89.2578125, 89.6484375, 90.0390625, 90.4296875, 90.8203125, 91.2109375, 91.6015625, 91.9921875, 92.3828125, 92.7734375, 93.1640625, 93.5546875, 93.9453125, 94.3359375, 94.7265625, 95.1171875, 95.5078125, 95.8984375, 96.2890625, 96.6796875, 97.0703125, 97.4609375, 97.8515625, 98.2421875, 98.6328125, 99.0234375, 99.4140625, 99.8046875, 100.1953125, 100.5859375, 100.9765625, 101.3671875, 101.7578125, 102.1484375, 102.5390625, 102.9296875, 103.3203125, 103.7109375, 104.1015625, 104.4921875, 104.8828125, 105.2734375, 105.6640625, 106.0546875, 106.4453125, 106.8359375, 107.2265625, 107.6171875, 108.0078125, 108.3984375, 108.7890625, 109.1796875, 109.5703125, 109.9609375, 110.3515625, 110.7421875, 111.1328125, 111.5234375, 111.9140625, 112.3046875, 112.6953125, 113.0859375, 113.4765625, 113.8671875, 114.2578125, 114.6484375, 115.0390625, 115.4296875, 115.8203125, 116.2109375, 116.6015625, 116.9921875, 117.3828125, 117.7734375, 118.1640625, 118.5546875, 118.9453125, 119.3359375, 119.7265625, 120.1171875, 120.5078125, 120.8984375, 121.2890625, 121.6796875, 122.0703125, 122.4609375, 122.8515625, 123.2421875, 123.6328125, 124.0234375, 124.4140625, 124.8046875, 125.1953125, 125.5859375, 125.9765625, 126.3671875, 126.7578125, 127.1484375, 127.5390625, 127.9296875, 128.3203125, 128.7109375, 129.1015625, 129.4921875, 129.8828125, 130.2734375, 130.6640625, 131.0546875, 131.4453125, 131.8359375, 132.2265625, 132.6171875, 133.0078125, 133.3984375, 133.7890625, 134.1796875, 134.5703125, 134.9609375, 135.3515625, 135.7421875, 136.1328125, 136.5234375, 136.9140625, 137.3046875, 137.6953125, 138.0859375, 138.4765625, 138.8671875, 139.2578125, 139.6484375, 140.0390625, 140.4296875, 140.8203125, 141.2109375, 141.6015625, 141.9921875, 142.3828125, 142.7734375, 143.1640625, 143.5546875, 143.9453125, 144.3359375, 144.7265625, 145.1171875, 145.5078125, 145.8984375, 146.2890625, 146.6796875, 147.0703125, 147.4609375, 147.8515625, 148.2421875, 148.6328125, 149.0234375, 149.4140625, 149.8046875, 150.1953125, 150.5859375, 150.9765625, 151.3671875, 151.7578125, 152.1484375, 152.5390625, 152.9296875, 153.3203125, 153.7109375, 154.1015625, 154.4921875, 154.8828125, 155.2734375, 155.6640625, 156.0546875, 156.4453125, 156.8359375, 157.2265625, 157.6171875, 158.0078125, 158.3984375, 158.7890625, 159.1796875, 159.5703125, 159.9609375, 160.3515625, 160.7421875, 161.1328125, 161.5234375, 161.9140625, 162.3046875, 162.6953125, 163.0859375, 163.4765625, 163.8671875, 164.2578125, 164.6484375, 165.0390625, 165.4296875, 165.8203125, 166.2109375, 166.6015625, 166.9921875, 167.3828125, 167.7734375, 168.1640625, 168.5546875, 168.9453125, 169.3359375, 169.7265625, 170.1171875, 170.5078125, 170.8984375, 171.2890625, 171.6796875, 172.0703125, 172.4609375, 172.8515625, 173.2421875, 173.6328125, 174.0234375, 174.4140625, 174.8046875, 175.1953125, 175.5859375, 175.9765625, 176.3671875, 176.7578125, 177.1484375, 177.5390625, 177.9296875, 178.3203125, 178.7109375, 179.1015625, 179.4921875, 179.8828125, 180.2734375, 180.6640625, 181.0546875, 181.4453125, 181.8359375, 182.2265625, 182.6171875, 183.0078125, 183.3984375, 183.7890625, 184.1796875, 184.5703125, 184.9609375, 185.3515625, 185.7421875, 186.1328125, 186.5234375, 186.9140625, 187.3046875, 187.6953125, 188.0859375, 188.4765625, 188.8671875, 189.2578125, 189.6484375, 190.0390625, 190.4296875, 190.8203125, 191.2109375, 191.6015625, 191.9921875, 192.3828125, 192.7734375, 193.1640625, 193.5546875, 193.9453125, 194.3359375, 194.7265625, 195.1171875, 195.5078125, 195.8984375, 196.2890625, 196.6796875, 197.0703125, 197.4609375, 197.8515625, 198.2421875, 198.6328125, 199.0234375, 199.4140625, 199.8046875]

last_time_s_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 9.999999999999707e-09
last_nee_invccm_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = [1.0000271604885632e+28, 1.0000281223874306e+28, 1.0000300739458203e+28, 1.0000330712949458e+28, 1.0000372001446317e+28, 1.0000425776217769e+28, 1.0000493547584691e+28, 1.000057719661787e+28, 1.0000679014055281e+28, 1.0000801746913118e+28, 1.0000948653350602e+28, 1.000112356642572e+28, 1.0001330967459333e+28, 1.000157606979873e+28, 1.000186491384755e+28, 1.0002204474303355e+28, 1.0002602780614263e+28, 1.0003069051722714e+28, 1.0003613846227557e+28, 1.000424922914783e+28, 1.0004988956507822e+28, 1.0005848678991618e+28, 1.0006846165942865e+28, 1.0008001550976697e+28, 1.0009337600457626e+28, 1.0010880006064805e+28, 1.0012657702605628e+28, 1.0014703212153797e+28, 1.0017053015477011e+28, 1.00197479515777e+28, 1.0022833645989397e+28, 1.0026360968261138e+28, 1.0030386518808702e+28, 1.003497314501426e+28, 1.00401904861213e+28, 1.0046115546084244e+28, 1.005283329310401e+28, 1.0060437284098677e+28, 1.0069030311834096e+28, 1.0078725071864418e+28, 1.008964484581313e+28, 1.0101924196856997e+28, 1.0115709672568072e+28, 1.0131160509535167e+28, 1.0148449333418774e+28, 1.016776284727425e+28, 1.0189302500177672e+28, 1.0213285127374086e+28, 1.0239943552340755e+28, 1.026952714035737e+28, 1.0302302292400296e+28, 1.0338552867447427e+28, 1.037858052060335e+28, 1.0422704943854967e+28, 1.0471263995753783e+28, 1.0524613705926288e+28, 1.0583128140037407e+28, 1.064719911069739e+28, 1.071723571984387e+28, 1.0793663718347098e+28, 1.0876924668990528e+28, 1.0967474899616363e+28, 1.1065784234074675e+28, 1.1172334489710844e+28, 1.1287617731469718e+28, 1.1412134274295572e+28, 1.1546390427365352e+28, 1.1690895975817766e+28, 1.1846161398013435e+28, 1.2012694818989237e+28, 1.2190998703629882e+28, 1.2381566296162407e+28, 1.2584877815853646e+28, 1.2801396422236863e+28, 1.303156396677776e+28, 1.3275796551574849e+28, 1.353447991943254e+28, 1.3807964703398783e+28, 1.4096561567586883e+28, 1.440053627472534e+28, 1.4720104719372166e+28, 1.5055427969009245e+28, 1.5406607358253152e+28, 1.5773679684118726e+28, 1.615661255258927e+28, 1.655529992862285e+28, 1.6969557943106546e+28, 1.7399121011103971e+28, 1.7843638315989459e+28, 1.830267071366655e+28, 1.8775688110005649e+28, 1.9262067362887351e+28, 1.9761090757757181e+28, 2.0271945102417032e+28, 2.079372148286001e+28, 2.1325415717342784e+28, 2.186592954058733e+28, 2.241407254405188e+28, 2.2968564891662527e+28, 2.352804082330418e+28, 2.409105295078786e+28, 2.4656077343053197e+28, 2.522151938907959e+28, 2.578572041848963e+28, 2.6346965051218944e+28, 2.6903489239031867e+28, 2.7453488953160134e+28, 2.7995129464087705e+28, 2.8526555151572284e+28, 2.9045899775548695e+28, 2.955129713167145e+28, 3.004089200904527e+28, 3.051285136227851e+28, 3.096537560546055e+28, 3.1396709932066795e+28, 3.1805155562247566e+28, 3.218908081749886e+28, 3.2546931922378914e+28, 3.2877243433762496e+28, 3.317864820011787e+28, 3.344988675646968e+28, 3.3689816064995134e+28, 3.3897417516638383e+28, 3.4071804115595286e+28, 3.421222677593054e+28, 3.4318079667961073e+28, 3.4388904561142283e+28, 3.442439411999593e+28, 3.4424394120004943e+28, 3.438890456116885e+28, 3.4318079668006355e+28, 3.4212226775995652e+28, 3.4071804115681827e+28, 3.3897417516744077e+28, 3.368981606511378e+28, 3.3449886756595696e+28, 3.3178648200244217e+28, 3.2877243433886793e+28, 3.2546931922500625e+28, 3.2189080817615643e+28, 3.1805155562357306e+28, 3.1396709932167335e+28, 3.096537560555057e+28, 3.0512851362360256e+28, 3.004089200912185e+28, 2.955129713174228e+28, 2.9045899775609353e+28, 2.8526555151620157e+28, 2.7995129464121786e+28, 2.745348895317904e+28, 2.6903489239035407e+28, 2.634696505120834e+28, 2.5785720418464284e+28, 2.522151938903765e+28, 2.465607734299234e+28, 2.409105295070602e+28, 2.352804082319852e+28, 2.2968564891530045e+28, 2.241407254389e+28, 2.186592954039424e+28, 2.1325415717115788e+28, 2.0793721482595424e+28, 2.027194510211177e+28, 1.9761090757404667e+28, 1.9262067362485258e+28, 1.8775688109561752e+28, 1.8302670713186362e+28, 1.7843638315477706e+28, 1.7399121010564366e+28, 1.6969557942542196e+28, 1.6555299928036422e+28, 1.6156612551983506e+28, 1.5773679683496227e+28, 1.5406607357615466e+28, 1.5055427968357767e+28, 1.4720104718707712e+28, 1.4400536274049743e+28, 1.4096561566902556e+28, 1.3807964702707558e+28, 1.3534479918735727e+28, 1.3275796550873206e+28, 1.303156396607188e+28, 1.2801396421528038e+28, 1.2584877815142917e+28, 1.2381566295450777e+28, 1.2190998702918295e+28, 1.2012694818278352e+28, 1.184616139730371e+28, 1.1690895975109463e+28, 1.1546390426658718e+28, 1.1412134273590844e+28, 1.1287617730767211e+28, 1.1172334489010835e+28, 1.1065784233377288e+28, 1.0967474898921936e+28, 1.0876924668299795e+28, 1.0793663717660612e+28, 1.071723571916185e+28, 1.0647199110020232e+28, 1.0583128139365847e+28, 1.052461370525959e+28, 1.0471263995092018e+28, 1.0422704943198277e+28, 1.0378580519951546e+28, 1.0338552866800072e+28, 1.0302302291755637e+28, 1.0269527139713788e+28, 1.0239943551696753e+28, 1.0213285126728229e+28, 1.018930249952868e+28, 1.0167762846620349e+28, 1.0148449332758121e+28, 1.0131160508866013e+28, 1.0115709671889946e+28, 1.0101924196168234e+28, 1.008964484510909e+28, 1.0078725071143172e+28, 1.0069030311098494e+28, 1.0060437283352183e+28, 1.0052833292347892e+28, 1.0046115545318789e+28, 1.0040190485347353e+28, 1.0034973144232907e+28, 1.0030386518020948e+28, 1.0026360967467975e+28, 1.0022833645191517e+28, 1.001974795077595e+28, 1.0017053014672442e+28, 1.001470321134799e+28, 1.0012657701800204e+28, 1.001088000526156e+28, 1.0009337599658433e+28, 1.000800155018347e+28, 1.000684616515737e+28, 1.0005848678215546e+28, 1.0004988955743285e+28, 1.0004249228396616e+28, 1.000361384549109e+28, 1.0003069051003724e+28, 1.0002602779914974e+28, 1.0002204473625416e+28, 1.0001864913193176e+28, 1.0001576069170403e+28, 1.0001330966858905e+28, 1.0001123565855343e+28, 1.0000948652813128e+28, 1.0000801746411e+28, 1.0000679013591032e+28, 1.000057719619421e+28, 1.000049354720475e+28, 1.0000425775884918e+28, 1.0000372001163626e+28, 1.0000330712720905e+28, 1.0000300739288271e+28, 1.0000281223766864e+28, 1.0000271604847822e+28]
last_analytic_solution_invccm_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = [1.0000030648083796e+28, 1.00000379400017e+28, 1.000004688797658e+28, 1.0000057848987579e+28, 1.0000071252508694e+28, 1.0000087614236908e+28, 1.00001075522082e+28, 1.0000131805673944e+28, 1.0000161257159851e+28, 1.0000196958184652e+28, 1.0000240159176006e+28, 1.0000292344187017e+28, 1.0000355271088388e+28, 1.0000431017988749e+28, 1.0000522036718956e+28, 1.0000631214305104e+28, 1.0000761943449455e+28, 1.000091820313788e+28, 1.0001104650596257e+28, 1.0001326725925817e+28, 1.0001590770857518e+28, 1.0001904163176948e+28, 1.000227546848249e+28, 1.0002714611048452e+28, 1.0003233065669587e+28, 1.0003844072461038e+28, 1.000456287667551e+28, 1.0005406995673826e+28, 1.0006396515242307e+28, 1.0007554417486398e+28, 1.0008906942539962e+28, 1.0010483986308936e+28, 1.0012319536410952e+28, 1.001445214837372e+28, 1.0016925464008276e+28, 1.0019788773672842e+28, 1.0023097623882416e+28, 1.0026914471392658e+28, 1.0031309384487644e+28, 1.0036360791724141e+28, 1.0042156277824855e+28, 1.0048793425764843e+28, 1.0056380703355138e+28, 1.006503839179283e+28, 1.0074899552715601e+28, 1.0086111029271116e+28, 1.0098834475589269e+28, 1.0113247407831531e+28, 1.0129544268692564e+28, 1.0147937495852744e+28, 1.0168658583437348e+28, 1.0191959124042176e+28, 1.0218111817353118e+28, 1.024741142983794e+28, 1.028017568844504e+28, 1.0316746089731713e+28, 1.0357488604392424e+28, 1.0402794255796702e+28, 1.0453079549911207e+28, 1.0508786732907098e+28, 1.0570383851880313e+28, 1.0638364593478604e+28, 1.0713247874874732e+28, 1.0795577161491138e+28, 1.0885919486206512e+28, 1.0984864145497501e+28, 1.1093021049124288e+28, 1.1211018701589063e+28, 1.1339501795708763e+28, 1.147912840126954e+28, 1.163056673488531e+28, 1.1794491500873908e+28, 1.1971579797190313e+28, 1.216250658520647e+28, 1.2367939727380134e+28, 1.258853460257845e+28, 1.282492831497232e+28, 1.3077733518938986e+28, 1.3347531889235528e+28, 1.3634867272755814e+28, 1.3940238565367406e+28, 1.4264092364541904e+28, 1.4606815455631575e+28, 1.4968727196587733e+28, 1.5350071872536515e+28, 1.5751011097794665e+28, 1.6171616348488138e+28, 1.6611861713795778e+28, 1.7071616957846963e+28, 1.7550640987328596e+28, 1.8048575821783525e+28, 1.8564941164299668e+28, 1.9099129669700951e+28, 1.9650403005376794e+28, 2.021788879646511e+28, 2.080057854219326e+28, 2.1397326583765182e+28, 2.2006850196267446e+28, 2.2627730867686465e+28, 2.3258416817344277e+28, 2.3897226793959067e+28, 2.454235518023601e+28, 2.519187841653559e+28, 2.584376274092e+28, 2.6495873226934743e+28, 2.7145984084055314e+28, 2.779179016905014e+28, 2.8430919639825793e+28, 2.9060947666886043e+28, 2.967941110161583e+28, 3.0283823985458494e+28, 3.087169376995158e+28, 3.144053810477541e+28, 3.198790203968944e+28, 3.251137547670477e+28, 3.300861070126547e+28, 3.3477339815757324e+28, 3.3915391895469196e+28, 3.432070968630399e+28, 3.469136566514048e+28, 3.5025577287810303e+28, 3.532172125616282e+28, 3.557834664458916e+28, 3.579418673756721e+28, 3.596816944313638e+28, 3.6099426162537827e+28, 3.618729901335284e+28, 3.6231346322095343e+28, 3.6231346322095343e+28, 3.618729901335284e+28, 3.6099426162537827e+28, 3.596816944313638e+28, 3.579418673756721e+28, 3.557834664458916e+28, 3.532172125616282e+28, 3.5025577287810303e+28, 3.469136566514048e+28, 3.432070968630399e+28, 3.3915391895469196e+28, 3.3477339815757324e+28, 3.300861070126547e+28, 3.251137547670477e+28, 3.198790203968944e+28, 3.144053810477541e+28, 3.087169376995158e+28, 3.0283823985458494e+28, 2.967941110161583e+28, 2.9060947666886043e+28, 2.8430919639825793e+28, 2.779179016905014e+28, 2.7145984084055314e+28, 2.6495873226934743e+28, 2.584376274092e+28, 2.519187841653559e+28, 2.454235518023601e+28, 2.3897226793959067e+28, 2.3258416817344277e+28, 2.2627730867686465e+28, 2.2006850196267446e+28, 2.1397326583765182e+28, 2.080057854219326e+28, 2.021788879646511e+28, 1.9650403005376794e+28, 1.9099129669700951e+28, 1.8564941164299668e+28, 1.8048575821783525e+28, 1.7550640987328596e+28, 1.7071616957846963e+28, 1.6611861713795778e+28, 1.6171616348488138e+28, 1.5751011097794665e+28, 1.5350071872536515e+28, 1.4968727196587733e+28, 1.4606815455631575e+28, 1.4264092364541904e+28, 1.3940238565367406e+28, 1.3634867272755814e+28, 1.3347531889235528e+28, 1.3077733518938986e+28, 1.282492831497232e+28, 1.258853460257845e+28, 1.2367939727380134e+28, 1.216250658520647e+28, 1.1971579797190313e+28, 1.1794491500873908e+28, 1.163056673488531e+28, 1.147912840126954e+28, 1.1339501795708763e+28, 1.1211018701589063e+28, 1.1093021049124288e+28, 1.0984864145497501e+28, 1.0885919486206512e+28, 1.0795577161491138e+28, 1.0713247874874732e+28, 1.0638364593478604e+28, 1.0570383851880313e+28, 1.0508786732907098e+28, 1.0453079549911207e+28, 1.0402794255796702e+28, 1.0357488604392424e+28, 1.0316746089731713e+28, 1.028017568844504e+28, 1.024741142983794e+28, 1.0218111817353118e+28, 1.0191959124042176e+28, 1.0168658583437348e+28, 1.0147937495852744e+28, 1.0129544268692564e+28, 1.0113247407831531e+28, 1.0098834475589269e+28, 1.0086111029271116e+28, 1.0074899552715601e+28, 1.006503839179283e+28, 1.0056380703355138e+28, 1.0048793425764843e+28, 1.0042156277824855e+28, 1.0036360791724141e+28, 1.0031309384487644e+28, 1.0026914471392658e+28, 1.0023097623882416e+28, 1.0019788773672842e+28, 1.0016925464008276e+28, 1.001445214837372e+28, 1.0012319536410952e+28, 1.0010483986308936e+28, 1.0008906942539962e+28, 1.0007554417486398e+28, 1.0006396515242307e+28, 1.0005406995673826e+28, 1.000456287667551e+28, 1.0003844072461038e+28, 1.0003233065669587e+28, 1.0002714611048452e+28, 1.000227546848249e+28, 1.0001904163176948e+28, 1.0001590770857518e+28, 1.0001326725925817e+28, 1.0001104650596257e+28, 1.000091820313788e+28, 1.0000761943449455e+28, 1.0000631214305104e+28, 1.0000522036718956e+28, 1.0000431017988749e+28, 1.0000355271088388e+28, 1.0000292344187017e+28, 1.0000240159176006e+28, 1.0000196958184652e+28, 1.0000161257159851e+28, 1.0000131805673944e+28, 1.00001075522082e+28, 1.0000087614236908e+28, 1.0000071252508694e+28, 1.0000057848987579e+28, 1.000004688797658e+28, 1.00000379400017e+28, 1.0000030648083796e+28]
x_cell_centers_cm_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = [0.390625, 1.171875, 1.953125, 2.734375, 3.515625, 4.296875, 5.078125, 5.859375, 6.640625, 7.421875, 8.203125, 8.984375, 9.765625, 10.546875, 11.328125, 12.109375, 12.890625, 13.671875, 14.453125, 15.234375, 16.015625, 16.796875, 17.578125, 18.359375, 19.140625, 19.921875, 20.703125, 21.484375, 22.265625, 23.046875, 23.828125, 24.609375, 25.390625, 26.171875, 26.953125, 27.734375, 28.515625, 29.296875, 30.078125, 30.859375, 31.640625, 32.421875, 33.203125, 33.984375, 34.765625, 35.546875, 36.328125, 37.109375, 37.890625, 38.671875, 39.453125, 40.234375, 41.015625, 41.796875, 42.578125, 43.359375, 44.140625, 44.921875, 45.703125, 46.484375, 47.265625, 48.046875, 48.828125, 49.609375, 50.390625, 51.171875, 51.953125, 52.734375, 53.515625, 54.296875, 55.078125, 55.859375, 56.640625, 57.421875, 58.203125, 58.984375, 59.765625, 60.546875, 61.328125, 62.109375, 62.890625, 63.671875, 64.453125, 65.234375, 66.015625, 66.796875, 67.578125, 68.359375, 69.140625, 69.921875, 70.703125, 71.484375, 72.265625, 73.046875, 73.828125, 74.609375, 75.390625, 76.171875, 76.953125, 77.734375, 78.515625, 79.296875, 80.078125, 80.859375, 81.640625, 82.421875, 83.203125, 83.984375, 84.765625, 85.546875, 86.328125, 87.109375, 87.890625, 88.671875, 89.453125, 90.234375, 91.015625, 91.796875, 92.578125, 93.359375, 94.140625, 94.921875, 95.703125, 96.484375, 97.265625, 98.046875, 98.828125, 99.609375, 100.390625, 101.171875, 101.953125, 102.734375, 103.515625, 104.296875, 105.078125, 105.859375, 106.640625, 107.421875, 108.203125, 108.984375, 109.765625, 110.546875, 111.328125, 112.109375, 112.890625, 113.671875, 114.453125, 115.234375, 116.015625, 116.796875, 117.578125, 118.359375, 119.140625, 119.921875, 120.703125, 121.484375, 122.265625, 123.046875, 123.828125, 124.609375, 125.390625, 126.171875, 126.953125, 127.734375, 128.515625, 129.296875, 130.078125, 130.859375, 131.640625, 132.421875, 133.203125, 133.984375, 134.765625, 135.546875, 136.328125, 137.109375, 137.890625, 138.671875, 139.453125, 140.234375, 141.015625, 141.796875, 142.578125, 143.359375, 144.140625, 144.921875, 145.703125, 146.484375, 147.265625, 148.046875, 148.828125, 149.609375, 150.390625, 151.171875, 151.953125, 152.734375, 153.515625, 154.296875, 155.078125, 155.859375, 156.640625, 157.421875, 158.203125, 158.984375, 159.765625, 160.546875, 161.328125, 162.109375, 162.890625, 163.671875, 164.453125, 165.234375, 166.015625, 166.796875, 167.578125, 168.359375, 169.140625, 169.921875, 170.703125, 171.484375, 172.265625, 173.046875, 173.828125, 174.609375, 175.390625, 176.171875, 176.953125, 177.734375, 178.515625, 179.296875, 180.078125, 180.859375, 181.640625, 182.421875, 183.203125, 183.984375, 184.765625, 185.546875, 186.328125, 187.109375, 187.890625, 188.671875, 189.453125, 190.234375, 191.015625, 191.796875, 192.578125, 193.359375, 194.140625, 194.921875, 195.703125, 196.484375, 197.265625, 198.046875, 198.828125, 199.609375]

last_time_s_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 9.999999999999707e-09
last_nee_invccm_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = [1.000717643253848e+28, 1.0007683813412653e+28, 1.0008725041883744e+28, 1.0010353862724743e+28, 1.001265292075115e+28, 1.001573619216965e+28, 1.0019752220912975e+28, 1.0024888150514186e+28, 1.0031374533872489e+28, 1.0039490891366097e+28, 1.0049571971615465e+28, 1.0062014648402274e+28, 1.007728536157132e+28, 1.0095927979126333e+28, 1.0118571922391598e+28, 1.0145940356547852e+28, 1.0178858205852365e+28, 1.0218259707584665e+28, 1.0265195172740256e+28, 1.0320836576604232e+28, 1.0386481560842608e+28, 1.0463555393188023e+28, 1.0553610404007887e+28, 1.06583224040707e+28, 1.0779483587638498e+28, 1.0918991442660156e+28, 1.1078833228063578e+28, 1.1261065639199292e+28, 1.1467789368127553e+28, 1.1701118376524867e+28, 1.1963143835406333e+28, 1.2255892846345461e+28, 1.2581282240886303e+28, 1.2941067954447338e+28, 1.3336790682888744e+28, 1.3769718747424011e+28, 1.4240789308816234e+28, 1.4750549275953099e+28, 1.529909743736147e+28, 1.5886029497068053e+28, 1.6510387808631315e+28, 1.7170617663904486e+28, 1.7864531997940888e+28, 1.8589286311941067e+28, 1.9341365487720332e+28, 2.0116583967939024e+28, 2.091010050722266e+28, 2.17164483643093e+28, 2.2529581411732265e+28, 2.3342936197626437e+28, 2.4149509517297964e+28, 2.494195055595817e+28, 2.5712666166247152e+28, 2.6453937363785517e+28, 2.715804468046226e+28, 2.7817399627474045e+28, 2.842467920598457e+28, 2.8972960178143843e+28, 2.945584968747202e+28, 2.986760880402381e+28, 3.020326567072795e+28, 3.0458715142488887e+28, 3.063080213415339e+28, 3.071738631757024e+28, 3.0717386317579868e+28, 3.0630802134183544e+28, 3.04587151425396e+28, 3.020326567079104e+28, 2.9867608804086336e+28, 2.945584968752566e+28, 2.8972960178183364e+28, 2.842467920600642e+28, 2.781739962747886e+28, 2.71580446804461e+28, 2.645393736374046e+28, 2.571266616617139e+28, 2.4941950555850465e+28, 2.4149509517153422e+28, 2.33429361974374e+28, 2.2529581411490434e+28, 2.1716448364006404e+28, 2.0910100506848315e+28, 2.0116583967485524e+28, 1.9341365487202354e+28, 1.8589286311370843e+28, 1.7864531997328379e+28, 1.71706176632577e+28, 1.651038780795609e+28, 1.588602949636992e+28, 1.5299097436646881e+28, 1.475054927522616e+28, 1.424078930808055e+28, 1.3769718746683756e+28, 1.3336790682146813e+28, 1.2941067953706036e+28, 1.2581282240146894e+28, 1.2255892845609587e+28, 1.1963143834676049e+28, 1.1701118375802226e+28, 1.146778936741441e+28, 1.1261065638496243e+28, 1.107883322737036e+28, 1.0918991441975136e+28, 1.0779483586957933e+28, 1.0658322403390038e+28, 1.0553610403322348e+28, 1.0463555392493396e+28, 1.0386481560133291e+28, 1.0320836575877336e+28, 1.0265195172000274e+28, 1.021825970683624e+28, 1.0178858205098729e+28, 1.0145940355792512e+28, 1.0118571921637866e+28, 1.009592797837868e+28, 1.0077285360835244e+28, 1.0062014647683816e+28, 1.0049571970920818e+28, 1.0039490890701288e+28, 1.003137453324423e+28, 1.002488814992941e+28, 1.0019752220378498e+28, 1.0015736191692357e+28, 1.001265292033857e+28, 1.0010353862384967e+28, 1.0008725041626176e+28, 1.0007683813248447e+28, 1.0007176432480044e+28]
last_analytic_solution_invccm_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = [1.00000341068468e+28, 1.00000520918596e+28, 1.000007902757499e+28, 1.0000119087986677e+28, 1.0000178253371915e+28, 1.0000265025741366e+28, 1.0000391398237375e+28, 1.000057415635523e+28, 1.000083660795618e+28, 1.0001210861045735e+28, 1.0001740792987776e+28, 1.0002485881673723e+28, 1.0003526097201813e+28, 1.0004968080345365e+28, 1.0006952859361325e+28, 1.0009665376637427e+28, 1.0013346107627171e+28, 1.0018305051994168e+28, 1.0024938355645127e+28, 1.003374777653662e+28, 1.0045363130624033e+28, 1.0060567740984929e+28, 1.0080326757518574e+28, 1.0105818012543875e+28, 1.0138464827076447e+28, 1.017996988460461e+28, 1.0232348948843088e+28, 1.029796282914654e+28, 1.0379545607546463e+28, 1.0480226756320194e+28, 1.0603544422269507e+28, 1.0753446866694197e+28, 1.0934278865987598e+28, 1.1150749836933716e+28, 1.1407880593070555e+28, 1.171092600034883e+28, 1.206527141087999e+28, 1.2476301630473863e+28, 1.2949242320955333e+28, 1.3488975134670378e+28, 1.4099829486790043e+28, 1.4785355628068673e+28, 1.554808550039544e+28, 1.6389289632670934e+28, 1.7308739941850338e+28, 1.8304489610652946e+28, 1.9372682086204385e+28, 2.0507401559456613e+28, 2.1700576941490023e+28, 2.2941950279866753e+28, 2.4219118728397754e+28, 2.5517656619228697e+28, 2.6821320963628277e+28, 2.811233995865594e+28, 2.937177998303741e+28, 3.05799823509923e+28, 3.1717057010054552e+28, 3.2763416682695617e+28, 3.3700331919249256e+28, 3.4510485381559006e+28, 3.5178502596968465e+28, 3.569143653199562e+28, 3.603918468041818e+28, 3.621481990581244e+28, 3.621481990581244e+28, 3.603918468041818e+28, 3.569143653199562e+28, 3.5178502596968465e+28, 3.4510485381559006e+28, 3.3700331919249256e+28, 3.2763416682695617e+28, 3.1717057010054552e+28, 3.05799823509923e+28, 2.937177998303741e+28, 2.811233995865594e+28, 2.6821320963628277e+28, 2.5517656619228697e+28, 2.4219118728397754e+28, 2.2941950279866753e+28, 2.1700576941490023e+28, 2.0507401559456613e+28, 1.9372682086204385e+28, 1.8304489610652946e+28, 1.7308739941850338e+28, 1.6389289632670934e+28, 1.554808550039544e+28, 1.4785355628068673e+28, 1.4099829486790043e+28, 1.3488975134670378e+28, 1.2949242320955333e+28, 1.2476301630473863e+28, 1.206527141087999e+28, 1.171092600034883e+28, 1.1407880593070555e+28, 1.1150749836933716e+28, 1.0934278865987598e+28, 1.0753446866694197e+28, 1.0603544422269507e+28, 1.0480226756320194e+28, 1.0379545607546463e+28, 1.029796282914654e+28, 1.0232348948843088e+28, 1.017996988460461e+28, 1.0138464827076447e+28, 1.0105818012543875e+28, 1.0080326757518574e+28, 1.0060567740984929e+28, 1.0045363130624033e+28, 1.003374777653662e+28, 1.0024938355645127e+28, 1.0018305051994168e+28, 1.0013346107627171e+28, 1.0009665376637427e+28, 1.0006952859361325e+28, 1.0004968080345365e+28, 1.0003526097201813e+28, 1.0002485881673723e+28, 1.0001740792987776e+28, 1.0001210861045735e+28, 1.000083660795618e+28, 1.000057415635523e+28, 1.0000391398237375e+28, 1.0000265025741366e+28, 1.0000178253371915e+28, 1.0000119087986677e+28, 1.000007902757499e+28, 1.00000520918596e+28, 1.00000341068468e+28]
x_cell_centers_cm_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = [0.78125, 2.34375, 3.90625, 5.46875, 7.03125, 8.59375, 10.15625, 11.71875, 13.28125, 14.84375, 16.40625, 17.96875, 19.53125, 21.09375, 22.65625, 24.21875, 25.78125, 27.34375, 28.90625, 30.46875, 32.03125, 33.59375, 35.15625, 36.71875, 38.28125, 39.84375, 41.40625, 42.96875, 44.53125, 46.09375, 47.65625, 49.21875, 50.78125, 52.34375, 53.90625, 55.46875, 57.03125, 58.59375, 60.15625, 61.71875, 63.28125, 64.84375, 66.40625, 67.96875, 69.53125, 71.09375, 72.65625, 74.21875, 75.78125, 77.34375, 78.90625, 80.46875, 82.03125, 83.59375, 85.15625, 86.71875, 88.28125, 89.84375, 91.40625, 92.96875, 94.53125, 96.09375, 97.65625, 99.21875, 100.78125, 102.34375, 103.90625, 105.46875, 107.03125, 108.59375, 110.15625, 111.71875, 113.28125, 114.84375, 116.40625, 117.96875, 119.53125, 121.09375, 122.65625, 124.21875, 125.78125, 127.34375, 128.90625, 130.46875, 132.03125, 133.59375, 135.15625, 136.71875, 138.28125, 139.84375, 141.40625, 142.96875, 144.53125, 146.09375, 147.65625, 149.21875, 150.78125, 152.34375, 153.90625, 155.46875, 157.03125, 158.59375, 160.15625, 161.71875, 163.28125, 164.84375, 166.40625, 167.96875, 169.53125, 171.09375, 172.65625, 174.21875, 175.78125, 177.34375, 178.90625, 180.46875, 182.03125, 183.59375, 185.15625, 186.71875, 188.28125, 189.84375, 191.40625, 192.96875, 194.53125, 196.09375, 197.65625, 199.21875]

In [ ]:
fontsize = 28
ncells = [1024, 768, 512, 256, 128]
dxcell = Lx / np.array(ncells)
prefix = "gaussian_diffusion_test_ncell_{n}_1_1_1_dts_1p0emin12"
unitsy = 1e28

fig, ax = plt.subplots(figsize=(10, 6))
for n, dx in zip(ncells, dxcell):
    tag = prefix.format(n=n)
    x = np.asarray(globals()[f"x_cell_centers_cm_{tag}"])
    nee_last = np.asarray(globals()[f"last_nee_invccm_{tag}"])/unitsy
    ax.plot(x, nee_last, linewidth=4, label=rf"$\Delta x_{{\mathrm{{cell}}}} = {dx:.3f}\,\mathrm{{cm}}$")

tag_ref = prefix.format(n=1024)
ax.plot(
    np.asarray(globals()[f"x_cell_centers_cm_{tag_ref}"]),
    np.asarray(globals()[f"last_analytic_solution_invccm_{tag_ref}"])/unitsy,
    color="k", linestyle="--", linewidth=4, label="analytic",
)
ax.set_xlim(40.0, 160.0)
ax.set_ylim(0.8,3.8)
ax.set_xlabel(r"$x \, [\mathrm{cm}]$", fontsize=fontsize)
ax.set_ylabel(r"$n_{ee} \, [\,10^{28}\,\mathrm{cm}^{-3}\,]$", fontsize=fontsize)
leg = ax.legend(framealpha=0.0, ncol=1, fontsize=20)
pf.apply_custom_settings(ax, leg, False)
fig.tight_layout()
fig.savefig("plots/gaussian_diffusion_test_n_ee_dxcell.pdf", bbox_inches="tight")
plt.show()

# Compute errors

In [ ]:
# Discrete error measurements
n_i = nee[-1, :, 0, 0] # inv ccm
n_i_star = analytic_solution_all_times[-1, :] # inv ccm
e_i = n_i - n_i_star # inv ccm
q_i = n_i - n_bg_ccm # inv ccm
q_i_star = n_i_star - n_bg_ccm # inv ccm
V_omega_ccm = Lx*Ly*Lz # ccm
V_i_ccm = dx*dy*dz # ccm
Q_star = np.sum(q_i_star*V_i_ccm) # adimensional
A_star = np.max(q_i_star) # adimensional

# Maximum absolute and relative errors
E_infty = np.max(np.abs(e_i)) # adimensional
epsilon_infty = E_infty / A_star # adimensional
print_variable("E_infty", E_infty)
print_variable("epsilon_infty", epsilon_infty)

# Integrated absolute error
E_1 = np.sum(np.abs(e_i)*V_i_ccm) # adimensional
epsilon_1 = E_1 / np.sum(np.abs(q_i_star)*V_i_ccm) # adimensional
print_variable("E_1", E_1)
print_variable("epsilon_1", epsilon_1)

# L2 and RMS errors
E_2 = np.sqrt(np.sum(e_i**2*V_i_ccm)) # adimensional
E_RMS = np.sqrt(np.sum(e_i**2*V_i_ccm)/V_omega_ccm) # adimensional
epsilon_2 = E_2 / np.sqrt(np.sum(q_i_star**2*V_i_ccm)) # adimensional
print_variable("E_2", E_2)
print_variable("E_RMS", E_RMS)
print_variable("epsilon_2", epsilon_2)

In [ ]:
###############################
# Numerical diffusion included
################################

E_infty_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 1.669157730784471e+26
epsilon_infty_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 0.006361964843064699
E_1_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 2.8385346667458073e+26
epsilon_1_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 0.005937098484046905
E_2_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 1.6141042139728044e+26
E_RMS_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 5.8436814604387944e+25
epsilon_2_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 0.0054196738777517964

E_infty_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 2.6036815319498393e+26
epsilon_infty_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 0.00992398694295719
E_1_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 4.496813083362876e+26
epsilon_1_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 0.00940556493906921
E_2_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 2.535162461627555e+26
E_RMS_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 9.178268508295294e+25
epsilon_2_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 0.008512308963819666

E_infty_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 5.2183591710820785e+26
epsilon_infty_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 0.019890466116226368
E_1_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 9.172344342863033e+26
epsilon_1_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 0.019184938034404925
E_2_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 5.123654666976469e+26
E_RMS_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 1.8549611312523074e+26
epsilon_2_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 0.017203683081209842

E_infty_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 1.8069522020994113e+27
epsilon_infty_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 0.06888522532971815
E_1_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 3.2889379013615804e+27
epsilon_1_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 0.06879164961920403
E_2_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 1.8048556417380967e+27
E_RMS_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 6.534275396279327e+26
epsilon_2_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 0.06060159492778404

E_infty_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 5.497433588242202e+27
epsilon_infty_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 0.20970708965363866
E_1_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 1.0841533889329293e+28
epsilon_1_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 0.22676226257010648
E_2_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 5.699845055147517e+27
E_RMS_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 2.0635643341861992e+27
epsilon_2_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 0.1913835617626123

###############################
# No numerical diffusion included
################################

# E_infty_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 4.569734162834142e+25
# epsilon_infty_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 0.0017498306049064886
# E_1_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 7.221150455240304e+25
# epsilon_1_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 0.001510380790867441
# E_2_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 4.30452598452464e+25
# E_RMS_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 1.5584048708869493e+25
# epsilon_2_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12 = 0.0014486802371742862

# E_infty_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 4.60281510881629e+25
# epsilon_infty_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 0.001768823267307593
# E_1_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 7.297303056911871e+25
# epsilon_1_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 0.0015263089497042754
# E_2_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 4.343078692396875e+25
# E_RMS_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 1.5723624420457646e+25
# epsilon_2_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12 = 0.001464268295748201

# E_infty_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 4.685619567188168e+25
# epsilon_infty_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 0.0018189164021180347
# E_1_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 7.499063553231595e+25
# epsilon_1_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 0.0015685094127648634
# E_2_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 4.442789549665867e+25
# E_RMS_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 1.6084616283921728e+25
# epsilon_2_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12 = 0.0015054456965149904

# E_infty_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 4.902864818521287e+25
# epsilon_infty_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 0.0020033425095306624
# E_1_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 8.249039339905267e+25
# epsilon_1_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 0.0017253761213308749
# E_2_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 4.768986324500695e+25
# E_RMS_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 1.7265574755534593e+25
# epsilon_2_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12 = 0.0016578166830843602

# E_infty_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 4.278512877997534e+25
# epsilon_infty_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 0.002060923672549788
# E_1_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 8.565125862318734e+25
# epsilon_1_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 0.0017915455846418268
# E_2_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 4.52847918123777e+25
# E_RMS_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 1.6394845888078498e+25
# epsilon_2_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12 = 0.0017089133269730565


In [ ]:
dx_over_s0_cm = (Lx/s0_cm) / np.array([1024, 768, 512, 256, 128])
E_infty_all = [E_infty_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12, E_infty_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12, E_infty_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12, E_infty_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12, E_infty_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12]
epsilon_infty_all = [epsilon_infty_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12, epsilon_infty_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12, epsilon_infty_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12, epsilon_infty_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12, epsilon_infty_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12]
E_1_all = [E_1_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12, E_1_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12, E_1_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12, E_1_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12, E_1_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12]
epsilon_1_all = [epsilon_1_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12, epsilon_1_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12, epsilon_1_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12, epsilon_1_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12, epsilon_1_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12]
E_2_all = [E_2_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12, E_2_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12, E_2_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12, E_2_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12, E_2_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12]
E_RMS_all = [E_RMS_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12, E_RMS_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12, E_RMS_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12, E_RMS_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12, E_RMS_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12]
epsilon_2_all = [epsilon_2_gaussian_diffusion_test_ncell_1024_1_1_1_dts_1p0emin12, epsilon_2_gaussian_diffusion_test_ncell_768_1_1_1_dts_1p0emin12, epsilon_2_gaussian_diffusion_test_ncell_512_1_1_1_dts_1p0emin12, epsilon_2_gaussian_diffusion_test_ncell_256_1_1_1_dts_1p0emin12, epsilon_2_gaussian_diffusion_test_ncell_128_1_1_1_dts_1p0emin12]

def linear_fit(x, y):
    """Least-squares fit of y = a*x + b. Returns (a, b)."""
    a, b = np.polyfit(np.asarray(x, dtype=float), np.asarray(y, dtype=float), 1)
    return float(a), float(b)

p_E_infty = linear_fit(np.log10(dx_over_s0_cm), np.log10(epsilon_infty_all))[0]
p_epsilon_infty = linear_fit(np.log10(dx_over_s0_cm), np.log10(epsilon_infty_all))[0]
p_E_1 = linear_fit(np.log10(dx_over_s0_cm), np.log10(E_1_all))[0]
p_epsilon_1 = linear_fit(np.log10(dx_over_s0_cm), np.log10(epsilon_1_all))[0]
p_E_2 = linear_fit(np.log10(dx_over_s0_cm), np.log10(E_2_all))[0]
p_E_RMS = linear_fit(np.log10(dx_over_s0_cm), np.log10(E_RMS_all))[0]
p_epsilon_2 = linear_fit(np.log10(dx_over_s0_cm), np.log10(epsilon_2_all))[0]

print(f"p_E_infty: {p_E_infty}")
print(f"p_epsilon_infty: {p_epsilon_infty}")
print(f"p_E_1: {p_E_1}")
print(f"p_epsilon_1: {p_epsilon_1}")
print(f"p_E_2: {p_E_2}")
print(f"p_E_RMS: {p_E_RMS}")
print(f"p_epsilon_2: {p_epsilon_2}")

C_E_infty = np.exp(linear_fit(np.log10(dx_over_s0_cm), np.log10(epsilon_infty_all))[1])
C_epsilon_infty = np.exp(linear_fit(np.log10(dx_over_s0_cm), np.log10(epsilon_infty_all))[1])
C_E_1 = np.exp(linear_fit(np.log10(dx_over_s0_cm), np.log10(E_1_all))[1])
C_epsilon_1 = np.exp(linear_fit(np.log10(dx_over_s0_cm), np.log10(epsilon_1_all))[1])
C_E_2 = np.exp(linear_fit(np.log10(dx_over_s0_cm), np.log10(E_2_all))[1])
C_E_RMS = np.exp(linear_fit(np.log10(dx_over_s0_cm), np.log10(E_RMS_all))[1])
C_epsilon_2 = np.exp(linear_fit(np.log10(dx_over_s0_cm), np.log10(epsilon_2_all))[1])

print(f"C_E_infty: {C_E_infty}")
print(f"C_epsilon_infty: {C_epsilon_infty}")
print(f"C_E_1: {C_E_1}")
print(f"C_epsilon_1: {C_epsilon_1}")
print(f"C_E_2: {C_E_2}")
print(f"C_E_RMS: {C_E_RMS}")
print(f"C_epsilon_2: {C_epsilon_2}")

In [ ]:
fontsize = 30
marker_size = 12
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(np.log10(dx_over_s0_cm), np.log10(epsilon_infty_all), label=r'$\epsilon_{\infty}\propto {\Delta x_{\mathrm{cell}}}^{' + f'{p_epsilon_infty:.3f}' + '}$', linewidth=4, marker='o', markersize=marker_size)
ax.plot(np.log10(dx_over_s0_cm), np.log10(epsilon_1_all), label=r'$\epsilon_1\propto {\Delta x_{\mathrm{cell}}}^{' + f'{p_epsilon_1:.3f}' + '}$', linewidth=4, marker='o', markersize=marker_size)
ax.plot(np.log10(dx_over_s0_cm), np.log10(epsilon_2_all), label=r'$\epsilon_2\propto {\Delta x_{\mathrm{cell}}}^{' + f'{p_epsilon_2:.3f}' + '}$', linewidth=4, marker='o', markersize=marker_size)
ax.set_ylabel(r'$\log_{10}\,\epsilon_i$', fontsize=fontsize)
# ax.set_xlim(np.log10(dx_over_s0_cm[0]), np.log10(dx_over_s0_cm[-1]))
ax.set_xlabel(r'$\log_{10}\,\,\Delta x_{\mathrm{cell}} / \sigma_0\,$', fontsize=fontsize)
leg = ax.legend(framealpha=0.0, ncol=1, fontsize=fontsize)
pf.apply_custom_settings(ax, leg, False)
fig.tight_layout()
fig.savefig("plots/gaussian_diffusion_test_epsilon_dxcell.pdf", bbox_inches="tight")
plt.show()